# Lexical Political Bias Neutralization — Kaggle training notebook

Self-contained implementation of `political_bias_neutralization_proposal.md`:
a **detect-then-edit** system for sentence-level lexical political bias in
news text (Stage 1: BABE/BASIL-trained span detector; Stage 2: seq2seq editor
under three data strategies — WNC transfer, LLM-synthesized pseudo-parallel,
unsupervised mask-and-infill/LEWIS-lite).

**Before running:** in the notebook's right-hand Settings panel, turn
**Internet ON** (required for pip installs, the HF Hub, and cloning BASIL)
and pick an **Accelerator** (GPU P100, or GPU T4 x2).

**What this notebook does, in order:**
1. Install the few packages not already in Kaggle's base image
2. Reconstruct the `biasneut` package from source (one `%%writefile` cell per
   module) — this **is** the entire codebase, not a summary of it
3. A ~1-minute sanity check with tiny random-weight models, to catch any
   wiring problem *before* spending real GPU-hours
4. Acquire BABE / BASIL / WNC and build leakage-safe splits
5. Train the Stage-1 detector (twice — a pipeline instance and a separate
   *independent* instance reserved for evaluation)
6. Train the Stage-2 editor under each strategy (A/B/C)
7. Run baselines + the full evaluation triad
8. A small inference demo

All artifacts land under `/kaggle/working/` (`data_cache/`, `runs/`,
`lexicons/`). See the "Cross-session resume" cell near the end for how to
carry `runs/` forward across Kaggle sessions (Kaggle's `/kaggle/working` does
not persist between separate edit sessions the way a long-lived server
would — the codebase's own checkpoint/resume logic (§7.4) only helps *within*
persisted storage, so you drive persistence via a Kaggle Dataset, as
described there).


## 1. Environment setup

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())


In [ ]:
# Kaggle's base image already ships torch/transformers/datasets/accelerate/
# sentencepiece/pandas/numpy/scipy/scikit-learn/matplotlib/nltk/pyyaml/tqdm.
# Only install what's missing, so we don't risk breaking the preinstalled,
# CUDA-matched torch build.
%pip install -q peft sentence-transformers evaluate sacrebleu sacremoses bert-score
# anthropic is only needed for real (non-dry-run) Strategy-B generation (Section 6B) when ANTHROPIC_API_KEY is set.
%pip install -q anthropic
# bitsandbytes is only needed for the optional QLoRA stretch editor arm (Section 6C-stretch).
%pip install -q bitsandbytes || echo "bitsandbytes install failed -- fine unless you plan to run the QLoRA stretch arm"


## 2. Reconstruct the `biasneut` package

Every cell below writes one source file of the actual, tested codebase to
`/kaggle/working/biasneut/...`. Nothing here is pseudocode or a stub — this
is the same package validated locally (87 passing tests, plus real end-to-end
runs against real BABE/BASIL/WNC data) before being packaged into this
notebook.

In [ ]:
import os

for d in [
    "biasneut", "biasneut/common", "biasneut/data", "biasneut/detector",
    "biasneut/editor", "biasneut/baselines", "biasneut/eval", "lexicons",
    "data_cache", "runs",
]:
    os.makedirs(f"/kaggle/working/{d}", exist_ok=True)
print("directories ready")


### 2.0 top-level package

In [ ]:
%%writefile /kaggle/working/biasneut/__init__.py
"""biasneut: detect-then-edit lexical political bias neutralization for news text."""

__version__ = "0.1.0"


In [ ]:
%%writefile /kaggle/working/biasneut/pipeline.py
"""End-to-end detect-then-edit inference pipeline (§5): Stage-1 localization
feeds Stage-2 span-marked editing. This is the modular architecture the
proposal keeps from Pryzant (§2.3, §3.1) — as opposed to a concurrent model
where detection is folded into the encoder rather than a separate module.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Protocol

from biasneut.detector.infer import DetectorInference, DetectorPrediction


@dataclass
class PipelineResult:
    source: str
    neutralized: str
    was_flagged_biased: bool
    bias_prob: float
    flagged_span: str | None


class Editor(Protocol):
    def neutralize(self, sentences: list[str], word_tags_per_sentence: list[list[str]], **kwargs) -> list[str]: ...


class BiasNeutralizationPipeline:
    """Wraps any Stage-2 editor that exposes ``neutralize(sentences,
    word_tags_per_sentence)`` — this matches ``SeqEditor`` (Strategies A/B and
    LEWIS-lite) directly; for the pure mask-and-infill arm, wrap
    ``MaskInfiller.infill`` in a small adapter with the same signature.
    """

    def __init__(self, detector: DetectorInference, editor: Editor, skip_unflagged: bool = True):
        self.detector = detector
        self.editor = editor
        self.skip_unflagged = skip_unflagged

    def __call__(self, sentences: list[str], **editor_kwargs) -> list[PipelineResult]:
        predictions: list[DetectorPrediction] = self.detector.predict(sentences)

        to_edit_idx = [
            i for i, p in enumerate(predictions) if p.is_biased or not self.skip_unflagged
        ]
        edited = [""] * len(sentences)
        if to_edit_idx:
            edit_sentences = [sentences[i] for i in to_edit_idx]
            edit_tags = [predictions[i].word_tags for i in to_edit_idx]
            outputs = self.editor.neutralize(edit_sentences, edit_tags, **editor_kwargs)
            for i, out in zip(to_edit_idx, outputs):
                edited[i] = out

        results = []
        for i, (sentence, pred) in enumerate(zip(sentences, predictions)):
            neutralized = edited[i] if i in to_edit_idx else sentence
            results.append(
                PipelineResult(
                    source=sentence,
                    neutralized=neutralized,
                    was_flagged_biased=pred.is_biased,
                    bias_prob=pred.bias_prob,
                    flagged_span=pred.biased_span_text,
                )
            )
        return results


class MaskInfillEditorAdapter:
    """Adapts ``MaskInfiller.infill`` (single sentence at a time, no beam
    search) to the batched ``Editor`` protocol so it can drive the same
    pipeline as the seq2seq editors."""

    def __init__(self, infiller):
        self.infiller = infiller

    def neutralize(self, sentences: list[str], word_tags_per_sentence: list[list[str]], **_) -> list[str]:
        return [self.infiller.infill(s, tags) for s, tags in zip(sentences, word_tags_per_sentence)]


### 2.1 `biasneut.common` — config, checkpoint/resume, compute, seeding, logging

In [ ]:
%%writefile /kaggle/working/biasneut/common/__init__.py



In [ ]:
%%writefile /kaggle/working/biasneut/common/checkpoint.py
"""Checkpoint/resume discipline for multi-session, quota-capped training (§7.4).

Kaggle sessions die mid-run (weekly quota, ~9-12h cap), so every long-running
loop in this codebase must be able to stop at an arbitrary step and pick back
up. ``CheckpointManager`` writes model/optimizer/scheduler/scaler state plus a
data-cursor (e.g. "shard index, row offset") atomically, and prunes all but
the ``keep_last`` most recent checkpoints to stay under Kaggle's ~20GB working
storage.
"""
from __future__ import annotations

import json
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch


@dataclass
class TrainingState:
    step: int
    epoch: int
    data_cursor: dict[str, Any]
    best_metric: float | None = None


class CheckpointManager:
    def __init__(self, output_dir: str | Path, keep_last: int = 2):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.keep_last = keep_last

    def _ckpt_dir(self, step: int) -> Path:
        return self.output_dir / f"checkpoint-{step:08d}"

    def save(
        self,
        step: int,
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer | None,
        scheduler: Any | None,
        state: TrainingState,
    ) -> Path:
        ckpt_dir = self._ckpt_dir(step)
        tmp_dir = self.output_dir / f".tmp-{step:08d}"
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir)
        tmp_dir.mkdir(parents=True)

        torch.save(model.state_dict(), tmp_dir / "model.pt")
        if optimizer is not None:
            torch.save(optimizer.state_dict(), tmp_dir / "optimizer.pt")
        if scheduler is not None:
            torch.save(scheduler.state_dict(), tmp_dir / "scheduler.pt")
        with open(tmp_dir / "state.json", "w") as f:
            json.dump(
                {"step": state.step, "epoch": state.epoch, "data_cursor": state.data_cursor,
                 "best_metric": state.best_metric},
                f,
            )

        # Atomic-ish: rename fully-written tmp dir into place, then prune.
        if ckpt_dir.exists():
            shutil.rmtree(ckpt_dir)
        tmp_dir.rename(ckpt_dir)
        self._prune()
        return ckpt_dir

    def _prune(self) -> None:
        ckpts = sorted(self.output_dir.glob("checkpoint-*"), key=lambda p: p.name)
        for stale in ckpts[: max(0, len(ckpts) - self.keep_last)]:
            shutil.rmtree(stale, ignore_errors=True)

    def latest(self) -> Path | None:
        ckpts = sorted(self.output_dir.glob("checkpoint-*"), key=lambda p: p.name)
        return ckpts[-1] if ckpts else None

    def load(
        self,
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer | None = None,
        scheduler: Any | None = None,
        map_location: str | torch.device = "cpu",
    ) -> TrainingState | None:
        """Resume from the latest checkpoint, or return None if none exists."""
        ckpt_dir = self.latest()
        if ckpt_dir is None:
            return None

        model.load_state_dict(torch.load(ckpt_dir / "model.pt", map_location=map_location))
        if optimizer is not None and (ckpt_dir / "optimizer.pt").exists():
            optimizer.load_state_dict(torch.load(ckpt_dir / "optimizer.pt", map_location=map_location))
        if scheduler is not None and (ckpt_dir / "scheduler.pt").exists():
            scheduler.load_state_dict(torch.load(ckpt_dir / "scheduler.pt", map_location=map_location))
        with open(ckpt_dir / "state.json") as f:
            payload = json.load(f)
        return TrainingState(**payload)


In [ ]:
%%writefile /kaggle/working/biasneut/common/compute.py
"""Memory-reduction stack helpers (§7.2): precision, checkpointing, 8-bit optim.

Each function degrades gracefully when the ideal option isn't available (e.g.
bf16 on a pre-Turing GPU, or bitsandbytes not installed) so the same config
works on a P100, a T4, or a laptop GPU without edits.
"""
from __future__ import annotations

import logging

import torch

from biasneut.common.config import ComputeConfig

logger = logging.getLogger(__name__)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def resolve_dtype(mixed_precision: str) -> torch.dtype:
    """bf16 requires tensor cores (Turing+); P100 (Pascal) silently has none,
    so callers on a P100 should request "fp16" per §7.2 point 3."""
    if mixed_precision == "bf16":
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        logger.warning("bf16 requested but not supported on this device; falling back to fp32")
        return torch.float32
    if mixed_precision == "fp16":
        return torch.float16
    return torch.float32


def enable_gradient_checkpointing(model: torch.nn.Module) -> None:
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    else:
        logger.warning("%s has no gradient_checkpointing_enable()", type(model).__name__)


def build_optimizer(model: torch.nn.Module, cfg: ComputeConfig, learning_rate: float,
                     weight_decay: float = 0.0) -> torch.optim.Optimizer:
    """AdamW, 8-bit via bitsandbytes when requested and available (§7.2 point 4)."""
    params = [p for p in model.parameters() if p.requires_grad]
    if cfg.optimizer == "adamw_8bit":
        try:
            import bitsandbytes as bnb

            return bnb.optim.AdamW8bit(params, lr=learning_rate, weight_decay=weight_decay)
        except ImportError:
            logger.warning("bitsandbytes not installed; falling back to torch.optim.AdamW")
    return torch.optim.AdamW(params, lr=learning_rate, weight_decay=weight_decay)


def autocast_context(dtype: torch.dtype):
    device_type = "cuda" if torch.cuda.is_available() else "cpu"
    enabled = dtype in (torch.bfloat16, torch.float16)
    return torch.autocast(device_type=device_type, dtype=dtype, enabled=enabled)


In [ ]:
%%writefile /kaggle/working/biasneut/common/config.py
"""Dataclass configs + YAML (de)serialization.

Kept deliberately simple (no Hydra/OmegaConf dependency) — every config is a
plain dataclass with defaults matching the proposal's §7 compute plan, loaded
from a YAML file and overridable via CLI ``--set key=value`` pairs in scripts.
"""
from __future__ import annotations

import dataclasses
import typing
from pathlib import Path
from typing import Any, TypeVar

import yaml

T = TypeVar("T")


def _coerce(value: str) -> Any:
    """Best-effort str -> {bool,int,float,str} coercion for CLI overrides."""
    if value.lower() in ("true", "false"):
        return value.lower() == "true"
    for cast in (int, float):
        try:
            return cast(value)
        except ValueError:
            continue
    return value


def _expand_dotted_keys(data: dict[str, Any]) -> dict[str, Any]:
    """Turn CLI-style dotted overrides (``compute.gradient_checkpointing=false``)
    into nested dicts (``{"compute": {"gradient_checkpointing": False}}``) so
    they merge cleanly with any nested config loaded from YAML."""
    expanded: dict[str, Any] = {}
    for key, value in data.items():
        if "." in key:
            head, rest = key.split(".", 1)
            expanded.setdefault(head, {})
            if not isinstance(expanded[head], dict):
                raise ValueError(f"Conflicting override for {head!r}: both a scalar and a nested key given")
            expanded[head][rest] = value
        else:
            expanded[key] = value
    # Recurse so multi-level dotted keys (a.b.c) also expand correctly.
    for key, value in list(expanded.items()):
        if isinstance(value, dict):
            expanded[key] = _expand_dotted_keys(value)
    return expanded


def _coerce_scalar_to_field_type(value: Any, field_type: Any) -> Any:
    """PyYAML's default (YAML-1.1) resolver only recognizes scientific notation
    as a float when it has an explicit decimal point (``2.0e-5``, not
    ``2e-5``) — the latter silently loads as a ``str``. Rather than rely on
    every config file getting that spelling right, coerce int/float fields
    back to their declared type when YAML handed us a string instead."""
    if field_type in (float, int) and isinstance(value, str):
        try:
            return field_type(value)
        except ValueError:
            return value
    return value


def _from_dict(cls: type[T], data: dict[str, Any]) -> T:
    """Recursively build a dataclass from a dict, reconstructing any nested
    dataclass fields (e.g. ``EditorConfig.compute: ComputeConfig``) instead of
    leaving them as plain dicts — required for the save_config/load_config
    round trip to actually produce a usable config object."""
    data = _expand_dotted_keys(data)
    field_names = {f.name for f in dataclasses.fields(cls)}
    unknown = set(data) - field_names
    if unknown:
        raise ValueError(f"Unknown config keys for {cls.__name__}: {sorted(unknown)}")

    type_hints = typing.get_type_hints(cls)
    kwargs = {}
    for key, value in data.items():
        field_type = type_hints.get(key)
        if dataclasses.is_dataclass(field_type) and isinstance(value, dict):
            kwargs[key] = _from_dict(field_type, value)
        else:
            kwargs[key] = _coerce_scalar_to_field_type(value, field_type)
    return cls(**kwargs)


def load_config(cls: type[T], path: str | Path | None, overrides: dict[str, Any] | None = None) -> T:
    """Build a dataclass ``cls`` from an optional YAML file plus overrides."""
    data: dict[str, Any] = {}
    if path is not None and Path(path).exists():
        with open(path) as f:
            data = yaml.safe_load(f) or {}
    if overrides:
        data.update(overrides)
    return _from_dict(cls, data)


def parse_cli_overrides(pairs: list[str]) -> dict[str, Any]:
    """Parse ``["lr=3e-4", "epochs=5"]`` style CLI overrides into a dict."""
    out: dict[str, Any] = {}
    for pair in pairs:
        key, _, value = pair.partition("=")
        out[key] = _coerce(value)
    return out


def save_config(cfg: Any, path: str | Path) -> None:
    with open(path, "w") as f:
        yaml.safe_dump(dataclasses.asdict(cfg), f, sort_keys=False)


@dataclasses.dataclass
class ComputeConfig:
    """§7.2 memory-reduction stack, as knobs rather than hardcoded behavior."""

    mixed_precision: str = "bf16"  # "bf16" | "fp16" | "no" — bf16 needs Turing+/tensor cores
    gradient_checkpointing: bool = True
    optimizer: str = "adamw_8bit"  # "adamw_8bit" | "adamw_torch"
    gradient_accumulation_steps: int = 1
    max_seq_length: int = 128
    seed: int = 42


@dataclasses.dataclass
class DetectorConfig:
    """Stage 1 — sentence + span detector (§5.1)."""

    backbone: str = "launch/POLITICS"
    fallback_backbone: str = "distilroberta-base"
    max_seq_length: int = 128
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    train_batch_size: int = 16
    eval_batch_size: int = 32
    num_epochs: int = 5
    warmup_ratio: float = 0.06
    span_loss_weight: float = 1.0
    sentence_loss_weight: float = 1.0
    output_dir: str = "runs/detector"
    checkpoint_every_steps: int = 200
    compute: ComputeConfig = dataclasses.field(default_factory=ComputeConfig)


@dataclasses.dataclass
class EditorConfig:
    """Stage 2 — seq2seq editor, shared across strategies A/B/C (§5.2)."""

    strategy: str = "a"  # "a" | "b" | "c"
    backbone: str = "t5-small"
    max_source_length: int = 128
    max_target_length: int = 128
    learning_rate: float = 3e-4
    train_batch_size: int = 16
    eval_batch_size: int = 32
    num_epochs: int = 4
    warmup_ratio: float = 0.06
    span_marker_start: str = "<bias>"
    span_marker_end: str = "</bias>"
    use_constrained_decoding: bool = True
    copy_bias_strength: float = 2.5
    num_beams: int = 4
    output_dir: str = "runs/editor"
    checkpoint_every_steps: int = 200
    compute: ComputeConfig = dataclasses.field(default_factory=ComputeConfig)


@dataclasses.dataclass
class QLoRAEditorConfig:
    """Stretch arm (§5.2, §7.3) — QLoRA'd 1-3B instruction-tuned decoder editor."""

    backbone: str = "Qwen/Qwen2.5-1.5B-Instruct"
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: tuple[str, ...] = ("q_proj", "k_proj", "v_proj", "o_proj")
    max_seq_length: int = 256
    learning_rate: float = 2e-4
    train_batch_size: int = 2
    gradient_accumulation_steps: int = 8
    num_epochs: int = 3
    output_dir: str = "runs/editor_qlora"


@dataclasses.dataclass
class EvalConfig:
    """§6 evaluation protocol."""

    independent_classifier_dir: str = "runs/detector_eval_independent"
    fluency_lm: str = "gpt2"
    grammaticality_model: str = "textattack/roberta-base-CoLA"
    sbert_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    bertscore_model: str = "roberta-large"
    num_bootstrap_samples: int = 1000
    seeds: tuple[int, ...] = (13, 42, 1337)
    output_dir: str = "runs/eval"


In [ ]:
%%writefile /kaggle/working/biasneut/common/logging_utils.py
"""Shared logging setup so scripts/notebooks get consistent, timestamped logs."""
from __future__ import annotations

import logging
import sys


def setup_logging(level: int = logging.INFO) -> None:
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
        stream=sys.stdout,
        force=True,
    )


In [ ]:
%%writefile /kaggle/working/biasneut/common/seeding.py
"""Deterministic seeding across python/numpy/torch (§6.5 — multi-seed reporting)."""
from __future__ import annotations

import random

import numpy as np
import torch


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


### 2.2 `biasneut.data` — BABE/BASIL/WNC loaders, splits, collation, pseudo-parallel generation

In [ ]:
%%writefile /kaggle/working/biasneut/data/__init__.py



In [ ]:
%%writefile /kaggle/working/biasneut/data/babe.py
"""BABE loader (§3.2, §4.1) — sentence-level bias label + a bag of biased
surface words, used for Stage-1 sentence-classification and (approximate)
span supervision.
"""
from __future__ import annotations

import ast
import logging

from biasneut.data.schema import DetectionExample
from biasneut.data.tokenize_utils import bio_tags_for_words, whitespace_tokenize

logger = logging.getLogger(__name__)

HF_DATASET_ID = "mediabiasgroup/BABE"


def _parse_biased_words(raw: str | list[str] | None) -> list[str]:
    if raw is None:
        return []
    if isinstance(raw, list):
        return raw
    raw = raw.strip()
    if not raw or raw == "[]":
        return []
    try:
        parsed = ast.literal_eval(raw)
        return list(parsed) if isinstance(parsed, (list, tuple)) else []
    except (ValueError, SyntaxError):
        logger.warning("Could not parse biased_words field: %r", raw)
        return []


def load_babe(split: str = "train") -> list[DetectionExample]:
    """Load BABE from the Hugging Face Hub and normalize to DetectionExample.

    BABE ships as a single split; callers should build their own leakage-safe
    train/dev/test split via ``biasneut.data.splits`` rather than relying on
    an HF-provided split name.
    """
    from datasets import load_dataset

    ds = load_dataset(HF_DATASET_ID, split=split)
    examples = []
    for row in ds:
        text = row["text"]
        tokens = whitespace_tokenize(text)
        biased_words = _parse_biased_words(row.get("biased_words"))
        bio_tags = bio_tags_for_words(tokens, biased_words) if biased_words else ["O"] * len(tokens)
        examples.append(
            DetectionExample(
                text=text,
                is_biased=bool(row["label"] == 1),
                bio_tags=bio_tags,
                source="babe",
                story_id=row.get("uuid"),
                outlet=row.get("outlet"),
            )
        )
    return examples


In [ ]:
%%writefile /kaggle/working/biasneut/data/basil.py
"""BASIL loader (§3.2, §4.1) — sentence + phrase-level lexical/informational
bias spans, source-clustered by story (same event, 3 outlets: hpo/fox/nyt).

We only load the *lexical* bias spans as detection targets: §2.1 draws the
lexical/informational distinction as the load-bearing scope decision for this
whole project, and Pryzant's copy-heavy editor is explicitly the wrong tool
for informational bias (§2.2). Informational spans are still surfaced on
``DetectionExample`` via the ``basil_info_biased`` note in provenance so
downstream analysis can report on the discarded half without training on it.

BASIL ships as raw per-article JSON in a GitHub repo (no HF Hub mirror with
spans), so this loader shallow-clones ``launchnlp/BASIL`` into a local cache
rather than hitting the GitHub API file-by-file.
"""
from __future__ import annotations

import json
import logging
import subprocess
import tarfile
import urllib.error
import urllib.request
from pathlib import Path

from biasneut.data.schema import DetectionExample
from biasneut.data.tokenize_utils import bio_tags_for_char_span, merge_bio_tags, whitespace_tokenize

logger = logging.getLogger(__name__)

REPO_URL = "https://github.com/launchnlp/BASIL"
REPO_SLUG = "launchnlp/BASIL"


def _download_tarball_fallback(repo_dir: Path) -> None:
    """Some sandboxed/CI environments allow plain HTTPS to GitHub's asset
    hosts while blocking `git clone` to github.com itself (observed in a
    Claude Code sandbox: `codeload.github.com` reachable, `github.com`
    times out) — fetch the repo as a tarball via stdlib `urllib`/`tarfile`
    instead of shelling out to `git`."""
    tmp_tar = repo_dir.parent / "BASIL.tar.gz"
    last_error: Exception | None = None
    for branch in ("main", "master"):
        url = f"https://codeload.github.com/{REPO_SLUG}/tar.gz/refs/heads/{branch}"
        try:
            with urllib.request.urlopen(url, timeout=60) as response, open(tmp_tar, "wb") as f:
                f.write(response.read())
            break
        except urllib.error.HTTPError as e:
            last_error = e
            continue
    else:
        raise RuntimeError(f"Could not fetch {REPO_SLUG} via git clone or tarball fallback") from last_error

    repo_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tmp_tar) as tar:
        # GitHub's tarball wraps everything in a single "<repo>-<branch>/"
        # root directory; strip it so repo_dir/articles, repo_dir/annotations
        # land directly under repo_dir (equivalent to `tar --strip-components=1`).
        members = []
        for member in tar.getmembers():
            parts = Path(member.name).parts
            if len(parts) <= 1:
                continue
            member.name = str(Path(*parts[1:]))
            members.append(member)
        tar.extractall(repo_dir, members=members, filter="data")
    tmp_tar.unlink(missing_ok=True)


def ensure_basil_repo(cache_dir: str | Path) -> Path:
    cache_dir = Path(cache_dir)
    repo_dir = cache_dir / "BASIL"
    if (repo_dir / "articles").exists() and (repo_dir / "annotations").exists():
        return repo_dir
    cache_dir.mkdir(parents=True, exist_ok=True)
    logger.info("Cloning %s into %s", REPO_URL, repo_dir)
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)],
            check=True, capture_output=True, text=True,
        )
    except (subprocess.CalledProcessError, FileNotFoundError) as e:
        detail = e.stderr.strip() if isinstance(e, subprocess.CalledProcessError) and e.stderr else str(e)
        logger.warning("git clone failed (%s); falling back to tarball download", detail)
        _download_tarball_fallback(repo_dir)
    return repo_dir


def _flatten_sentences(body_paragraphs: list[list[str]]) -> list[str]:
    return [sent for para in body_paragraphs for sent in para]


def _load_article_pair(article_path: Path, annotation_path: Path) -> list[DetectionExample]:
    article = json.loads(article_path.read_text())
    annotation = json.loads(annotation_path.read_text())

    sentences = _flatten_sentences(article["body-paragraphs"])
    story_id = article.get("triplet-uuid", article.get("uuid"))
    # BASIL's raw "source" field is inconsistently cased across articles
    # (observed: "nyt"/"NYT", "fox"/"FOX", "hpo"/"HPO" for the same outlet).
    outlet = article.get("source", "").lower() or None

    lex_tags_by_sentence = [["O"] * len(whitespace_tokenize(s)) for s in sentences]
    info_flagged = [False] * len(sentences)

    for phrase in annotation.get("phrase-level-annotations", []):
        span_text = phrase.get("txt", "")
        if not span_text.strip():
            continue
        # BASIL doesn't give an explicit sentence index for each phrase
        # annotation, so we locate it by substring search across the
        # article's sentences (first match; spans are short and specific
        # enough in practice that collisions are rare).
        sent_idx = next((i for i, s in enumerate(sentences) if span_text in s), None)
        if sent_idx is None:
            continue
        if phrase.get("bias") == "lex":
            tokens = whitespace_tokenize(sentences[sent_idx])
            span_tags = bio_tags_for_char_span(tokens, sentences[sent_idx], span_text)
            lex_tags_by_sentence[sent_idx] = merge_bio_tags(lex_tags_by_sentence[sent_idx], span_tags)
        elif phrase.get("bias") == "inf":
            info_flagged[sent_idx] = True

    examples = []
    for i, sent in enumerate(sentences):
        bio_tags = lex_tags_by_sentence[i]
        is_lex_biased = any(t != "O" for t in bio_tags)
        examples.append(
            DetectionExample(
                text=sent,
                is_biased=is_lex_biased,
                bio_tags=bio_tags,
                source="basil",
                story_id=story_id,
                outlet=outlet,
            )
        )
    return examples


def load_basil(cache_dir: str | Path = "data_cache") -> list[DetectionExample]:
    repo_dir = ensure_basil_repo(cache_dir)
    articles_dir = repo_dir / "articles"
    annotations_dir = repo_dir / "annotations"

    examples: list[DetectionExample] = []
    n_pairs = 0
    for article_path in sorted(articles_dir.rglob("*.json")):
        rel = article_path.relative_to(articles_dir)
        annotation_path = annotations_dir / rel.parent / (article_path.stem + "_ann.json")
        if not annotation_path.exists():
            logger.warning("No annotation file for %s, skipping", article_path)
            continue
        examples.extend(_load_article_pair(article_path, annotation_path))
        n_pairs += 1

    logger.info("Loaded BASIL: %d article/annotation pairs -> %d sentences", n_pairs, len(examples))
    return examples


In [ ]:
%%writefile /kaggle/working/biasneut/data/cache_io.py
"""Read back the JSON caches written by ``scripts/prepare_data.py`` into
dataclass examples."""
from __future__ import annotations

import json
from pathlib import Path

from biasneut.data.schema import DetectionExample, EditExample


def load_detection_examples(path: str | Path) -> list[DetectionExample]:
    data = json.loads(Path(path).read_text())
    return [DetectionExample(**row) for row in data]


def load_edit_examples(path: str | Path) -> list[EditExample]:
    data = json.loads(Path(path).read_text())
    return [EditExample(**row) for row in data]


In [ ]:
%%writefile /kaggle/working/biasneut/data/collate.py
"""Detector collation: subword alignment of whitespace-level BIO tags.

Standard NER-style alignment — only the first subword of each whitespace
token carries the tag; continuation subwords and special tokens get -100 so
the token-classification loss ignores them.
"""
from __future__ import annotations

from dataclasses import dataclass

import torch

from biasneut.data.schema import DetectionExample
from biasneut.data.tokenize_utils import whitespace_tokenize

TAG2ID = {"O": 0, "B-BIAS": 1, "I-BIAS": 2}
ID2TAG = {v: k for k, v in TAG2ID.items()}
IGNORE_INDEX = -100


@dataclass
class DetectorCollator:
    tokenizer: object
    max_length: int = 128

    def __call__(self, batch: list[DetectionExample]) -> dict[str, torch.Tensor]:
        word_lists = [whitespace_tokenize(ex.text) for ex in batch]
        encoding = self.tokenizer(
            word_lists,
            is_split_into_words=True,
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        token_labels = torch.full_like(encoding["input_ids"], IGNORE_INDEX)
        for i, ex in enumerate(batch):
            word_ids = encoding.word_ids(batch_index=i)
            bio_tags = ex.bio_tags or ["O"] * len(word_lists[i])
            prev_word_id = None
            for j, word_id in enumerate(word_ids):
                if word_id is None:
                    continue
                if word_id != prev_word_id and word_id < len(bio_tags):
                    token_labels[i, j] = TAG2ID[bio_tags[word_id]]
                prev_word_id = word_id

        sentence_labels = torch.tensor([1 if ex.is_biased else 0 for ex in batch], dtype=torch.long)

        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "sentence_labels": sentence_labels,
            "token_labels": token_labels,
        }


In [ ]:
%%writefile /kaggle/working/biasneut/data/pseudo_parallel.py
"""Strategy B — LLM-synthesized pseudo-parallel corpus (§4.2, §4.3).

Generation happens offline (no GPU cost) against one or more LLM providers;
the resulting pairs are training-side only and must never leak into
evaluation (§4.3: "Synthetic data never touches evaluation"). This module
implements generation plus the three mandatory circularity mitigations:

1. multi-model generation (``clients`` accepts >=1 distinct model families)
2. independent-classifier filtering (reject targets that don't reduce bias)
3. semantic-similarity floor (reject targets that drift from the source)

Human spot-check (the fourth mitigation) is inherently manual; ``sample_for_
human_review`` exports a random subset for that step rather than automating it.
"""
from __future__ import annotations

import csv
import logging
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Protocol

from biasneut.data.schema import DetectionExample, EditExample

logger = logging.getLogger(__name__)

NEUTRALIZE_PROMPT_TEMPLATE = """You are helping build a training set for a lexical bias neutralization model.

Rewrite the sentence below to remove loaded/partisan word choice (e.g. \
epithets, slanted verbs, framing adjectives) while:
- preserving every factual claim, entity, and quote exactly
- changing as few words as possible (this is a local edit task, not a rewrite)
- keeping the same sentence structure and length where possible

If the sentence is already neutral, return it unchanged.
Return ONLY the rewritten sentence, no explanation, no quotes.

Sentence: {sentence}"""


class LLMClient(Protocol):
    name: str

    def generate(self, prompt: str) -> str: ...


@dataclass
class AnthropicClient:
    """Thin wrapper so the generation loop doesn't depend on the SDK shape."""

    model: str = "claude-sonnet-5"
    name: str = "anthropic"
    max_tokens: int = 256

    def __post_init__(self):
        import anthropic

        self._client = anthropic.Anthropic()

    def generate(self, prompt: str) -> str:
        resp = self._client.messages.create(
            model=self.model,
            max_tokens=self.max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.content[0].text.strip()


@dataclass
class EchoClient:
    """No-op client for tests/dry-runs: returns the input unchanged so the
    generation/filtering pipeline can be exercised without any API key."""

    name: str = "echo"

    def generate(self, prompt: str) -> str:
        # Extract the sentence back out of the templated prompt.
        marker = "Sentence: "
        idx = prompt.rfind(marker)
        return prompt[idx + len(marker):].strip() if idx != -1 else prompt


def generate_pseudo_parallel(
    examples: list[DetectionExample],
    clients: list[LLMClient],
    prompt_template: str = NEUTRALIZE_PROMPT_TEMPLATE,
) -> list[EditExample]:
    """One candidate per (example, client) — deliberately not deduplicated
    here so downstream filtering can compare candidates from different model
    families against the same source sentence."""
    pairs: list[EditExample] = []
    for ex in examples:
        if not ex.is_biased:
            continue
        prompt = prompt_template.format(sentence=ex.text)
        for client in clients:
            try:
                target = client.generate(prompt)
            except Exception:
                logger.exception("Generation failed for client=%s on: %s", client.name, ex.text[:80])
                continue
            pairs.append(
                EditExample(
                    source=ex.text,
                    target=target,
                    biased_span=None,
                    strategy="llm_synth",
                    provenance=f"model={client.name}",
                )
            )
    return pairs


def filter_pseudo_parallel(
    pairs: list[EditExample],
    bias_score_fn: Callable[[str], float],
    similarity_fn: Callable[[str, str], float],
    similarity_floor: float = 0.6,
    require_bias_drop: bool = True,
) -> list[EditExample]:
    """Apply §4.3's circularity mitigations.

    ``bias_score_fn`` and ``similarity_fn`` must come from the *independent*
    evaluator (a BABE-trained classifier instance that never saw generation
    outputs, and an SBERT model) — never from the generating LLM itself.
    """
    kept = []
    for pair in pairs:
        sim = similarity_fn(pair.source, pair.target)
        if sim < similarity_floor:
            continue
        if require_bias_drop:
            source_score = bias_score_fn(pair.source)
            target_score = bias_score_fn(pair.target)
            if target_score >= source_score:
                continue
        kept.append(pair)
    logger.info("Filtered pseudo-parallel pairs: %d -> %d", len(pairs), len(kept))
    return kept


def sample_for_human_review(pairs: list[EditExample], n: int = 200, seed: int = 42,
                             out_path: str | Path = "human_review_sample.csv") -> Path:
    sample = random.Random(seed).sample(pairs, k=min(n, len(pairs)))
    out_path = Path(out_path)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["source", "target", "provenance", "bias_removed_ok", "meaning_preserved_ok", "fluent_ok"])
        for pair in sample:
            writer.writerow([pair.source, pair.target, pair.provenance, "", "", ""])
    return out_path


In [ ]:
%%writefile /kaggle/working/biasneut/data/schema.py
"""Shared example schemas so BABE/BASIL/WNC/pseudo-parallel data all normalize
to the same shape before hitting the detector or editor."""
from __future__ import annotations

from dataclasses import dataclass


@dataclass
class DetectionExample:
    """One sentence for Stage-1 training/eval.

    ``bio_tags`` is None when only a sentence-level label is available (BABE
    without word spans); the token-classification loss is masked out for
    those examples rather than penalizing absent span supervision.
    """

    text: str
    is_biased: bool
    bio_tags: list[str] | None  # "O" / "B-BIAS" / "I-BIAS" per whitespace token, or None
    source: str  # "babe" | "basil"
    story_id: str | None = None  # for leakage-safe splitting (§6.5)
    outlet: str | None = None


@dataclass
class EditExample:
    """One (biased -> neutral) pair for Stage-2 training, from any strategy."""

    source: str
    target: str
    biased_span: str | None
    strategy: str  # "wnc" | "llm_synth" | "unsupervised"
    provenance: str  # free-text note, e.g. generating model name for Strategy B


In [ ]:
%%writefile /kaggle/working/biasneut/data/splits.py
"""Leakage-safe splitting (§6.5): "same-story sentences must not straddle the
split boundary." We split whole groups (story id, falling back to outlet, or
the example's own index when the source has no grouping key) into
train/dev/test using a greedy load-balancer so that group sizes need not be
uniform for the split ratios to come out close to the target.
"""
from __future__ import annotations

import random
from collections import defaultdict
from typing import Callable, TypeVar

T = TypeVar("T")

GroupKeyFn = Callable[[T], str | None]


def default_group_key(example) -> str | None:
    """Group by story_id if present, else by outlet, else ungrouped."""
    story_id = getattr(example, "story_id", None)
    if story_id:
        return f"story::{story_id}"
    outlet = getattr(example, "outlet", None)
    if outlet:
        return f"outlet::{outlet}"
    return None


def group_disjoint_split(
    examples: list[T],
    group_key_fn: GroupKeyFn = default_group_key,
    ratios: tuple[float, float, float] = (0.8, 0.1, 0.1),
    seed: int = 42,
) -> tuple[list[T], list[T], list[T]]:
    """Split ``examples`` into (train, dev, test) such that no group (e.g. all
    sentences from one story, across outlets) straddles the boundary.
    Examples with no group key are treated as singleton groups.
    """
    if abs(sum(ratios) - 1.0) > 1e-6:
        raise ValueError(f"ratios must sum to 1.0, got {ratios}")

    groups: dict[str, list[int]] = defaultdict(list)
    for i, ex in enumerate(examples):
        key = group_key_fn(ex) or f"__singleton__::{i}"
        groups[key].append(i)

    group_keys = list(groups.keys())
    random.Random(seed).shuffle(group_keys)

    n_total = len(examples)
    targets = [r * n_total for r in ratios]
    counts = [0, 0, 0]
    split_indices: tuple[list[int], list[int], list[int]] = ([], [], [])

    for key in group_keys:
        idxs = groups[key]
        # Assign this whole group to whichever split is furthest below its
        # target share (a simple greedy load balancer for approximate ratios).
        deficits = [targets[j] - counts[j] for j in range(3)]
        target_split = max(range(3), key=lambda j: deficits[j])
        split_indices[target_split].extend(idxs)
        counts[target_split] += len(idxs)

    train = [examples[i] for i in split_indices[0]]
    dev = [examples[i] for i in split_indices[1]]
    test = [examples[i] for i in split_indices[2]]
    return train, dev, test


def assert_no_leakage(*splits: list, group_key_fn: GroupKeyFn = default_group_key) -> None:
    """Raise if any group key appears in more than one split (safety net to
    call right before training, not just at split-creation time)."""
    seen: dict[str, int] = {}
    for split_idx, split in enumerate(splits):
        for ex in split:
            key = group_key_fn(ex)
            if key is None:
                continue
            if key in seen and seen[key] != split_idx:
                raise ValueError(f"Leakage detected: group {key!r} appears in multiple splits")
            seen[key] = split_idx


In [ ]:
%%writefile /kaggle/working/biasneut/data/tokenize_utils.py
"""Whitespace tokenization + BIO tagging shared by BABE and BASIL loaders.

Both source datasets give us *words* that are biased, not char offsets into a
canonical tokenization, so we standardize on simple whitespace tokenization
for BIO tags at the data layer; the detector's collation step (§data/collate)
re-aligns these to subword tokens at train time.
"""
from __future__ import annotations

import re

_WORD_RE = re.compile(r"\S+")


def whitespace_tokenize(text: str) -> list[str]:
    return _WORD_RE.findall(text)


def _normalize(tok: str) -> str:
    return tok.strip('"“”‘’\'.,!?;:()[]').lower()


def bio_tags_for_words(tokens: list[str], biased_words: list[str]) -> list[str]:
    """Tag each whitespace token O/B-BIAS/I-BIAS by matching against a bag of
    biased surface words (BABE's ``biased_words`` list). Contiguous runs of
    matched tokens become a single B-/I- span; this is an approximation since
    BABE gives an unordered bag of words rather than spans, but it is the best
    signal available and matches how Pryzant-style detectors consume BABE.
    """
    normalized_targets = {_normalize(w) for w in biased_words if w.strip()}
    tags = ["O"] * len(tokens)
    if not normalized_targets:
        return tags

    in_span = False
    for i, tok in enumerate(tokens):
        if _normalize(tok) in normalized_targets:
            tags[i] = "I-BIAS" if in_span else "B-BIAS"
            in_span = True
        else:
            in_span = False
    return tags


def bio_tags_for_char_span(tokens: list[str], text: str, span_text: str) -> list[str]:
    """Tag tokens covered by the first occurrence of ``span_text`` inside
    ``text`` (BASIL's phrase-level ``txt`` annotations)."""
    tags = ["O"] * len(tokens)
    idx = text.find(span_text)
    if idx == -1 or not span_text.strip():
        return tags

    span_end = idx + len(span_text)
    cursor = 0
    in_span = False
    for i, tok in enumerate(tokens):
        start = text.find(tok, cursor)
        if start == -1:
            start = cursor
        end = start + len(tok)
        cursor = end
        overlaps = start < span_end and end > idx
        if overlaps:
            tags[i] = "I-BIAS" if in_span else "B-BIAS"
            in_span = True
        else:
            in_span = False
    return tags


def merge_bio_tags(a: list[str], b: list[str]) -> list[str]:
    """OR two BIO taggings of the same token sequence together (used when a
    sentence has multiple, possibly overlapping, biased spans)."""
    out = []
    in_span = False
    for ta, tb in zip(a, b):
        biased = ta != "O" or tb != "O"
        if biased:
            out.append("I-BIAS" if in_span else "B-BIAS")
        else:
            out.append("O")
        in_span = biased
    return out


In [ ]:
%%writefile /kaggle/working/biasneut/data/wnc.py
"""WNC loader (§3.1, §4.1, Strategy A) — Wikipedia Neutrality Corpus, used
*only* for editor pretraining, never as primary supervision (the whole
premise of this project is that WNC's Wikipedia register does not transfer
cleanly to news, per §2.1/§4.2).

File format (confirmed from the locally-vendored ``bias_data/bias_data/WNC``
release): 7-column TSV, no header —
``id, src_tok, tgt_tok, src_raw, tgt_raw, src_pos_tags, tgt_dep_tags``.
We use only the raw (untokenized) columns; the BERT-wordpiece columns are
Pryzant-specific artifacts we don't need with a modern tokenizer.
"""
from __future__ import annotations

import csv
import difflib
from pathlib import Path

from biasneut.data.schema import EditExample

_COLUMNS = ("id", "src_tok", "tgt_tok", "src_raw", "tgt_raw", "src_pos", "tgt_dep")


def _extract_removed_span(src_raw: str, tgt_raw: str) -> str | None:
    """Best-effort extraction of the word(s) removed/changed going src->tgt,
    for reporting/inspection only (training uses the full src/tgt pair)."""
    sm = difflib.SequenceMatcher(a=src_raw.split(), b=tgt_raw.split())
    removed = []
    for tag, i1, i2, _, _ in sm.get_opcodes():
        if tag in ("delete", "replace"):
            removed.extend(src_raw.split()[i1:i2])
    return " ".join(removed) if removed else None


def load_wnc(path: str | Path, max_examples: int | None = None) -> list[EditExample]:
    path = Path(path)
    examples = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        for row in reader:
            if len(row) < 5:
                continue
            row_dict = dict(zip(_COLUMNS, row))
            src_raw, tgt_raw = row_dict["src_raw"], row_dict["tgt_raw"]
            examples.append(
                EditExample(
                    source=src_raw,
                    target=tgt_raw,
                    biased_span=_extract_removed_span(src_raw, tgt_raw),
                    strategy="wnc",
                    provenance=f"WNC id={row_dict['id']}",
                )
            )
            if max_examples is not None and len(examples) >= max_examples:
                break
    return examples


### 2.3 `biasneut.detector` — Stage-1 dual-head model, training, inference

In [ ]:
%%writefile /kaggle/working/biasneut/detector/__init__.py



In [ ]:
%%writefile /kaggle/working/biasneut/detector/infer.py
"""Detector inference wrapper — shared by the editor (span localization) and
the evaluation harness (independent Aggregate Bias Score classifier, §5.1,
§6.1). A *separately trained* instance must be used for evaluation than the
one used anywhere in training/filtering, to keep it independent (§4.3, §6.1).
"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoTokenizer

from biasneut.data.collate import ID2TAG
from biasneut.data.tokenize_utils import whitespace_tokenize
from biasneut.detector.model import BiasDetector


@dataclass
class DetectorPrediction:
    text: str
    is_biased: bool
    bias_prob: float
    word_tags: list[str]
    words: list[str]

    @property
    def biased_span_text(self) -> str | None:
        biased_words = [w for w, t in zip(self.words, self.word_tags) if t != "O"]
        return " ".join(biased_words) if biased_words else None


class DetectorInference:
    def __init__(self, model: BiasDetector, tokenizer, device: torch.device | None = None, max_length: int = 128):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device or next(model.parameters()).device
        self.max_length = max_length
        self.model.eval()

    @classmethod
    def from_pretrained(cls, model_dir: str | Path, backbone: str | None = None) -> "DetectorInference":
        """Reload a checkpoint written by ``biasneut.detector.train.save_pretrained``.

        Rebuilds the encoder architecture from the config.json saved
        alongside the fine-tuned weights (no redownload of the original
        pretrained backbone) unless ``backbone`` is explicitly given, e.g. to
        load an older checkpoint saved before config.json was persisted.
        """
        model_dir = Path(model_dir)
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        if backbone is not None:
            model = BiasDetector(backbone=backbone)
        else:
            encoder_config = AutoConfig.from_pretrained(model_dir)
            model = BiasDetector(encoder_config=encoder_config)
        state_dict = torch.load(model_dir / "pytorch_model.bin", map_location="cpu")
        model.load_state_dict(state_dict)
        return cls(model, tokenizer)

    @torch.no_grad()
    def predict(self, sentences: list[str]) -> list[DetectorPrediction]:
        word_lists = [whitespace_tokenize(s) for s in sentences]
        encoding = self.tokenizer(
            word_lists, is_split_into_words=True, truncation=True, padding=True,
            max_length=self.max_length, return_tensors="pt",
        ).to(self.device)

        out = self.model(input_ids=encoding["input_ids"], attention_mask=encoding["attention_mask"])
        sentence_probs = F.softmax(out.sentence_logits, dim=-1)[:, 1]
        token_preds = out.token_logits.argmax(-1)

        predictions = []
        for i, words in enumerate(word_lists):
            word_ids = encoding.word_ids(batch_index=i)
            word_tags = ["O"] * len(words)
            seen_words = set()
            for j, word_id in enumerate(word_ids):
                if word_id is None or word_id in seen_words or word_id >= len(words):
                    continue
                seen_words.add(word_id)
                word_tags[word_id] = ID2TAG[token_preds[i, j].item()]

            predictions.append(
                DetectorPrediction(
                    text=sentences[i],
                    is_biased=bool(sentence_probs[i].item() > 0.5),
                    bias_prob=sentence_probs[i].item(),
                    word_tags=word_tags,
                    words=words,
                )
            )
        return predictions

    def bias_score(self, sentence: str) -> float:
        """Convenience single-sentence scorer for eval/filtering call sites."""
        return self.predict([sentence])[0].bias_prob


In [ ]:
%%writefile /kaggle/working/biasneut/detector/metrics.py
"""Sentence-level and BIO span-level metrics for the detector.

Deliberately dependency-free (no seqeval) — span extraction and P/R/F1 are
~20 lines and this is the only place we need them.
"""
from __future__ import annotations

from biasneut.data.collate import ID2TAG


def sentence_prf1(preds: list[int], golds: list[int]) -> dict[str, float]:
    tp = sum(1 for p, g in zip(preds, golds) if p == 1 and g == 1)
    fp = sum(1 for p, g in zip(preds, golds) if p == 1 and g == 0)
    fn = sum(1 for p, g in zip(preds, golds) if p == 0 and g == 1)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = sum(1 for p, g in zip(preds, golds) if p == g) / len(preds) if preds else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy}


def extract_spans(tag_ids: list[int]) -> set[tuple[int, int]]:
    """Extract (start, end_exclusive) index pairs for each B-/I- run."""
    spans = set()
    start = None
    for i, tag_id in enumerate(tag_ids + [-100]):
        tag = ID2TAG.get(tag_id, "O") if tag_id != -100 else "O"
        if tag == "B-BIAS":
            if start is not None:
                spans.add((start, i))
            start = i
        elif tag == "I-BIAS":
            if start is None:
                start = i
        else:
            if start is not None:
                spans.add((start, i))
                start = None
    return spans


def span_prf1(pred_tags: list[list[int]], gold_tags: list[list[int]]) -> dict[str, float]:
    """Exact-match span F1, ignoring positions labeled -100 in the gold (they
    contribute no gold spans there, matching standard NER eval conventions)."""
    tp = fp = fn = 0
    for preds, golds in zip(pred_tags, gold_tags):
        # Mask predictions at ignored gold positions so padding/continuation
        # subwords never manufacture spurious spans.
        masked_preds = [p if g != -100 else -100 for p, g in zip(preds, golds)]
        pred_spans = extract_spans(masked_preds)
        gold_spans = extract_spans(golds)
        tp += len(pred_spans & gold_spans)
        fp += len(pred_spans - gold_spans)
        fn += len(gold_spans - pred_spans)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


In [ ]:
%%writefile /kaggle/working/biasneut/detector/model.py
"""Stage-1 detector (§5.1): shared encoder + sentence head + BIO token head.

Backbone defaults to POLITICS (`launch/POLITICS`, a roberta-base-scale
encoder pretrained by contrasting same-story articles across the ideological
spectrum — note this is roberta-BASE scale in the actual released checkpoint,
125M params, not roberta-large as sometimes described) with plain
DistilRoBERTa as a lighter fallback. Both are small enough to fully
fine-tune on a single free GPU (§5.1), so no PEFT is used here.
"""
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel, PretrainedConfig

from biasneut.data.collate import TAG2ID


@dataclass
class DetectorOutput:
    sentence_logits: torch.Tensor  # (batch, 2)
    token_logits: torch.Tensor  # (batch, seq_len, num_tags)
    loss: torch.Tensor | None = None
    sentence_loss: torch.Tensor | None = None
    token_loss: torch.Tensor | None = None


class BiasDetector(nn.Module):
    def __init__(
        self,
        backbone: str | None = "launch/POLITICS",
        encoder_config: PretrainedConfig | None = None,
        num_tags: int = len(TAG2ID),
        dropout: float = 0.1,
        sentence_loss_weight: float = 1.0,
        span_loss_weight: float = 1.0,
    ):
        """Build from a pretrained ``backbone`` name (downloads weights — the
        training-time path), or from an already-loaded ``encoder_config``
        (random-init encoder body, immediately overwritten by a fine-tuned
        state dict — the ``from_pretrained`` reload path, see
        ``biasneut.detector.infer.DetectorInference.from_pretrained``)."""
        super().__init__()
        if encoder_config is not None:
            config = encoder_config
            self.encoder = AutoModel.from_config(config)
        else:
            config = AutoConfig.from_pretrained(backbone)
            self.encoder = AutoModel.from_pretrained(backbone, config=config)
        hidden_size = config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.sentence_head = nn.Linear(hidden_size, 2)
        self.token_head = nn.Linear(hidden_size, num_tags)

        self.sentence_loss_weight = sentence_loss_weight
        self.span_loss_weight = span_loss_weight

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        sentence_labels: torch.Tensor | None = None,
        token_labels: torch.Tensor | None = None,
    ) -> DetectorOutput:
        encoded = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(encoded.last_hidden_state)

        pooled = sequence_output[:, 0, :]  # CLS / <s> token
        sentence_logits = self.sentence_head(pooled)
        token_logits = self.token_head(sequence_output)

        loss = sentence_loss = token_loss = None
        if sentence_labels is not None and token_labels is not None:
            sentence_loss = nn.functional.cross_entropy(sentence_logits, sentence_labels)
            token_loss = nn.functional.cross_entropy(
                token_logits.view(-1, token_logits.size(-1)),
                token_labels.view(-1),
                ignore_index=-100,
            )
            loss = self.sentence_loss_weight * sentence_loss + self.span_loss_weight * token_loss

        return DetectorOutput(
            sentence_logits=sentence_logits,
            token_logits=token_logits,
            loss=loss,
            sentence_loss=sentence_loss,
            token_loss=token_loss,
        )


In [ ]:
%%writefile /kaggle/working/biasneut/detector/train.py
"""Stage-1 detector training loop (§5.1, §7.4 checkpoint/resume discipline)."""
from __future__ import annotations

import logging
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from biasneut.common.checkpoint import CheckpointManager, TrainingState
from biasneut.common.compute import (
    autocast_context,
    build_optimizer,
    enable_gradient_checkpointing,
    get_device,
    resolve_dtype,
)
from biasneut.common.config import DetectorConfig
from biasneut.data.collate import DetectorCollator
from biasneut.data.schema import DetectionExample
from biasneut.detector.metrics import sentence_prf1, span_prf1
from biasneut.detector.model import BiasDetector

logger = logging.getLogger(__name__)


@torch.no_grad()
def evaluate(model: BiasDetector, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    all_sentence_preds, all_sentence_golds = [], []
    all_token_preds, all_token_golds = [], []
    total_loss, n_batches = 0.0, 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        total_loss += out.loss.item()
        n_batches += 1

        all_sentence_preds.extend(out.sentence_logits.argmax(-1).cpu().tolist())
        all_sentence_golds.extend(batch["sentence_labels"].cpu().tolist())
        all_token_preds.extend(out.token_logits.argmax(-1).cpu().tolist())
        all_token_golds.extend(batch["token_labels"].cpu().tolist())

    metrics = {"loss": total_loss / max(n_batches, 1)}
    metrics.update({f"sentence_{k}": v for k, v in sentence_prf1(all_sentence_preds, all_sentence_golds).items()})
    metrics.update({f"span_{k}": v for k, v in span_prf1(all_token_preds, all_token_golds).items()})
    model.train()
    return metrics


def train_detector(
    train_examples: list[DetectionExample],
    dev_examples: list[DetectionExample],
    cfg: DetectorConfig,
    resume: bool = True,
) -> BiasDetector:
    device = get_device()
    dtype = resolve_dtype(cfg.compute.mixed_precision)

    tokenizer = AutoTokenizer.from_pretrained(cfg.backbone)
    model = BiasDetector(
        backbone=cfg.backbone,
        sentence_loss_weight=cfg.sentence_loss_weight,
        span_loss_weight=cfg.span_loss_weight,
    ).to(device)
    if cfg.compute.gradient_checkpointing:
        enable_gradient_checkpointing(model.encoder)

    collator = DetectorCollator(tokenizer=tokenizer, max_length=cfg.max_seq_length)
    train_loader = DataLoader(train_examples, batch_size=cfg.train_batch_size, shuffle=True, collate_fn=collator)
    dev_loader = DataLoader(dev_examples, batch_size=cfg.eval_batch_size, shuffle=False, collate_fn=collator)

    optimizer = build_optimizer(model, cfg.compute, cfg.learning_rate, cfg.weight_decay)
    total_steps = len(train_loader) * cfg.num_epochs // max(cfg.compute.gradient_accumulation_steps, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(cfg.warmup_ratio * total_steps), num_training_steps=total_steps
    )

    ckpt_mgr = CheckpointManager(cfg.output_dir)
    start_epoch, skip_steps, global_step = 0, 0, 0
    if resume:
        state = ckpt_mgr.load(model, optimizer, scheduler, map_location=device)
        if state is not None:
            start_epoch = state.data_cursor.get("epoch", 0)
            skip_steps = state.data_cursor.get("step_in_epoch", 0)
            global_step = state.step
            logger.info("Resumed from step %d (epoch %d, step_in_epoch %d)", global_step, start_epoch, skip_steps)

    model.train()
    for epoch in range(start_epoch, cfg.num_epochs):
        for step, batch in enumerate(train_loader):
            if epoch == start_epoch and step < skip_steps:
                continue
            batch = {k: v.to(device) for k, v in batch.items()}

            with autocast_context(dtype):
                out = model(**batch)
                loss = out.loss / cfg.compute.gradient_accumulation_steps
            loss.backward()

            if (step + 1) % cfg.compute.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % cfg.checkpoint_every_steps == 0:
                    ckpt_mgr.save(
                        global_step, model, optimizer, scheduler,
                        TrainingState(step=global_step, epoch=epoch,
                                       data_cursor={"epoch": epoch, "step_in_epoch": step + 1}),
                    )

        skip_steps = 0
        metrics = evaluate(model, dev_loader, device)
        logger.info("epoch %d dev metrics: %s", epoch, metrics)
        ckpt_mgr.save(
            global_step, model, optimizer, scheduler,
            TrainingState(step=global_step, epoch=epoch + 1, data_cursor={"epoch": epoch + 1, "step_in_epoch": 0},
                          best_metric=metrics["span_f1"]),
        )

    return model


def save_pretrained(model: BiasDetector, tokenizer, output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), output_dir / "pytorch_model.bin")
    tokenizer.save_pretrained(output_dir)
    # Persist the encoder's own config so DetectorInference.from_pretrained
    # can rebuild the exact architecture without redownloading the original
    # pretrained backbone (see biasneut.detector.model.BiasDetector's
    # encoder_config path).
    model.encoder.config.save_pretrained(output_dir)


### 2.4 `biasneut.editor` — Stage-2 seq2seq editor, span marking, constrained decoding, strategies A/B/C, QLoRA stretch

In [ ]:
%%writefile /kaggle/working/biasneut/editor/__init__.py



In [ ]:
%%writefile /kaggle/working/biasneut/editor/collate.py
"""Editor training collation: mark the diff-derived biased span in the
source, tokenize (source, target) pairs for seq2seq training."""
from __future__ import annotations

from dataclasses import dataclass

import torch

from biasneut.data.schema import EditExample
from biasneut.editor.span_marking import diff_word_tags, mark_text


@dataclass
class EditorCollator:
    tokenizer: object
    max_source_length: int = 128
    max_target_length: int = 128
    task_prefix: str = ""

    def __call__(self, batch: list[EditExample]) -> dict[str, torch.Tensor]:
        marked_sources = []
        for ex in batch:
            tags = diff_word_tags(ex.source, ex.target)
            marked_sources.append(self.task_prefix + mark_text(ex.source, tags))

        model_inputs = self.tokenizer(
            marked_sources, truncation=True, padding=True, max_length=self.max_source_length, return_tensors="pt",
        )
        labels = self.tokenizer(
            text_target=[ex.target for ex in batch],
            truncation=True, padding=True, max_length=self.max_target_length, return_tensors="pt",
        )["input_ids"]
        labels[labels == self.tokenizer.pad_token_id] = -100

        model_inputs["labels"] = labels
        return model_inputs


In [ ]:
%%writefile /kaggle/working/biasneut/editor/constrained_decoding.py
"""Copy-biased decoding (§5.2): "bias the decoder toward copying non-flagged
tokens ... the operational core of Pryzant's inductive bias."

Honest scope note: Pryzant's own copy-heavy inductive bias comes from a
*trained* pointer-generator decoder (position-aware copy distribution),
which is architecturally tied to his custom LSTM decoder (`seq2seq/model.py`
`PointerSeq2Seq`) — we deliberately did not port that decoder forward
(§3.1: drop the concurrent model's machinery). Reimplementing a true
pointer-generator on top of a modern HF T5/BART decoder would require
surgery on the decoder's output projection, not just a logits processor.

What we implement instead is a decode-time *vocabulary bias*: at every
generation step, boost the logits of subword tokens that occur in the
**non-flagged** portion of the source sentence, so the model is nudged
toward reusing exact source vocabulary for content it wasn't told to edit.
This is a softer, position-agnostic approximation of the same inductive
bias — not a hard copy constraint — and is reported as such rather than
oversold as a faithful pointer-generator port.
"""
from __future__ import annotations

import torch
from transformers import LogitsProcessor, LogitsProcessorList

from biasneut.data.tokenize_utils import whitespace_tokenize


class CopyBiasLogitsProcessor(LogitsProcessor):
    """Boosts source-vocabulary logits, one allowed set per *source sentence*.

    ``generate()`` expands the batch to ``num_sources * num_beams`` rows
    (``repeat_interleave``, so row ``r`` belongs to source ``r // num_beams``).
    Indexing ``scores`` by source position instead would bias beam ``k`` of
    sentence ``i`` toward sentence ``i + k``'s vocabulary and leave every row
    past ``num_sources`` untouched — i.e. actively reward copying words from
    a *different* sentence.
    """

    def __init__(self, allowed_token_ids_per_batch: list[set[int]], bias_strength: float):
        self.allowed_token_ids_per_batch = allowed_token_ids_per_batch
        self.bias_strength = bias_strength
        self._index_cache: dict[int, torch.Tensor] = {}

    def _indices(self, source_idx: int, device: torch.device) -> torch.Tensor:
        """Cached id tensor per source — this runs at every decoding step."""
        cached = self._index_cache.get(source_idx)
        if cached is None or cached.device != device:
            allowed = self.allowed_token_ids_per_batch[source_idx]
            cached = torch.tensor(sorted(allowed), device=device, dtype=torch.long)
            self._index_cache[source_idx] = cached
        return cached

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        num_sources = len(self.allowed_token_ids_per_batch)
        if num_sources == 0:
            return scores

        num_rows = scores.shape[0]
        beams_per_source = max(1, num_rows // num_sources)
        for row in range(num_rows):
            source_idx = min(row // beams_per_source, num_sources - 1)
            if not self.allowed_token_ids_per_batch[source_idx]:
                continue
            idx = self._indices(source_idx, scores.device)
            scores[row, idx] = scores[row, idx] + self.bias_strength
        return scores


def non_flagged_token_ids(tokenizer, source_text: str, word_tags: list[str]) -> set[int]:
    """Subword vocab ids for words *outside* the flagged span in one source
    sentence (i.e. content the model should be biased toward copying)."""
    words = whitespace_tokenize(source_text)
    non_flagged_words = [w for w, t in zip(words, word_tags) if t == "O"]
    if not non_flagged_words:
        return set()
    ids = tokenizer(" ".join(non_flagged_words), add_special_tokens=False)["input_ids"]
    return set(ids)


def build_copy_bias_processor(
    tokenizer,
    sources: list[str],
    word_tags_per_source: list[list[str]],
    bias_strength: float = 2.5,
) -> LogitsProcessorList:
    allowed = [non_flagged_token_ids(tokenizer, src, tags) for src, tags in zip(sources, word_tags_per_source)]
    return LogitsProcessorList([CopyBiasLogitsProcessor(allowed, bias_strength)])


In [ ]:
%%writefile /kaggle/working/biasneut/editor/lewis.py
"""Strategy C, arm 2 (§3.4): LEWIS-style pseudo-parallel synthesis + generator
training.

Scope note: LEWIS's full architecture is a RoBERTa insert/replace/delete
*tagger* plus a BART *generator*, where the tagger is trained on labeled
edits and used to synthesize pseudo-parallel data at scale. We already have
a trained tagger — Stage 1's span detector — so we reuse it for masking
decisions instead of training a second bespoke tagger, and reuse the
Mask-and-Infill MLM (``biasneut.editor.mask_infill``) as the "style-specific
LM" that synthesizes the neutral side. The synthesized pairs then train the
*same* small seq2seq editor used by Strategies A/B via
``train_seq2seq_editor``, giving a real, working LEWIS-lite arm rather than a
from-scratch reimplementation of its tagger.
"""
from __future__ import annotations

import logging

from biasneut.data.schema import DetectionExample, EditExample
from biasneut.detector.infer import DetectorInference
from biasneut.editor.mask_infill import MaskInfiller

logger = logging.getLogger(__name__)


def synthesize_lewis_pairs(
    biased_examples: list[DetectionExample],
    detector: DetectorInference,
    infiller: MaskInfiller,
) -> list[EditExample]:
    """For each detector-flagged sentence, mask the predicted biased span(s)
    and infill with the (optionally neutral-domain-adapted) MLM, producing a
    synthetic (source, target) pseudo-parallel pair."""
    pairs = []
    predictions = detector.predict([ex.text for ex in biased_examples])
    for ex, pred in zip(biased_examples, predictions):
        if not pred.is_biased or all(t == "O" for t in pred.word_tags):
            continue
        target = infiller.infill(ex.text, pred.word_tags)
        if target.strip() == ex.text.strip():
            continue
        pairs.append(
            EditExample(
                source=ex.text,
                target=target,
                biased_span=pred.biased_span_text,
                strategy="unsupervised",
                provenance="lewis_lite: detector-masked + mlm-infilled",
            )
        )
    logger.info("Synthesized %d LEWIS-lite pseudo-parallel pairs from %d candidates",
                len(pairs), len(biased_examples))
    return pairs


In [ ]:
%%writefile /kaggle/working/biasneut/editor/mask_infill.py
"""Strategy C, arm 1 (§3.4, §4.2): fully unsupervised Mask-and-Infill.

Scope note vs. the literature this is adapted from: Malmi et al.'s Masker
selects mask positions from *disagreement between two domain-specific MLMs'
token likelihoods*. We already have a stronger, directly-supervised masking
signal from Stage 1 (a BABE/BASIL-trained span detector), so we use that to
choose *where* to mask, and reserve the MLM for *what* to infill — same
delete+infill shape as Masker/Mask-and-Infill, cheaper mask-selection step.
The infiller MLM is optionally domain-adapted by continuing its MLM
objective on human-labeled *neutral* sentences (BABE/BASIL label=0), nudging
its infill distribution toward neutral register — a lightweight stand-in for
Masker's "target-domain MLM."
"""
from __future__ import annotations

import logging

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling

from biasneut.common.compute import autocast_context, build_optimizer, get_device, resolve_dtype
from biasneut.common.config import ComputeConfig
from biasneut.data.tokenize_utils import whitespace_tokenize

logger = logging.getLogger(__name__)


class MaskInfiller:
    def __init__(self, backbone: str = "roberta-base"):
        self.tokenizer = AutoTokenizer.from_pretrained(backbone)
        self.model = AutoModelForMaskedLM.from_pretrained(backbone)
        self.model.eval()

    def to(self, device: torch.device) -> "MaskInfiller":
        self.model.to(device)
        return self

    @property
    def device(self) -> torch.device:
        return next(self.model.parameters()).device

    @torch.no_grad()
    def infill(self, sentence: str, word_tags: list[str], top_k: int = 1) -> str:
        """Replace each flagged word with the MLM's top prediction, filled in
        one joint forward pass (all mask positions predicted simultaneously,
        matching Wu et al. 2019's non-autoregressive infilling)."""
        words = whitespace_tokenize(sentence)
        mask_token = self.tokenizer.mask_token
        masked_words = [mask_token if tag != "O" else w for w, tag in zip(words, word_tags)]
        if mask_token not in masked_words:
            return sentence

        masked_text = " ".join(masked_words)
        encoding = self.tokenizer(masked_text, return_tensors="pt").to(self.device)
        logits = self.model(**encoding).logits[0]

        mask_positions = (encoding["input_ids"][0] == self.tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        predicted_ids = encoding["input_ids"][0].clone()
        for pos in mask_positions:
            top_id = logits[pos].topk(top_k).indices[-1]
            predicted_ids[pos] = top_id

        return self.tokenizer.decode(predicted_ids, skip_special_tokens=True)

    def finetune_on_neutral_corpus(
        self,
        neutral_sentences: list[str],
        compute_cfg: ComputeConfig | None = None,
        epochs: int = 1,
        batch_size: int = 16,
        learning_rate: float = 5e-5,
        mlm_probability: float = 0.15,
    ) -> None:
        """Continue the MLM objective on neutral-labeled sentences only, so
        the infiller's predictions lean toward neutral register."""
        compute_cfg = compute_cfg or ComputeConfig()
        device = get_device()
        dtype = resolve_dtype(compute_cfg.mixed_precision)
        self.model.to(device).train()

        encodings = [self.tokenizer(s, truncation=True, max_length=128) for s in neutral_sentences]
        collator = DataCollatorForLanguageModeling(self.tokenizer, mlm=True, mlm_probability=mlm_probability)
        loader = DataLoader(encodings, batch_size=batch_size, shuffle=True, collate_fn=collator)

        optimizer = build_optimizer(self.model, compute_cfg, learning_rate)
        for epoch in range(epochs):
            total_loss = 0.0
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with autocast_context(dtype):
                    out = self.model(**batch)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                total_loss += out.loss.item()
            logger.info("MLM neutral-domain adaptation epoch %d: loss=%.4f", epoch, total_loss / len(loader))

        self.model.eval()


In [ ]:
%%writefile /kaggle/working/biasneut/editor/qlora_decoder.py
"""Stretch editor arm (§5.2, §7.2, §7.3): QLoRA'd 1-3B instruction-tuned
decoder for higher fluency than the small seq2seq editors, still feasible on
a single free T4.

QLoRA = 4-bit NF4 quantization of the frozen base + double quantization +
LoRA adapters trained in bf16 + a paged optimizer (bitsandbytes). None of the
core pipeline needs this (§7.3) — it's optional and gated behind the
``qlora`` extra (``pip install -e ".[qlora]"``) so the rest of the codebase
has no hard bitsandbytes dependency.

Checkpointing here intentionally bypasses ``CheckpointManager``: saving a
full state_dict of a 4-bit-quantized base model is both wasteful and
fragile, whereas a LoRA adapter is a few tens of MB and PEFT already knows
how to (de)serialize it — so we save/resume just the adapter + step counter.
"""
from __future__ import annotations

import json
import logging
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from biasneut.common.config import QLoRAEditorConfig
from biasneut.data.schema import EditExample

logger = logging.getLogger(__name__)

INSTRUCTION_TEMPLATE = (
    "### Instruction:\n"
    "Rewrite the sentence to remove loaded political language while preserving its meaning.\n\n"
    "### Sentence:\n{source}\n\n### Neutralized:\n"
)


def _load_quantized_causal_lm(backbone: str):
    import bitsandbytes  # noqa: F401  (import error here is the clear failure point if extra not installed)
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    return AutoModelForCausalLM.from_pretrained(backbone, quantization_config=bnb_config, device_map="auto")


def build_qlora_model(cfg: QLoRAEditorConfig):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(cfg.backbone)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = _load_quantized_causal_lm(cfg.backbone)
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules),
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer


class _InstructionCollator:
    def __init__(self, tokenizer, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch: list[EditExample]) -> dict[str, torch.Tensor]:
        input_ids_list, labels_list = [], []
        for ex in batch:
            prompt = INSTRUCTION_TEMPLATE.format(source=ex.source)
            full_text = prompt + ex.target + self.tokenizer.eos_token

            prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
            full_ids = self.tokenizer(full_text, add_special_tokens=False, truncation=True,
                                       max_length=self.max_length)["input_ids"]

            labels = list(full_ids)
            prompt_len = min(len(prompt_ids), len(labels))
            for i in range(prompt_len):
                labels[i] = -100

            input_ids_list.append(full_ids)
            labels_list.append(labels)

        max_len = max(len(ids) for ids in input_ids_list)
        pad_id = self.tokenizer.pad_token_id
        input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
        labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
        for i, (ids, lbls) in enumerate(zip(input_ids_list, labels_list)):
            input_ids[i, : len(ids)] = torch.tensor(ids)
            attention_mask[i, : len(ids)] = 1
            labels[i, : len(lbls)] = torch.tensor(lbls)

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def _adapter_checkpoint_dir(output_dir: str | Path) -> Path:
    return Path(output_dir) / "adapter"


def train_qlora_editor(
    train_examples: list[EditExample],
    cfg: QLoRAEditorConfig,
    resume: bool = True,
) -> tuple:
    import bitsandbytes as bnb

    model, tokenizer = build_qlora_model(cfg)
    collator = _InstructionCollator(tokenizer, cfg.max_seq_length)
    loader = DataLoader(train_examples, batch_size=cfg.train_batch_size, shuffle=True, collate_fn=collator)

    optimizer = bnb.optim.PagedAdamW8bit(
        [p for p in model.parameters() if p.requires_grad], lr=cfg.learning_rate
    )

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    progress_path = output_dir / "progress.json"
    start_epoch, global_step = 0, 0

    if resume and _adapter_checkpoint_dir(output_dir).exists() and progress_path.exists():
        from peft import PeftModel

        model = PeftModel.from_pretrained(model, _adapter_checkpoint_dir(output_dir), is_trainable=True)
        progress = json.loads(progress_path.read_text())
        start_epoch, global_step = progress["epoch"], progress["step"]
        logger.info("Resumed QLoRA editor from epoch %d, step %d", start_epoch, global_step)

    model.train()
    for epoch in range(start_epoch, cfg.num_epochs):
        for step, batch in enumerate(loader):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss / cfg.gradient_accumulation_steps
            loss.backward()

            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                global_step += 1

        model.save_pretrained(_adapter_checkpoint_dir(output_dir))
        progress_path.write_text(json.dumps({"epoch": epoch + 1, "step": global_step}))
        logger.info("QLoRA editor epoch %d complete (step %d)", epoch, global_step)

    return model, tokenizer


In [ ]:
%%writefile /kaggle/working/biasneut/editor/seq2seq_model.py
"""Stage-2 primary editor (§5.2): T5-small/BART-base seq2seq over
span-marked input, with optional decode-time copy bias (§5.2, §7.3 — small
enough to fully fine-tune on a free GPU, no PEFT needed for this arm)."""
from __future__ import annotations

from pathlib import Path

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from biasneut.editor.constrained_decoding import build_copy_bias_processor
from biasneut.editor.span_marking import mark_text


class SeqEditor:
    def __init__(self, model, tokenizer, task_prefix: str = ""):
        self.model = model
        self.tokenizer = tokenizer
        self.task_prefix = task_prefix

    @classmethod
    def from_pretrained_backbone(cls, backbone: str = "t5-small") -> "SeqEditor":
        tokenizer = AutoTokenizer.from_pretrained(backbone)
        model = AutoModelForSeq2SeqLM.from_pretrained(backbone)
        task_prefix = "neutralize: " if backbone.startswith("t5") else ""
        editor = cls(model, tokenizer, task_prefix=task_prefix)
        editor._add_span_markers()
        return editor

    @classmethod
    def load(cls, model_dir: str | Path) -> "SeqEditor":
        model_dir = Path(model_dir)
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
        task_prefix = (model_dir / "task_prefix.txt").read_text() if (model_dir / "task_prefix.txt").exists() else ""
        return cls(model, tokenizer, task_prefix=task_prefix)

    def save(self, model_dir: str | Path) -> None:
        model_dir = Path(model_dir)
        model_dir.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(model_dir)
        self.tokenizer.save_pretrained(model_dir)
        (model_dir / "task_prefix.txt").write_text(self.task_prefix)

    def _add_span_markers(self, span_start: str = "<bias>", span_end: str = "</bias>") -> None:
        added = self.tokenizer.add_special_tokens({"additional_special_tokens": [span_start, span_end]})
        if added:
            self.model.resize_token_embeddings(len(self.tokenizer))

    def to(self, device: torch.device) -> "SeqEditor":
        self.model.to(device)
        return self

    @property
    def device(self) -> torch.device:
        return next(self.model.parameters()).device

    @torch.no_grad()
    def neutralize(
        self,
        sentences: list[str],
        word_tags_per_sentence: list[list[str]],
        num_beams: int = 4,
        max_new_tokens: int = 128,
        use_constrained_decoding: bool = True,
        copy_bias_strength: float = 2.5,
        no_repeat_ngram_size: int = 3,
    ) -> list[str]:
        marked = [
            self.task_prefix + mark_text(s, tags)
            for s, tags in zip(sentences, word_tags_per_sentence)
        ]
        inputs = self.tokenizer(marked, return_tensors="pt", padding=True, truncation=True, max_length=256).to(
            self.device
        )

        logits_processor = None
        if use_constrained_decoding:
            logits_processor = build_copy_bias_processor(
                self.tokenizer, sentences, word_tags_per_sentence, bias_strength=copy_bias_strength
            )

        # The copy-bias boost above pushes up every non-flagged token uniformly
        # at every step, which without a repetition guard lets beam search loop
        # on whichever boosted token scores highest (observed: real outputs
        # collapsing to "protest protest protest..."). no_repeat_ngram_size is
        # the standard HF safeguard against exactly this.
        output_ids = self.model.generate(
            **inputs,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens,
            logits_processor=logits_processor,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )
        return self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)


In [ ]:
%%writefile /kaggle/working/biasneut/editor/span_marking.py
"""Span marking: the interface between Stage-1 localization and Stage-2
editing. The editor always sees its input as a sentence with the flagged
span wrapped in marker tokens — whether that span came from a gold
biased/neutral diff (training, Strategies A/B) or from the Stage-1 detector's
predictions (inference, and Strategy C's mask-and-infill).
"""
from __future__ import annotations

import difflib

from biasneut.data.tokenize_utils import whitespace_tokenize


def diff_word_tags(source: str, target: str) -> list[str]:
    """BIO-tag source words that were deleted or replaced going source->target.

    This is how Strategy A/B training pairs (which have a real target but no
    explicit span annotation) get span supervision for the editor: the
    surface diff of a real biased->neutral rewrite *is* the span that
    mattered, which is exactly Pryzant's own single-word-subset framing.
    """
    src_words = whitespace_tokenize(source)
    tgt_words = whitespace_tokenize(target)
    matcher = difflib.SequenceMatcher(a=src_words, b=tgt_words)

    tags = ["O"] * len(src_words)
    for tag, i1, i2, _, _ in matcher.get_opcodes():
        if tag in ("delete", "replace"):
            for i in range(i1, i2):
                tags[i] = "I-BIAS" if i > i1 else "B-BIAS"
    return tags


def mark_span_from_tags(
    words: list[str],
    word_tags: list[str],
    span_start: str = "<bias>",
    span_end: str = "</bias>",
) -> str:
    """Wrap contiguous B-/I-BIAS runs in ``words`` with marker tokens."""
    out: list[str] = []
    in_span = False
    for word, tag in zip(words, word_tags):
        is_biased = tag != "O"
        if is_biased and not in_span:
            out.append(span_start)
        if not is_biased and in_span:
            out.append(span_end)
        out.append(word)
        in_span = is_biased
    if in_span:
        out.append(span_end)
    return " ".join(out)


def mark_text(text: str, word_tags: list[str], span_start: str = "<bias>", span_end: str = "</bias>") -> str:
    return mark_span_from_tags(whitespace_tokenize(text), word_tags, span_start, span_end)


In [ ]:
%%writefile /kaggle/working/biasneut/editor/train_common.py
"""Generic seq2seq fine-tuning loop shared by Strategy A (WNC pretrain +
adapt) and Strategy B (LLM-synthesized pseudo-parallel) — they differ only in
which `EditExample`s are passed in and whether a checkpoint is used to warm
start (§5.2, §7.4)."""
from __future__ import annotations

import logging

import torch
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

from biasneut.common.checkpoint import CheckpointManager, TrainingState
from biasneut.common.compute import (
    autocast_context,
    build_optimizer,
    enable_gradient_checkpointing,
    get_device,
    resolve_dtype,
)
from biasneut.common.config import EditorConfig
from biasneut.data.schema import EditExample
from biasneut.editor.collate import EditorCollator
from biasneut.editor.seq2seq_model import SeqEditor

logger = logging.getLogger(__name__)


@torch.no_grad()
def evaluate_loss(editor: SeqEditor, loader: DataLoader, device: torch.device) -> float:
    editor.model.eval()
    total, n = 0.0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = editor.model(**batch)
        total += out.loss.item()
        n += 1
    editor.model.train()
    return total / max(n, 1)


def train_seq2seq_editor(
    train_examples: list[EditExample],
    dev_examples: list[EditExample],
    cfg: EditorConfig,
    editor: SeqEditor | None = None,
    resume: bool = True,
) -> SeqEditor:
    if not train_examples:
        raise ValueError(
            "train_seq2seq_editor got 0 training examples. For Strategy C/LEWIS this "
            "usually means the detector flagged no biased spans in the candidate set "
            "(synthesize_lewis_pairs skips any sentence with an all-'O' prediction) — "
            "check the detector's span recall before assuming this is a data bug."
        )

    device = get_device()
    dtype = resolve_dtype(cfg.compute.mixed_precision)

    if editor is None:
        editor = SeqEditor.from_pretrained_backbone(cfg.backbone)
    editor.to(device)
    if cfg.compute.gradient_checkpointing:
        enable_gradient_checkpointing(editor.model)

    collator = EditorCollator(
        tokenizer=editor.tokenizer,
        max_source_length=cfg.max_source_length,
        max_target_length=cfg.max_target_length,
        task_prefix=editor.task_prefix,
    )
    train_loader = DataLoader(train_examples, batch_size=cfg.train_batch_size, shuffle=True, collate_fn=collator)
    dev_loader = DataLoader(dev_examples, batch_size=cfg.eval_batch_size, shuffle=False, collate_fn=collator)

    optimizer = build_optimizer(editor.model, cfg.compute, cfg.learning_rate)
    total_steps = len(train_loader) * cfg.num_epochs // max(cfg.compute.gradient_accumulation_steps, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(cfg.warmup_ratio * total_steps), num_training_steps=max(total_steps, 1)
    )

    ckpt_mgr = CheckpointManager(cfg.output_dir)
    start_epoch, skip_steps, global_step = 0, 0, 0
    if resume:
        state = ckpt_mgr.load(editor.model, optimizer, scheduler, map_location=device)
        if state is not None:
            start_epoch = state.data_cursor.get("epoch", 0)
            skip_steps = state.data_cursor.get("step_in_epoch", 0)
            global_step = state.step
            logger.info("Resumed editor from step %d (epoch %d)", global_step, start_epoch)

    editor.model.train()
    for epoch in range(start_epoch, cfg.num_epochs):
        for step, batch in enumerate(train_loader):
            if epoch == start_epoch and step < skip_steps:
                continue
            batch = {k: v.to(device) for k, v in batch.items()}

            with autocast_context(dtype):
                out = editor.model(**batch)
                loss = out.loss / cfg.compute.gradient_accumulation_steps
            loss.backward()

            if (step + 1) % cfg.compute.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(editor.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % cfg.checkpoint_every_steps == 0:
                    ckpt_mgr.save(
                        global_step, editor.model, optimizer, scheduler,
                        TrainingState(step=global_step, epoch=epoch,
                                       data_cursor={"epoch": epoch, "step_in_epoch": step + 1}),
                    )

        skip_steps = 0
        dev_loss = evaluate_loss(editor, dev_loader, device)
        logger.info("epoch %d dev loss: %.4f", epoch, dev_loss)
        ckpt_mgr.save(
            global_step, editor.model, optimizer, scheduler,
            TrainingState(step=global_step, epoch=epoch + 1, data_cursor={"epoch": epoch + 1, "step_in_epoch": 0},
                          best_metric=dev_loss),
        )

    return editor


In [ ]:
%%writefile /kaggle/working/biasneut/editor/train_strategy_a.py
"""Strategy A (§4.2): pretrain the editor on WNC, then optionally continue
training on in-domain news pairs (from Strategy B's pseudo-parallel data, or
Strategy C's synthesized pairs) for domain adaptation.

Run standalone, this measures how far pure Wikipedia transfer gets on news
(§6.3's "Pryzant off-the-shelf" baseline uses the pretrain-only checkpoint)."""
from __future__ import annotations

import logging

from biasneut.common.config import EditorConfig
from biasneut.data.schema import EditExample
from biasneut.editor.seq2seq_model import SeqEditor
from biasneut.editor.train_common import train_seq2seq_editor

logger = logging.getLogger(__name__)


def run_strategy_a(
    wnc_train: list[EditExample],
    wnc_dev: list[EditExample],
    cfg: EditorConfig,
    adapt_train: list[EditExample] | None = None,
    adapt_dev: list[EditExample] | None = None,
) -> SeqEditor:
    logger.info("Strategy A phase 1: WNC pretraining on %d pairs", len(wnc_train))
    editor = train_seq2seq_editor(wnc_train, wnc_dev, cfg)

    if adapt_train:
        logger.info("Strategy A phase 2: domain adaptation on %d in-domain pairs", len(adapt_train))
        adapt_cfg = EditorConfig(**{**cfg.__dict__, "output_dir": cfg.output_dir + "_adapted"})
        editor = train_seq2seq_editor(adapt_train, adapt_dev or wnc_dev, adapt_cfg, editor=editor)

    return editor


In [ ]:
%%writefile /kaggle/working/biasneut/editor/train_strategy_b.py
"""Strategy B (§4.2): train the editor on an LLM-synthesized pseudo-parallel
corpus (see ``biasneut.data.pseudo_parallel`` for generation + the mandatory
§4.3 circularity mitigations, which must be applied *before* pairs reach this
function — this module only trains on whatever it's given)."""
from __future__ import annotations

import logging

from biasneut.common.config import EditorConfig
from biasneut.data.schema import EditExample
from biasneut.editor.seq2seq_model import SeqEditor
from biasneut.editor.train_common import train_seq2seq_editor

logger = logging.getLogger(__name__)


def run_strategy_b(
    pseudo_train: list[EditExample],
    pseudo_dev: list[EditExample],
    cfg: EditorConfig,
    warm_start: SeqEditor | None = None,
) -> SeqEditor:
    logger.info("Strategy B: training on %d filtered LLM-synthesized pairs", len(pseudo_train))
    if not pseudo_train:
        raise ValueError("No pseudo-parallel pairs survived filtering — check §4.3 thresholds")
    return train_seq2seq_editor(pseudo_train, pseudo_dev, cfg, editor=warm_start)


In [ ]:
%%writefile /kaggle/working/biasneut/editor/train_strategy_c.py
"""Strategy C orchestration (§4.2): fully unsupervised editing, two arms.

Arm 1 (mask-and-infill) needs no seq2seq training at all — Stage 1's
detector + a (optionally neutral-domain-adapted) MLM infiller are sufficient
at inference time; see ``biasneut.editor.mask_infill.MaskInfiller.infill``.

Arm 2 (LEWIS-lite) uses arm 1 to synthesize pseudo-parallel pairs at scale,
then trains the shared seq2seq editor on them, giving a second, independently
inspectable Strategy-C variant with (likely) better fluency at the cost of
losing arm 1's "no training at all" property.
"""
from __future__ import annotations

import logging

from biasneut.common.config import EditorConfig
from biasneut.data.schema import DetectionExample
from biasneut.detector.infer import DetectorInference
from biasneut.editor.lewis import synthesize_lewis_pairs
from biasneut.editor.mask_infill import MaskInfiller
from biasneut.editor.seq2seq_model import SeqEditor
from biasneut.editor.train_common import train_seq2seq_editor

logger = logging.getLogger(__name__)


def run_strategy_c_lewis(
    biased_train: list[DetectionExample],
    biased_dev: list[DetectionExample],
    detector: DetectorInference,
    infiller: MaskInfiller,
    cfg: EditorConfig,
) -> SeqEditor:
    train_pairs = synthesize_lewis_pairs(biased_train, detector, infiller)
    dev_pairs = synthesize_lewis_pairs(biased_dev, detector, infiller)
    return train_seq2seq_editor(train_pairs, dev_pairs, cfg)


### 2.5 `biasneut.baselines` — trivial + LLM-zero-shot baselines

In [ ]:
%%writefile /kaggle/working/biasneut/baselines/__init__.py



In [ ]:
%%writefile /kaggle/working/biasneut/baselines/trivial.py
"""Mandatory honesty-check baselines (§6.3) — every one of these is annoyingly
easy to beat trivially and surprisingly hard to beat for real, which is
exactly why the proposal insists on reporting them.

Note on "Pryzant off-the-shelf (WNC-trained)": the original 2019 release
(``neutralizing-bias-master/``) is pinned to ``pytorch_pretrained_bert==0.3.0``
and ``torch==1.1.0`` and is not load-bearing-compatible with a modern stack
(see project survey). Rather than resurrect that environment, we substitute
our own Strategy-A editor *after WNC pretraining but before any news
adaptation* — i.e. ``train_strategy_a.run_strategy_a(..., adapt_train=None)``
— which answers the exact same question the baseline is meant to answer
("how far does pure Wikipedia transfer get on news?") using the same modern
architecture as the rest of this codebase, so the comparison stays apples to
apples with our own ablation arms.
"""
from __future__ import annotations

from biasneut.data.pseudo_parallel import NEUTRALIZE_PROMPT_TEMPLATE, LLMClient
from biasneut.data.tokenize_utils import whitespace_tokenize
from biasneut.detector.infer import DetectorInference
from biasneut.editor.seq2seq_model import SeqEditor


def copy_input_baseline(sentences: list[str]) -> list[str]:
    """Upper bound on preservation, zero transfer strength by construction."""
    return list(sentences)


def delete_flagged_word_baseline(sentences: list[str], word_tags_per_sentence: list[list[str]]) -> list[str]:
    """Pryzant's trivial editor: delete every token the detector flagged,
    keep everything else untouched."""
    outputs = []
    for sentence, tags in zip(sentences, word_tags_per_sentence):
        words = whitespace_tokenize(sentence)
        kept = [w for w, t in zip(words, tags) if t == "O"]
        outputs.append(" ".join(kept))
    return outputs


def delete_flagged_word_with_detector(sentences: list[str], detector: DetectorInference) -> list[str]:
    predictions = detector.predict(sentences)
    return delete_flagged_word_baseline(sentences, [p.word_tags for p in predictions])


def pryzant_offshelf_baseline(sentences: list[str], word_tags_per_sentence: list[list[str]],
                               wnc_pretrained_editor: SeqEditor, num_beams: int = 4) -> list[str]:
    """See module docstring: this *is* the WNC-pretrain-only Strategy-A
    checkpoint, used as the stand-in for Pryzant's original release."""
    return wnc_pretrained_editor.neutralize(
        sentences, word_tags_per_sentence, num_beams=num_beams, use_constrained_decoding=False,
    )


def llm_zero_shot_baseline(sentences: list[str], client: LLMClient,
                            prompt_template: str = NEUTRALIZE_PROMPT_TEMPLATE) -> list[str]:
    """Strong-but-not-free reference ceiling — contextualizes what a small,
    free-GPU system gives up (§6.3)."""
    return [client.generate(prompt_template.format(sentence=s)) for s in sentences]


### 2.6 `biasneut.eval` — transfer/preservation/fluency triad, significance testing, Pareto, harness

In [ ]:
%%writefile /kaggle/working/biasneut/eval/__init__.py



In [ ]:
%%writefile /kaggle/working/biasneut/eval/fluency.py
"""Fluency (§6.1, axis 3): perplexity under a fixed external LM + CoLA
grammaticality. "The guardrail that stops the model from 'neutralizing' by
mangling the sentence." The LM must stay fixed across all systems compared
(never the model being evaluated) so scores are comparable.
"""
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

from biasneut.common.compute import get_device


@dataclass
class FluencyResult:
    perplexities: list[float]
    grammatical_probs: list[float]


class FluencyScorer:
    def __init__(self, lm_name: str = "gpt2", grammaticality_model: str = "textattack/roberta-base-CoLA",
                 device: torch.device | None = None):
        self.device = device or get_device()

        self.lm_tokenizer = AutoTokenizer.from_pretrained(lm_name)
        self.lm = AutoModelForCausalLM.from_pretrained(lm_name).to(self.device).eval()

        self.cola_tokenizer = AutoTokenizer.from_pretrained(grammaticality_model)
        self.cola_model = AutoModelForSequenceClassification.from_pretrained(grammaticality_model).to(
            self.device
        ).eval()

    @torch.no_grad()
    def perplexity(self, sentence: str) -> float:
        if not sentence.strip():
            return float("inf")
        encoding = self.lm_tokenizer(sentence, return_tensors="pt").to(self.device)
        # A single-token sequence has no next-token position to score against,
        # so the causal-LM loss is NaN (observed from a degenerate one-token
        # editor output on real data) — treat it the same as empty: maximally
        # non-fluent, not a silent NaN that poisons the aggregate mean.
        if encoding["input_ids"].shape[1] < 2:
            return float("inf")
        out = self.lm(**encoding, labels=encoding["input_ids"])
        return torch.exp(out.loss).item()

    @torch.no_grad()
    def grammatical_prob(self, sentence: str) -> float:
        encoding = self.cola_tokenizer(sentence, return_tensors="pt", truncation=True).to(self.device)
        logits = self.cola_model(**encoding).logits
        probs = F.softmax(logits, dim=-1)[0]
        # textattack/roberta-base-CoLA: label 1 = acceptable/grammatical.
        return probs[1].item()

    def score(self, sentences: list[str]) -> FluencyResult:
        return FluencyResult(
            perplexities=[self.perplexity(s) for s in sentences],
            grammatical_probs=[self.grammatical_prob(s) for s in sentences],
        )


In [ ]:
%%writefile /kaggle/working/biasneut/eval/harness.py
"""Full evaluation harness (§6): orchestrates the triad + baselines + Pareto
+ significance testing into one report per experiment run.
"""
from __future__ import annotations

import json
import logging
from dataclasses import asdict, dataclass
from pathlib import Path

from biasneut.detector.infer import DetectorInference
from biasneut.eval.fluency import FluencyResult, FluencyScorer
from biasneut.eval.lexicon import LexiconBiasScorer
from biasneut.eval.pareto import ParetoPoint, plot_pareto
from biasneut.eval.preservation import PreservationResult, PreservationScorer
from biasneut.eval.significance import bootstrap_ci, mcnemar_test
from biasneut.eval.transfer_strength import TransferStrengthResult, aggregate_bias_score

logger = logging.getLogger(__name__)


@dataclass
class SystemEvalResult:
    name: str
    transfer: TransferStrengthResult
    preservation: PreservationResult
    fluency: FluencyResult

    def summary(self) -> dict:
        return {
            "name": self.name,
            "frac_neutral": self.transfer.frac_neutral,
            "mean_prob_drop": self.transfer.mean_prob_drop,
            "mean_bertscore_f1": sum(self.preservation.bertscore_f1) / len(self.preservation.bertscore_f1),
            "mean_sbert_cosine": sum(self.preservation.sbert_cosine) / len(self.preservation.sbert_cosine),
            "sari": self.preservation.sari,
            "bleu": self.preservation.bleu,
            "mean_perplexity": sum(self.fluency.perplexities) / len(self.fluency.perplexities),
            "mean_grammatical_prob": sum(self.fluency.grammatical_probs) / len(self.fluency.grammatical_probs),
        }


class EvaluationHarness:
    def __init__(
        self,
        independent_detector: DetectorInference,
        preservation_scorer: PreservationScorer,
        fluency_scorer: FluencyScorer,
        lexicon_scorer: LexiconBiasScorer | None = None,
    ):
        self.independent_detector = independent_detector
        self.preservation_scorer = preservation_scorer
        self.fluency_scorer = fluency_scorer
        self.lexicon_scorer = lexicon_scorer

    def evaluate_system(
        self, name: str, sources: list[str], outputs: list[str], references: list[str] | None = None,
    ) -> SystemEvalResult:
        transfer = aggregate_bias_score(sources, outputs, self.independent_detector, self.lexicon_scorer)
        preservation = self.preservation_scorer.score(sources, outputs, references)
        fluency = self.fluency_scorer.score(outputs)
        return SystemEvalResult(name=name, transfer=transfer, preservation=preservation, fluency=fluency)

    def evaluate_all(
        self, systems: dict[str, list[str]], sources: list[str], references: list[str] | None = None,
    ) -> dict[str, SystemEvalResult]:
        return {name: self.evaluate_system(name, sources, outputs, references) for name, outputs in systems.items()}


def compare_to_baseline(results: dict[str, SystemEvalResult], system_name: str, baseline_name: str) -> dict:
    """Significance tests for one system against one baseline (§6.5)."""
    system, baseline = results[system_name], results[baseline_name]

    system_correct = [prob < 0.5 for prob in system.transfer.output_probs]
    baseline_correct = [prob < 0.5 for prob in baseline.transfer.output_probs]
    mcnemar = mcnemar_test(system_correct, baseline_correct)

    sbert_mean, sbert_lo, sbert_hi = bootstrap_ci(system.preservation.sbert_cosine)

    return {
        "system": system_name,
        "baseline": baseline_name,
        "mcnemar_transfer_p_value": mcnemar.p_value,
        "mcnemar_statistic": mcnemar.statistic,
        "sbert_cosine_mean": sbert_mean,
        "sbert_cosine_ci": (sbert_lo, sbert_hi),
    }


def build_markdown_report(results: dict[str, SystemEvalResult]) -> str:
    header = "| System | Frac Neutral | Mean Prob Drop | SBERT Cos | BERTScore F1 | SARI | BLEU | Perplexity | P(grammatical) |\n"
    sep = "|---|---|---|---|---|---|---|---|---|\n"
    rows = []
    for name, r in results.items():
        s = r.summary()
        rows.append(
            f"| {name} | {s['frac_neutral']:.3f} | {s['mean_prob_drop']:.3f} | {s['mean_sbert_cosine']:.3f} | "
            f"{s['mean_bertscore_f1']:.3f} | {s['sari']:.2f} | {s['bleu']:.2f} | {s['mean_perplexity']:.1f} | "
            f"{s['mean_grammatical_prob']:.3f} |"
        )
    return header + sep + "\n".join(rows)


def run_full_evaluation(
    results: dict[str, SystemEvalResult],
    output_dir: str | Path,
    baseline_name: str = "copy_input",
) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    report = build_markdown_report(results)
    (output_dir / "report.md").write_text(report)

    comparisons = {
        name: compare_to_baseline(results, name, baseline_name)
        for name in results if name != baseline_name
    }
    (output_dir / "significance.json").write_text(json.dumps(comparisons, indent=2))

    points = [
        ParetoPoint(label=name, bias_reduction=r.transfer.mean_prob_drop,
                    preservation=sum(r.preservation.sbert_cosine) / len(r.preservation.sbert_cosine))
        for name, r in results.items()
    ]
    plot_pareto(points, str(output_dir / "pareto.png"))

    summaries = {name: r.summary() for name, r in results.items()}
    (output_dir / "summaries.json").write_text(json.dumps(summaries, indent=2))
    logger.info("Evaluation report written to %s", output_dir)


In [ ]:
%%writefile /kaggle/working/biasneut/eval/lexicon.py
"""Linguistic lexicon features (Recasens et al. 2013's word lists, as
released with Pryzant et al. 2020) — one of the few directly reusable
assets from the existing codebase (``neutralizing-bias-master/src/lexicons``).
Used as one signal inside the Aggregate Bias Score (§6.1), independent of any
trained classifier.
"""
from __future__ import annotations

from pathlib import Path

from biasneut.data.tokenize_utils import whitespace_tokenize

_LEXICON_FILES = {
    "assertives": "assertives_hooper1975.txt",
    "entailed_arg": "entailed_arg_berant2012.txt",
    "entailed": "entailed_berant2012.txt",
    "entailing_arg": "entailing_arg_berant2012.txt",
    "entailing": "entailing_berant2012.txt",
    "factives": "factives_hooper1975.txt",
    "hedges": "hedges_hyland2005.txt",
    "implicatives": "implicatives_karttunen1971.txt",
    "negative": "negative_liu2005.txt",
    "npov": "npov_lexicon.txt",
    "positive": "positive_liu2005.txt",
    "report_verbs": "report_verbs.txt",
    "strong_subjectives": "strong_subjectives_riloff2003.txt",
    "weak_subjectives": "weak_subjectives_riloff2003.txt",
}

# Lists that indicate loaded/subjective language, as opposed to purely
# descriptive resources (positive/negative, entailment lexicons) which we
# load for completeness but don't count toward the bias score.
_BIAS_INDICATIVE = {"assertives", "factives", "hedges", "implicatives", "strong_subjectives", "weak_subjectives"}


def load_lexicons(lexicon_dir: str | Path) -> dict[str, set[str]]:
    lexicon_dir = Path(lexicon_dir)
    lexicons = {}
    for name, filename in _LEXICON_FILES.items():
        path = lexicon_dir / filename
        if not path.exists():
            continue
        with open(path, encoding="utf-8", errors="ignore") as f:
            words = {line.strip().lower() for line in f if line.strip() and not line.startswith("#")}
        lexicons[name] = words
    return lexicons


class LexiconBiasScorer:
    def __init__(self, lexicon_dir: str | Path):
        self.lexicons = load_lexicons(lexicon_dir)
        self.bias_words: set[str] = set()
        for name in _BIAS_INDICATIVE:
            self.bias_words |= self.lexicons.get(name, set())

    def score(self, sentence: str) -> float:
        """Fraction of tokens in ``sentence`` found in a bias-indicative lexicon."""
        tokens = [t.strip('"“”‘’\'.,!?;:()[]').lower() for t in whitespace_tokenize(sentence)]
        if not tokens:
            return 0.0
        hits = sum(1 for t in tokens if t in self.bias_words)
        return hits / len(tokens)


In [ ]:
%%writefile /kaggle/working/biasneut/eval/pareto.py
"""Pareto frontier reporting (§6.2): "plot bias-reduction vs. preservation as
a Pareto frontier across decoding settings, rather than collapsing to one
number. The whole point of the task is the trade-off."
"""
from __future__ import annotations

from dataclasses import dataclass


@dataclass
class ParetoPoint:
    label: str
    bias_reduction: float  # higher is better
    preservation: float  # higher is better


def pareto_frontier(points: list[ParetoPoint]) -> list[ParetoPoint]:
    """Return the non-dominated points (higher-is-better on both axes)."""
    frontier = []
    for p in points:
        dominated = any(
            (q.bias_reduction >= p.bias_reduction and q.preservation >= p.preservation and
             (q.bias_reduction > p.bias_reduction or q.preservation > p.preservation))
            for q in points
        )
        if not dominated:
            frontier.append(p)
    return sorted(frontier, key=lambda p: p.bias_reduction)


def plot_pareto(points: list[ParetoPoint], output_path: str, title: str = "Bias reduction vs. preservation") -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    frontier = pareto_frontier(points)
    frontier_labels = {p.label for p in frontier}

    fig, ax = plt.subplots(figsize=(7, 5))
    xs = [p.bias_reduction for p in points]
    ys = [p.preservation for p in points]
    colors = ["#d62728" if p.label in frontier_labels else "#7f7f7f" for p in points]
    ax.scatter(xs, ys, c=colors, s=60, zorder=3)
    for p in points:
        ax.annotate(p.label, (p.bias_reduction, p.preservation), textcoords="offset points", xytext=(5, 5),
                    fontsize=8)

    if len(frontier) > 1:
        fx = [p.bias_reduction for p in frontier]
        fy = [p.preservation for p in frontier]
        ax.plot(fx, fy, "--", color="#d62728", alpha=0.6, zorder=2, label="Pareto frontier")
        ax.legend()

    ax.set_xlabel("Bias reduction (transfer strength)")
    ax.set_ylabel("Content preservation")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


In [ ]:
%%writefile /kaggle/working/biasneut/eval/preservation.py
"""Content preservation (§6.1, axis 2): BERTScore + SBERT cosine + SARI + BLEU.

"Preservation alone rewards the copy-input degenerate solution, so it is
never read in isolation" — this module only computes the numbers; joint
interpretation happens in ``eval.harness``.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import evaluate
import sacrebleu
from sentence_transformers import SentenceTransformer, util


@dataclass
class PreservationResult:
    bertscore_f1: list[float]
    sbert_cosine: list[float]
    sari: float
    bleu: float


class PreservationScorer:
    def __init__(self, sbert_model: str = "sentence-transformers/all-MiniLM-L6-v2",
                 bertscore_model: str = "roberta-large"):
        self._sbert = SentenceTransformer(sbert_model)
        self._bertscore_model = bertscore_model
        self._bertscore = evaluate.load("bertscore")
        self._sari = evaluate.load("sari")

    def score(self, sources: list[str], outputs: list[str], references: list[str] | None = None) -> PreservationResult:
        """``references`` are human-neutralized targets when available (e.g.
        held-out WNC pairs or a human-annotated subset); for arms without a
        gold target, pass ``sources`` again so SARI/BLEU degrade gracefully
        to measuring similarity against the input instead of a gold rewrite.
        """
        references = references or sources

        # A degenerate (empty-string) generation is a real, expected failure
        # mode for an undertrained seq2seq editor — it should score as a
        # preservation failure, not crash the whole evaluation run. bert_score's
        # own empty-string special case (bert_score/utils.py:sent_encode) calls
        # tokenizer.build_inputs_with_special_tokens([]) directly, which raises
        # AttributeError on some transformers/tokenizer version combinations;
        # substituting a non-empty placeholder sidesteps that path entirely.
        outputs = [o if o.strip() else "(no output generated)" for o in outputs]

        bert_out = self._bertscore.compute(
            predictions=outputs, references=references, model_type=self._bertscore_model
        )

        src_emb = self._sbert.encode(sources, convert_to_tensor=True)
        out_emb = self._sbert.encode(outputs, convert_to_tensor=True)
        sbert_cosine = util.pairwise_cos_sim(src_emb, out_emb).tolist()

        sari_result = self._sari.compute(
            sources=sources, predictions=outputs, references=[[r] for r in references]
        )
        bleu = sacrebleu.corpus_bleu(outputs, [references]).score

        return PreservationResult(
            bertscore_f1=bert_out["f1"],
            sbert_cosine=sbert_cosine,
            sari=sari_result["sari"],
            bleu=bleu,
        )


In [ ]:
%%writefile /kaggle/working/biasneut/eval/significance.py
"""Statistical rigor (§6.5): "≥3 seeds with mean ± std; paired significance
testing (McNemar for the classifier, bootstrap CIs for generation metrics)
against baselines. Report negative results."
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy import stats


@dataclass
class McNemarResult:
    statistic: float
    p_value: float
    n_a_only_correct: int
    n_b_only_correct: int


def mcnemar_test(system_a_correct: list[bool], system_b_correct: list[bool], exact_threshold: int = 25) -> McNemarResult:
    """Paired test on binary correctness (e.g. "classifier says neutral")
    between two systems on the same examples. Uses the exact binomial test
    for small discordant-pair counts (recommended when n01+n10 < 25) and the
    chi-square approximation otherwise — both are the standard McNemar
    variants; we don't pull in statsmodels for one test.
    """
    a_only = sum(1 for a, b in zip(system_a_correct, system_b_correct) if a and not b)
    b_only = sum(1 for a, b in zip(system_a_correct, system_b_correct) if b and not a)
    n_discordant = a_only + b_only

    if n_discordant == 0:
        return McNemarResult(statistic=0.0, p_value=1.0, n_a_only_correct=a_only, n_b_only_correct=b_only)

    if n_discordant < exact_threshold:
        p_value = stats.binomtest(min(a_only, b_only), n_discordant, 0.5).pvalue
        statistic = float(min(a_only, b_only))
    else:
        statistic = (abs(a_only - b_only) - 1) ** 2 / n_discordant
        p_value = 1 - stats.chi2.cdf(statistic, df=1)

    return McNemarResult(statistic=statistic, p_value=p_value, n_a_only_correct=a_only, n_b_only_correct=b_only)


def bootstrap_ci(values: list[float], n_samples: int = 1000, ci: float = 0.95, seed: int = 42) -> tuple[float, float, float]:
    """Return (point_estimate, ci_low, ci_high) for the mean of ``values``."""
    values_arr = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = np.empty(n_samples)
    n = len(values_arr)
    for i in range(n_samples):
        sample_idx = rng.integers(0, n, size=n)
        means[i] = values_arr[sample_idx].mean()
    alpha = (1 - ci) / 2
    lo, hi = np.quantile(means, [alpha, 1 - alpha])
    return float(values_arr.mean()), float(lo), float(hi)


@dataclass
class SeedAggregateResult:
    mean: float
    std: float
    per_seed: list[float]


def aggregate_across_seeds(per_seed_values: list[float]) -> SeedAggregateResult:
    arr = np.asarray(per_seed_values, dtype=float)
    return SeedAggregateResult(mean=float(arr.mean()), std=float(arr.std(ddof=1) if len(arr) > 1 else 0.0),
                                per_seed=list(per_seed_values))


In [ ]:
%%writefile /kaggle/working/biasneut/eval/transfer_strength.py
"""Transfer strength / Aggregate Bias Score (§6.1, axis 1).

"fraction of outputs flagged neutral by an independent BABE-trained
classifier, plus the mean drop in its bias probability (source -> output).
Independence from the training signal is essential; watch for Goodhart once
optimizing against it."

The ``detector`` passed in here MUST be a separately-trained instance from
any detector used elsewhere in the pipeline (training-time span localization,
Strategy-B filtering, Strategy-C masking) — see §4.3 / §5.1's "doubles as
evaluator" note. We don't enforce that at the type level (nothing stops a
caller from passing the same instance), so callers are responsible for
keeping two on-disk checkpoints and only ever loading the "_eval" one here.
"""
from __future__ import annotations

from dataclasses import dataclass

from biasneut.detector.infer import DetectorInference
from biasneut.eval.lexicon import LexiconBiasScorer


@dataclass
class TransferStrengthResult:
    frac_neutral: float
    mean_prob_drop: float
    source_probs: list[float]
    output_probs: list[float]

    @property
    def per_example_drop(self) -> list[float]:
        return [s - o for s, o in zip(self.source_probs, self.output_probs)]


def aggregate_bias_score(
    sources: list[str],
    outputs: list[str],
    independent_detector: DetectorInference,
    lexicon_scorer: LexiconBiasScorer | None = None,
) -> TransferStrengthResult:
    source_preds = independent_detector.predict(sources)
    output_preds = independent_detector.predict(outputs)

    source_probs = [p.bias_prob for p in source_preds]
    output_probs = [p.bias_prob for p in output_preds]

    if lexicon_scorer is not None:
        # Blend classifier probability with the lexicon signal (mean of the
        # two), matching the multi-signal spirit of §6.1 without collapsing
        # to a single opaque score computed by one model alone.
        output_probs = [
            0.5 * p + 0.5 * lexicon_scorer.score(o) for p, o in zip(output_probs, outputs)
        ]
        source_probs = [
            0.5 * p + 0.5 * lexicon_scorer.score(s) for p, s in zip(source_probs, sources)
        ]

    frac_neutral = sum(1 for p in output_preds if not p.is_biased) / len(output_preds) if output_preds else 0.0
    mean_prob_drop = (
        sum(s - o for s, o in zip(source_probs, output_probs)) / len(source_probs) if source_probs else 0.0
    )

    return TransferStrengthResult(
        frac_neutral=frac_neutral,
        mean_prob_drop=mean_prob_drop,
        source_probs=source_probs,
        output_probs=output_probs,
    )


### 2.7 Pryzant lexicon files

Reused as-is from `neutralizing-bias-master/src/lexicons/` (14 small word
lists totaling ~180KB) — embedded directly rather than fetched at runtime, so
this section has no dependency on the original GitHub repo staying up.

In [ ]:
%%writefile /kaggle/working/lexicons/assertives_hooper1975.txt
#########################################################################
# List of assertive verbs
# extracted from:
# Joan B. Hooper. 1975. On assertive predicates. In J. Kimball, editor,
# Syntax and Semantics, volume 4, pages 91–124. Academic Press, New York.
#########################################################################

think
believe
suppose
expect
imagine
guess
seem
appear
figure
acknowledge
admit
affirm
allege
answer
argue
assert
assure
certify
charge
claim
contend
declare
divulge
emphasize
explain
grant
guarantee
hint
hypothesize
imply
indicate
insist
intimate
maintain
mention
point out
predict
prophesy
postulate
remark
reply
report
say
state
suggest
swear
testify
theorize
verify
vow
write
agree
afraid
certain
sure
clear
obvious
evident
calculate
decide
deduce
estimate
hope
presume
surmise
suspect

In [ ]:
%%writefile /kaggle/working/lexicons/entailed_arg_berant2012.txt
evaluate
bill
recharge
feel
cheer
greet
swing
flick
navigate
exist
shut
evade
satisfy
detect
pardon
heighten
research
presume
surpass
jerk
communicate
light
prolong
warn
magnify
experience
set
unite
retire
hasten
stop
help
entail
adjust
move
blame
mitigate
like
overload
refund
measure
visualize
sing
fry
exhaust
inhibit
rehearse
prompt
arrive
enlist
exchange
spy
for
double
stroke
wanna
on
neglect
exacerbate
emphasize
represent
separate
scrape
unify
aggravate
yank
analyze
foresee
find
swap
steal
let
convert
surprise
devastate
fashion
articulate
demand
define
construct
t
prefer
design
chase
distinguish
spark
give
dont
disconnect
resume
discount
split
total
reorganize
determine
color
speed
underestimate
lease
warrant
symbolize
reach
discard
form
scour
renew
traverse
fulfil
assist
twist
suit
discipline
broaden
portray
reproduce
infect
survive
direct
enjoy
squeeze
flush
drop
bury
torment
reshape
cease
eat
undermine
spit
center
allege
exclude
snap
deliver
strip
remember
select
write
embroider
agree
tempt
restrict
dislike
in
tighten
skip
relish
ravage
implement
spring
stifle
position
cut
arouse
telephone
affect
scrap
enable
remove
commit
suppose
admire
stick
create
dip
promote
drive
add
cost
fix
treasure
top
embed
attach
obscure
lick
rape
dispute
boast
withstand
disappoint
terminate
apply
rename
belong
render
disguise
practice
curb
attack
crawl
attempt
soak
sketch
meet
phone
notice
circle
restore
hurt
charge
hug
populate
acknowledge
exhibit
complicate
piss
impose
criticise
rejuvenate
sow
collect
analyse
contravene
finish
publicize
discover
widen
weigh
speak
amplify
infuriate
creep
doubt
revise
elicit
protest
tax
alternate
catch
drown
decide
quantify
recognize
link
love
trust
achieve
heal
justify
envy
recall
jam
contest
revolutionize
conquer
imply
activate
defy
constrain
tag
star
renounce
compose
confirm
guarantee
pass
lighten
to
shout
entertain
roast
travel
inform
found
strengthen
operate
tell
beat
propose
smash
savor
authorize
sink
bar
outperform
sprinkle
grab
wake
explore
cherish
sacrifice
grow
formulate
ruin
open
diminish
e-mail
dye
model
practise
favor
regain
itch
recreate
stretch
repeat
extract
preserve
clean
donate
propagate
reassure
obey
commission
bother
leak
pick
imagine
enroll
bomb
empty
specialize
about
ditch
include
mimic
exclaim
embrace
credit
puzzle
regard
back
permeate
show
reminisce
ignore
pitch
administer
relax
slip
figure
wonder
broadcast
ship
hope
foster
leave
watch
wont
ensure
knock
think
release
divide
cover
relieve
accuse
upset
starve
verify
unplug
refinance
explain
spread
ban
decline
coordinate
utilize
update
frequent
see
compromise
avoid
brush
aid
weather
force
transfer
line
replicate
soften
surrender
deprive
elevate
yell
freeze
cooperate
induce
degrade
involve
present
push
invoke
refuse
categorize
stress
lean
cheat
build
mature
slay
prepare
breathe
brighten
die
provoke
propel
sustain
correspond
deter
lend
animate
expedite
minimize
voice
undertake
hinder
disable
renovate
stuff
wind
trim
accelerate
trash
suppress
reconsider
resist
rely
improve
intend
merge
shoot
decipher
duplicate
yield
weave
occupy
gotta
disprove
accumulate
resign
criticize
connect
invade
fire
raise
unveil
advertise
load
clinch
talk
institute
attain
throw
spot
reimburse
counter
harvest
multiply
ease
roll
suspend
estimate
disqualify
crank
guard
engulf
absorb
dissect
roam
discontinue
affirm
uphold
repair
scream
sum
confiscate
appreciate
liberate
generate
blunt
console
gather
intrigue
modify
sort
battle
plead
overturn
float
utilise
arise
pamper
expect
melt
picture
cook
place
trail
supply
wish
arrest
customize
combine
regulate
not
impede
evolve
bust
tire
fit
encode
praise
sign
sail
honor
post
feature
prevent
smile
forge
stab
reserve
qualify
disregard
consist
extend
code
contemplate
accommodate
harbor
strain
honour
shower
blacklist
end
discourage
enforce
disclose
don
lure
compete
astound
corrupt
dress
calm
point
interpret
oppress
betray
outline
adore
fabricate
encrypt
recycle
compound
thrust
weaken
correct
shorten
increase
score
compare
stock
state
narrow
offset
endow
streamline
lobby
pray
remain
contain
pronounce
ask
nurture
spend
harass
kiss
bend
compensate
reform
concern
empower
reinvent
deepen
accept
market
enclose
slash
fetch
signal
drink
organize
produce
slam
envision
worsen
defend
run
convene
congratulate
prove
react
vacate
violate
install
time
revisit
translate
mix
convince
salute
displace
seal
refine
clutch
call
assure
pursue
oversee
tolerate
please
document
fight
wait
overpower
integrate
sample
detest
scratch
behave
expel
hone
eliminate
lift
assert
bombard
erect
tip
parse
hold
threaten
buy
reap
perceive
precede
devote
burst
awaken
teach
fold
nominate
invite
forbid
explode
rest
accompany
recite
plug
omit
transform
disrupt
blast
part
ponder
pose
work
shed
respond
strive
support
unlock
hit
flip
recognise
trace
comprise
tear
compliment
massage
shrink
outweigh
compile
tuck
irritate
seek
live
put
ingest
signify
waive
prohibit
illustrate
distribute
emit
assess
tilt
program
conserve
try
fall
dispense
soothe
structure
treat
level
restrain
contaminate
flow
moderate
grill
declare
wield
stir
forecast
caress
amass
reread
participate
allow
glorify
distort
damage
fulfill
advance
emulate
haunt
ridicule
trick
persuade
shift
happen
harm
recover
rule
cleanse
relinquish
sharpen
slice
confront
lower
count
edit
rub
heed
feed
remodel
capture
from
pierce
lessen
favour
rectify
jump
mourn
kick
respect
emphasise
injure
view
motivate
gain
diss
diversify
change
rate
compel
instruct
wow
garner
publish
sift
hurry
occur
stage
invalidate
oppose
interfere
perform
exceed
refill
mark
project
exaggerate
misread
recomend
shatter
bear
assign
imitate
settle
guess
fling
didnt
begin
plan
control
outlive
comply
seduce
receive
devour
rent
revitalize
grace
remind
sponsor
transmit
name
ride
clear
enlighten
launch
deem
forget
emerge
drag
monitor
mention
stimulate
reprint
terrify
rally
log
hate
spoil
proclaim
reflect
start
screw
celebrate
rob
water
encourage
recount
lose
chart
admit
rouse
reuse
slow
swear
comprehend
indicate
trade
redistribute
uncover
concede
clog
suffer
poison
predate
hide
sound
alleviate
file
hack
review
last
replenish
oblige
disturb
deduce
reject
lack
dissolve
resemble
counteract
zip
climb
rewrite
scare
saturate
adopt
predict
forfeit
tailor
send
revoke
mandate
crack
torture
blog
entice
amuse
tap
stain
personalize
read
postpone
chair
screen
disperse
wander
mail
mirror
average
derive
license
announce
loose
seize
taste
cure
want
adapt
alter
outpace
condemn
assault
test
step
resent
expose
consider
loan
condone
stabilize
say
lay
possess
energize
echo
own
suck
tickle
continue
porn
reduce
taunt
differentiate
register
browse
slide
halt
pour
blow
worry
amaze
display
grieve
polish
commence
resell
cross
sport
depict
adorn
finance
reccomend
match
at
dampen
overtake
depend
abandon
circumvent
smack
remark
sway
shop
dismiss
field
desert
insult
mobilize
store
lead
punish
don't
interview
win
shape
fuck
touch
redo
suspect
dub
miss
focus
rebuild
breach
reveal
order
pop
elude
face
delight
mock
forgive
surf
request
insure
shave
carve
evoke
damn
elect
inject
nullify
conclude
sense
deserve
express
constitute
haul
bang
effect
misuse
fancy
curse
outlaw
study
award
free
label
drain
tick
record
deduct
engender
close
blink
switch
serve
reckon
mend
pee
schedule
carry
allocate
turn
stitch
cry
debate
reinstall
radiate
decorate
originate
educate
email
tweak
withdraw
map
bore
encircle
dismantle
contribute
bite
poke
summon
advise
chew
tour
visit
mount
pull
sneak
promise
challenge
flash
contradict
debut
pump
complete
smooth
wreck
purchase
note
repel
impress
wipe
wear
calculate
sweep
pause
overlook
trigger
sell
perfect
convey
merit
annoy
overwhelm
swallow
cater
lie
comfort
describe
alert
perpetuate
stalk
act
risk
demonstrate
kill
maintain
need
invest
obstruct
keep
amend
pollute
abuse
urge
advocate
stay
peel
wash
reward
discharge
compute
preach
outdo
standardize
flee
recommend
furnish
down
dispatch
manipulate
follow
tease
search
undo
escape
scrutinize
insist
cancel
strike
rearrange
incorporate
solicit
go
manufacture
book
grasp
leap
toss
decode
snatch
locate
track
handle
learn
interrupt
laugh
stand
audit
wrestle
hesitate
configure
penetrate
dump
disappear
shine
revive
grunt
comment
juggle
enhance
denote
shock
champion
withhold
replace
plot
wasnt
isnt
resize


In [ ]:
%%writefile /kaggle/working/lexicons/entailed_berant2012.txt
persuade
live
weaken
personalize
protect
picture
accept
hire
concentrate
receive
light
cock
condemn
sway
debate
recognize
proclaim
deepen
talk
practise
uncover
log
at
cool
form
comply
tolerate
reccomend
abuse
assist
scrub
devote
bury
clean
spit
interpret
praise
intensify
rotate
adjust
remind
collect
see
attach
mitigate
envision
expedite
settle
deprive
spell
invent
wont
speed
cut
fry
sink
center
trick
swim
renounce
level
assign
sense
explain
diss
chart
hesitate
acknowledge
raise
delete
supervise
crawl
exhaust
boast
illuminate
foster
seal
choose
refund
knit
pick
look
narrow
creep
suit
trim
accuse
resign
applaud
rehearse
taunt
differentiate
decide
conclude
amaze
unveil
deduce
separate
advise
reject
simulate
soothe
split
discard
imitate
sneak
observe
absorb
educate
comprise
fancy
reply
t
affect
ring
despise
spread
modify
mean
streamline
amass
duplicate
magnify
prolong
prove
shrink
spark
overtake
strip
spend
measure
associate
vacate
roll
toss
retrieve
charge
caution
retire
roast
regain
translate
fabricate
replicate
compound
elicit
repeat
outpace
overcome
slam
write
arouse
punch
emit
operate
star
open
cover
treasure
restrict
support
postpone
lean
renew
position
wash
scour
sample
tickle
hurry
structure
disconnect
elevate
push
increase
swallow
force
reform
imagine
analyse
move
redo
revere
impose
cheat
stretch
name
gather
own
outperform
blow
don
elude
control
come
obscure
propagate
return
explode
work
accommodate
damage
create
yield
forgive
for
blame
constitute
greet
hate
devour
speak
smash
state
favor
extract
reach
decline
adapt
top
recycle
unplug
mend
rest
agree
regulate
continue
overturn
seduce
permeate
direct
leap
carry
restore
endow
exhibit
amplify
happen
mix
resent
drain
disguise
summon
swell
destroy
occupy
disrupt
travel
ensure
aggravate
grill
tighten
don't
snatch
wonder
invalidate
portray
interfere
fulfill
disturb
infuriate
indicate
award
appraise
compensate
demonstrate
institute
rectify
stabilize
evoke
hack
commence
pierce
bother
in
reap
provoke
caress
enroll
permit
point
insure
reinvest
add
feel
amuse
encourage
discourage
sort
reassure
flow
run
endure
land
deduct
outdo
shoot
necessitate
engulf
tie
think
catch
place
visualize
pray
hasten
launch
harass
suggest
qualify
forge
surprise
shift
gotta
disappear
outlaw
read
resemble
irritate
declare
impart
exclude
recite
emphasise
want
brighten
gamble
stuff
wish
explore
crush
contravene
defer
animate
color
lie
parse
redistribute
starve
exclaim
sketch
cancel
inhale
part
allow
invade
throw
forfeit
nurture
originate
guarantee
adorn
react
honor
view
loosen
love
customize
survey
dismiss
respect
encode
last
perpetuate
itch
wake
figure
condone
incorporate
oppose
bear
dodge
grace
code
to
label
frequent
envelop
suppress
fit
fool
swap
stroke
surpass
help
break
doesnt
urge
grow
depict
distribute
wet
render
navigate
bang
treat
join
feed
authorize
compete
wield
restructure
oblige
stimulate
threaten
congratulate
astound
defy
free
chair
communicate
define
brush
dont
strengthen
spy
console
defeat
signify
consist
obey
suppose
dominate
voice
corrupt
assert
compare
encompass
integrate
select
conceive
tape
defend
consume
ponder
propel
extend
scare
stab
confiscate
overlook
prefer
transmit
initiate
fire
deliver
beat
inject
covet
lay
slash
rally
accelerate
reprint
freeze
call
air
find
execute
overload
research
damn
heed
scatter
like
fade
torment
bill
store
contradict
exchange
neglect
merge
achieve
arrest
enjoy
shed
copy
revoke
dip
revise
implement
ingest
assume
bring
sift
search
end
match
appoint
prepare
tap
respond
convey
bust
accumulate
convene
stop
upset
mark
patronize
simplify
project
favour
engage
stress
sum
inform
depend
didnt
omit
edit
breathe
torture
contribute
manage
screw
play
coach
erase
discuss
compute
describe
conceal
thrust
say
choke
demand
unify
mock
recommend
spill
reuse
isolate
round
recreate
ignore
line
capture
revolutionize
protest
plot
smile
mistake
recount
express
soak
confirm
piss
kick
adore
symbolize
discipline
yell
hone
put
chase
scream
relish
forecast
sweep
circle
excuse
recharge
desire
squeeze
press
confront
overthrow
bite
drop
balance
blink
impede
clutch
audit
recomend
eliminate
predate
inhabit
exemplify
forget
convert
wave
drag
reckon
disable
counter
foresee
bar
upgrade
invite
switch
halt
stick
echo
cleanse
chew
complicate
lessen
watch
mold
negate
degrade
avail
reinstall
broaden
provide
dismantle
burst
occur
check
ship
betray
chop
reward
knock
engender
turn
withstand
harvest
forsake
cost
mail
minimize
die
expect
skip
drive
rewrite
restrain
stay
retain
discover
refill
honour
resume
stifle
concern
exert
weave
hamper
subscribe
count
lose
block
scrutinize
delay
emulate
worry
relieve
fly
indulge
found
sign
recognise
appeal
study
mature
plug
shorten
curb
salute
estimate
mimic
depress
organize
regard
try
survive
criticise
categorize
divide
harbor
grab
expel
articulate
programme
offset
massage
decode
challenge
revisit
baffle
deter
emphasize
spot
preach
swing
worsen
dislike
pump
compile
wind
surf
validate
sponsor
order
from
let
persecute
descend
infect
dissolve
advance
envy
transport
certify
assault
sell
enforce
criticize
convince
recover
instruct
not
arise
quote
taste
alleviate
champion
establish
cheer
grant
rejoin
sabotage
spring
hit
enhance
reconstruct
soften
participate
disprove
decorate
jump
shape
appear
pamper
escape
fulfil
poison
raid
include
designate
undergo
screen
publish
swear
conquer
field
compliment
dispatch
complete
shatter
cross
scrape
undertake
boost
broadcast
elaborate
supply
wipe
grieve
shout
tailor
have
blunt
sacrifice
slip
understand
earn
show
correct
multiply
resurrect
assess
tilt
comfort
recall
accompany
clear
fall
anger
awaken
combine
rob
penetrate
rain
sail
stock
learn
repair
guess
sew
enable
grasp
remodel
on
secure
re-read
bombard
invoke
savor
injure
flush
hurt
pause
double
file
stalk
coordinate
erect
trust
underestimate
hug
specify
rejuvenate
gain
misunderstand
satisfy
entice
preserve
dress
warrant
repel
set
predict
register
slay
expose
amend
dispute
tax
slide
enclose
pop
request
relax
relinquish
counteract
step
suspect
abandon
tuck
violate
elect
revive
endanger
mirror
affirm
replace
remember
resell
mobilize
remain
review
telephone
effect
pass
porn
prohibit
cooperate
reserve
nourish
aid
book
cook
insist
schedule
delude
reveal
analyze
scrap
concede
refine
mask
handle
exist
relate
rely
flip
annoy
grunt
enlighten
devastate
automate
meet
prompt
heighten
behave
allocate
stir
withdraw
disregard
displace
peel
miss
juggle
deem
heal
leave
possess
resist
shut
motivate
oversee
discontinue
celebrate
smack
pound
announce
tire
bore
lengthen
lend
reconsider
flash
snap
exceed
maintain
transform
head
improve
fold
undermine
stage
fascinate
curse
pursue
trace
ditch
trigger
admit
release
dispense
manipulate
imply
drown
battle
post
loose
furnish
forbid
reinvent
laugh
sow
zip
email
acquire
begin
equip
hope
loan
tease
hinder
trade
assure
detect
cater
calculate
lease
face
pronounce
reread
heat
construct
lead
haul
overpower
lack
signal
tag
puzzle
advocate
refresh
surrender
uninstall
derive
comprehend
topple
change
crank
perceive
climb
waste
sing
mount
feature
ridicule
trail
wait
score
alter
contaminate
disperse
outweigh
weather
fashion
touch
benefit
channel
link
send
interrupt
contemplate
lift
trash
total
finish
blast
steer
devise
suck
test
reopen
dump
spin
attack
invest
dye
admire
decipher
focus
pose
mention
lighten
spare
grind
value
seize
notice
locate
close
decrease
delight
frighten
moderate
kiss
perfect
flick
merit
give
draw
empty
roam
about
commission
need
pull
insult
fling
attempt
liberate
span
outlast
replenish
crack
entertain
plan
distract
revitalize
slice
wander
mourn
strive
go
pollute
rouse
tick
regenerate
denote
jam
exacerbate
tear
evolve
fuck
dedicate
shock
lick
haunt
win
bow
follow
influence
glorify
involve
pitch
misread
eat
e-mail
pioneer
sound
unlock
contest
ruin
risk
reflect
visit
rent
strain
deserve
jerk
reminisce
drink
embed
induce
shave
refuse
track
credit
sustain
garner
steal
nullify
spawn
reorganize
encrypt
service
attain
please
spoil
polish
saturate
harm
limit
exaggerate
ravage
hand
enlist
debut
refinance
diminish
breach
uphold
misinterpret
bomb
ban
precede
calm
display
blog
manufacture
inspect
lure
wow
design
enact
average
specialize
stain
stand
finance
frame
model
pretend
wreck
pee
buy
conserve
water
tempt
browse
suffer
down
insert
act
interview
circumvent
blacklist
doubt
reshape
rub
carve
obstruct
aint
wasnt
isnt
resize


In [ ]:
%%writefile /kaggle/working/lexicons/entailing_arg_berant2012.txt
presume
charge
reload
regret
entertain
proclaim
brush
discard
sail
outlive
seal
elude
perform
notice
condone
compound
seek
dare
exceed
yank
dictate
illuminate
extend
alert
expand
smell
specify
relate
swim
accelerate
quit
benefit
imply
tweak
beckon
accuse
minimize
wish
drown
reimburse
assess
anticipate
speak
bake
consume
shrink
wield
propel
arise
recruit
trade
thank
bid
duplicate
excite
didnt
love
discount
degrade
stress
exemplify
screen
wipe
associate
worsen
jerk
analyse
counter
reclaim
alternate
lift
allow
convince
amaze
ravage
pretend
invent
require
qualify
manufacture
not
favor
plan
trail
label
initiate
commission
wanna
undo
restart
neglect
shatter
number
rearrange
set
decorate
hurry
forecast
rent
guarantee
embody
access
fling
organise
search
rejuvenate
cross
meet
aid
rotate
fit
punish
widen
picture
reprint
corrupt
exist
project
adore
thrust
water
roll
plunge
harass
mold
end
waste
weaken
pour
focus
plead
obstruct
crack
yield
deprive
rename
clean
spell
dye
accomplish
seduce
confine
despise
house
shove
jump
savor
ruin
enforce
skip
curb
point
log
fuel
ring
endanger
trash
customize
edit
devote
guess
envy
award
spawn
disguise
encourage
train
supply
need
allocate
idolize
outweigh
insert
saturate
hide
thrill
murder
argue
sport
abide
infuriate
discontinue
employ
dub
rely
generate
conceal
inspect
dampen
pin
float
tackle
rob
outline
fill
telephone
observe
encircle
renew
smash
bill
stuff
inhabit
appreciate
demonstrate
sabotage
pee
illustrate
sue
revitalize
scatter
gather
survey
collapse
complete
circulate
rediscover
diversify
count
talk
enlarge
anger
bend
suspect
structure
stir
insist
emphasise
swap
grab
overload
resolve
matter
realise
embed
cherish
hope
attain
repair
pardon
admire
withdraw
choke
challenge
alter
hijack
necessitate
spark
weather
limit
track
feature
waive
straighten
sustain
warrant
spend
enlighten
pay
scream
amuse
boil
round
warn
institute
honor
run
inform
whip
nominate
demolish
devour
empower
engulf
censor
top
spray
catch
awaken
look
worship
renounce
turn
devise
wreck
unlock
mitigate
email
stain
simulate
exercise
spare
recover
massage
reinvent
lengthen
reiterate
interpret
resume
wash
volunteer
sprinkle
insure
heed
equip
irritate
shed
dial
suppose
enlist
bombard
drain
elaborate
blog
dump
integrate
compete
cry
soak
attribute
evade
conceive
endorse
penetrate
express
resell
slice
erect
inject
loosen
slide
compute
contemplate
transmit
transfer
trust
nurture
forget
regard
gotta
clinch
criticise
reckon
liberate
initialize
report
alleviate
condemn
concern
clog
validate
relax
finance
forgive
offer
forbid
refuse
form
engage
disable
shop
fear
threaten
puzzle
persuade
assist
digest
intend
transport
trace
multiply
undermine
concede
peel
energize
exclaim
suspend
collaborate
solicit
exacerbate
entail
stick
urge
evoke
eradicate
group
tax
tilt
delude
found
masturbate
dispatch
cook
emit
lighten
dissolve
invoke
remind
uphold
consist
terrify
loan
renovate
zip
lessen
formulate
risk
preclude
update
refurbish
squeeze
wrap
shave
expedite
scrap
interfere
override
defy
depend
help
aim
confuse
buck
rewrite
repeat
rejoin
wind
attack
doesnt
refute
assemble
sacrifice
bump
tie
populate
discharge
pinch
disclose
foster
flood
secure
sing
pioneer
achieve
unload
acknowledge
repay
fancy
reap
copy
relieve
cast
postpone
underestimate
modify
spin
safeguard
dive
assimilate
parse
wander
terminate
swear
confirm
mend
shout
average
retake
recognise
release
render
grunt
calm
store
captivate
simplify
categorize
wrestle
allege
revoke
crawl
utilize
occur
spoil
wait
fade
foresee
heat
target
chase
furnish
like
halt
revolutionize
blend
influence
don't
hinder
differentiate
part
bore
suit
encrypt
glorify
contract
navigate
reinforce
mail
star
impede
laugh
fake
poke
book
dispose
dismiss
lie
prolong
encode
ponder
accomodate
cheer
accentuate
refund
cater
circumvent
conclude
restrain
apply
uncover
wear
tempt
model
baffle
poison
trim
soothe
lay
translate
persecute
trick
document
revere
construct
compress
reuse
deepen
hug
soften
juggle
slash
exaggerate
hit
examine
carry
respond
tighten
neutralize
reminisce
standardize
beg
lobby
proceed
incorporate
term
expect
nourish
enact
taunt
smack
bow
decide
revamp
care
authorize
recreate
heighten
adopt
frequent
dispense
reduce
regain
indicate
ingest
arrange
grow
bang
visualize
aggravate
oppose
function
summon
praise
test
disobey
overturn
contravene
reinstall
schedule
relive
misread
mistake
convert
honour
sow
close
polish
snatch
phone
reassure
prosecute
supervise
afford
delight
detail
combat
await
overwhelm
contain
try
loathe
rub
act
fire
facilitate
decline
steal
protest
bomb
pitch
lean
dissect
nurse
jeopardize
purchase
derive
behold
predate
commit
bust
rouse
memorize
knit
appoint
request
outlaw
burst
sniff
escort
recount
connect
disregard
change
interrupt
nail
explode
wonder
present
chew
spill
react
annoy
improve
open
score
toss
realize
dot
constitute
draw
congratulate
tickle
diminish
outpace
amass
stitch
frustrate
appear
experience
freeze
swallow
denote
forfeit
ban
exude
publicize
inhale
rate
tip
hack
guard
taste
flow
administer
comfort
restructure
handle
dismantle
intimidate
pollute
fall
chair
roast
announce
exert
elicit
judge
itch
compile
re-read
programme
let
applaud
misrepresent
erode
satisfy
garner
mimic
stalk
mock
deceive
sweep
switch
arrive
offend
fetch
overlook
scrub
resign
center
chop
face
depress
weigh
cancel
scratch
obey
deplete
redo
seize
inherit
stab
decrease
save
reconcile
leap
grieve
dance
covet
emphasize
crush
watch
inspire
advance
insult
stretch
enable
value
oblige
teach
impress
gain
issue
equate
speed
happen
host
disprove
fight
hamper
execute
surpass
replenish
note
channel
amplify
misinterpret
reccomend
split
return
confer
emulate
compensate
think
flick
wont
commence
design
command
outlast
cooperate
entice
submit
relish
lease
regenerate
adorn
prompt
fund
broadcast
reference
educate
clarify
light
establish
consider
misplace
originate
haul
oversee
name
induce
reproduce
scare
mobilize
record
resent
pamper
hail
mirror
begin
fuck
respect
narrow
mow
boost
echo
exit
map
confess
ignite
recycle
delay
affirm
perceive
signal
exchange
distribute
expose
surrender
jam
stand
infect
precede
obscure
go
ease
register
program
estimate
survive
refill
compel
distract
stop
eliminate
miss
melt
complement
offset
rush
mount
decode
frighten
infuse
assume
promise
forgo
adapt
ship
vacate
prescribe
clutch
oppress
about
lure
harbor
cock
embroider
transform
reread
defer
moderate
sort
propagate
roam
post
reflect
total
harvest
certify
declare
relinquish
capture
sample
heal
entrust
tailor
personalize
outdo
denounce
kidnap
prove
escalate
detest
reveal
deem
contribute
justify
rebuild
quote
place
own
revive
recite
possess
blow
caress
remain
mature
shorten
tease
specialize
file
surf
slip
arrest
rain
suck
trouble
astound
exhibit
mourn
refine
start
endow
span
boast
envision
distort
refinance
recomend
resemble
patronize
stifle
overcome
belong
desert
blacklist
kiss
comment
swing
market
omit
answer
forsake
diss
rally
accumulate
conduct
discipline
dont
question
bag
reshape
slam
slap
animate
doubt
cite
lack
concentrate
reconstruct
break
unveil
rank
down
dodge
confront
please
ridicule
excuse
browse
petition
assure
clip
craft
provoke
curse
choose
enclose
contradict
cease
live
merit
convey
comprise
drag
screw
impact
abuse
depict
damn
prohibit
sift
grant
unleash
pronounce
empty
click
tape
withhold
breed
shut
flash
shift
service
bet
cure
piss
time
stock
designate
produce
sew
yell
dispute
install
shine
color
emerge
coach
e-mail
tune
recall
plot
tire
circle
level
lend
pierce
pound
convene
notify
grind
visit
tour
climb
tear
creep
predict
contest
blink
negate
introduce
blast
finish
strip
destroy
acquire
mark
unravel
wow
scrape
don
solve
sway
hesitate
dominate
activate
deal
enrich
show
deserve
bother
overpower
drink
shower
correspond
console
identify
caution
disperse
haunt
behave
gamble
display
defeat
hold
demand
dedicate
block
retain
reform
contaminate
misuse
press
preach
continue
smooth
cleanse
salute
expel
figure
advocate
outperform
centralize
escape
commend
provide
pop
slay
relay
pursue
bug
scour
unzip
flee
audit
assert
delete
hammer
reorganize
include
stimulate
sound
fabricate
reserve
comprehend
spring
flip
decipher
snap
pinpoint
tag
address
instruct
disrupt
smile
suffer
scrutinize
battle
pause
upgrade
infiltrate
ditch
die
radiate
resurrect
blunt
remark
balance
optimize
signify
stun
back
deduct
succeed
chart
isnt
wasnt
size


In [ ]:
%%writefile /kaggle/working/lexicons/entailing_berant2012.txt
sweep
experience
undermine
customize
conserve
foresee
adopt
employ
center
recover
enlighten
lean
denounce
brush
argue
recognise
glorify
heighten
chat
rehearse
unveil
tape
frequent
delay
comprise
embark
resist
recomend
kidnap
facilitate
dry
commit
descend
deprive
rain
translate
hail
deepen
spin
tune
prompt
gather
visit
hang
decrease
anticipate
hurry
locate
smooth
compose
articulate
not
expedite
clip
heat
dip
focus
deceive
crawl
forfeit
dismantle
designate
regret
pardon
insult
map
pause
realise
energize
cancel
oversee
swim
discharge
star
light
upgrade
close
nominate
repay
embroider
cleanse
matter
squeeze
sneak
subscribe
dress
condemn
waive
proclaim
exert
bug
elaborate
resolve
discontinue
astound
uncover
end
categorize
caution
decline
compete
comfort
explode
flip
model
bow
remark
assimilate
coach
incorporate
envision
disobey
wont
impair
telephone
loathe
stretch
privatize
imply
centralize
collect
double
expand
broaden
test
minimize
activate
overcome
peel
enact
evaluate
unify
overturn
cast
discard
regain
bill
alert
conclude
develop
retrieve
sketch
manufacture
correspond
intensify
provoke
recite
surpass
overpower
bang
draft
lick
hit
radiate
command
boast
reopen
encompass
cherish
entitle
back
defer
tilt
refresh
target
scratch
search
try
massage
hasten
revamp
undo
boost
jerk
tout
ingest
ride
restructure
bet
compare
relocate
refurbish
idolize
inflict
fool
widen
mention
compile
possess
outdo
suck
wanna
evade
operate
happen
conceal
spray
generate
shatter
volunteer
admit
burn
hack
rejoin
warrant
fish
criticize
integrate
compliment
despise
relish
sing
crush
comment
prefer
detail
spend
reject
tailor
outpace
misuse
loosen
repair
inhale
fit
prescribe
remain
vacate
entice
penetrate
send
skip
stock
enrich
reminisce
show
enlarge
arise
blend
detest
empty
obscure
convene
originate
damage
inhabit
interrupt
circle
assure
degrade
fry
shove
dont
snap
ponder
refute
stick
preclude
perform
excite
anger
mark
grant
assess
pinpoint
display
establish
tweak
reconstruct
elicit
program
begin
stab
annoy
win
derive
evoke
tickle
enlist
afford
head
secure
endow
lend
sense
cheer
advertise
deter
group
predict
fall
function
weather
arrive
withhold
outweigh
film
require
plunge
draw
reckon
grab
lay
picture
beg
speed
commence
torment
propose
disable
pretend
wow
switch
doesnt
die
censor
memorize
match
frustrate
announce
impose
bar
reiterate
highlight
entrust
lighten
risk
tighten
desire
seek
smash
break
suspend
reinvent
decorate
consist
dissect
circulate
choke
shout
outline
delete
sniff
burst
let
occupy
unload
forgo
nourish
emerge
forgive
misrepresent
respond
reimburse
watch
relax
worship
change
inspect
disregard
celebrate
code
exist
propagate
caress
ignite
compute
excuse
include
shield
yield
elude
adorn
encode
service
pronounce
market
engulf
inhibit
complement
delude
alternate
thrill
exceed
aid
avoid
didnt
oblige
turn
illustrate
publicize
pee
return
sail
slam
handle
link
waste
empower
imitate
exercise
structure
compel
induce
endanger
applaud
dodge
abide
unleash
moderate
impart
heal
persuade
don
practice
distinguish
soothe
demolish
denote
lie
heed
imagine
rule
formulate
poison
affirm
analyze
hug
constitute
pick
counter
log
combat
deplete
contemplate
inspire
lengthen
terrify
pound
capture
govern
spare
broadcast
trigger
expel
portray
trump
dispense
admire
place
shorten
mobilize
zip
multiply
postpone
ring
transmit
obtain
accomplish
jam
plot
curse
obey
sow
wish
dissolve
assault
load
accumulate
disprove
shift
dismiss
join
attain
halt
revere
spill
reproduce
reverse
sink
reorganize
utilize
drink
presume
stimulate
survey
scour
terminate
agree
assign
roast
tip
react
familiarize
part
store
summon
quit
trouble
note
shop
automate
channel
favor
apply
emphasize
number
warn
rely
gotta
neglect
erase
shine
persecute
add
run
fuck
dare
educate
edit
negotiate
figure
discover
hide
stuff
state
overload
necessitate
unite
ridicule
commend
shed
avail
initialize
refine
renovate
document
dot
confiscate
personalize
swear
chart
laugh
confuse
echo
depict
thrust
uphold
wet
boot
treasure
exemplify
punish
exclaim
perfect
freeze
tag
yell
savor
judge
swing
encircle
condone
reload
crave
narrow
urge
face
topple
poke
roll
juggle
flash
suppress
retake
scrutinize
reccomend
eradicate
precede
populate
embody
leave
convince
beckon
haul
think
modify
visualize
fancy
ban
promote
request
exchange
stop
pose
recount
purify
grind
soften
mitigate
preview
hammer
contradict
worsen
utilise
uninstall
diversify
furnish
level
bust
come
verify
transport
cheat
slice
salute
blame
spawn
grow
endure
reap
desert
involve
post
diminish
fade
wait
spring
aim
decipher
intimidate
proceed
nurse
rediscover
replenish
refund
restart
dampen
intrigue
wield
wind
hinder
resemble
weigh
exit
obstruct
audit
slow
simulate
interest
alleviate
escape
pamper
co-founded
ratify
realize
plead
parse
masturbate
connect
slash
stifle
reward
underestimate
emulate
discourage
organise
deem
taste
outlast
sue
classify
isolate
entertain
hook
fire
devise
cooperate
counteract
itch
rewrite
contest
relive
puzzle
depress
tax
remember
advocate
sway
exacerbate
resent
blast
assemble
wrap
download
confirm
fund
regulate
inherit
allow
oppress
dive
corrupt
melt
garner
covet
transfer
license
rape
deal
engage
bump
capitalize
reclaim
learn
don't
occur
cite
sample
relieve
endorse
pioneer
amuse
award
render
curb
jump
score
optimize
rebuild
sabotage
programme
negate
distort
leap
mold
count
spoil
flee
execute
infect
pin
contain
mandate
live
sort
issue
assert
seize
answer
congratulate
murder
finish
split
span
rub
tackle
steal
air
evolve
cater
scrub
mourn
scream
redesign
feature
grunt
resign
slide
comprehend
advance
express
discipline
reprint
revive
value
cock
calm
withdraw
escort
clarify
flow
infuriate
wake
mix
grip
infiltrate
pour
steer
carry
instruct
mend
suppose
stitch
transform
perceive
animate
s
clinch
reread
bomb
stir
devour
harm
water
injure
hesitate
copy
register
haunt
arrange
pitch
hope
estimate
embrace
set
revitalize
earn
misread
fulfill
seduce
resume
shave
justify
unravel
specify
promise
file
dispatch
reveal
reform
debate
emphasise
slip
frame
down
petition
unzip
sacrifice
offset
escalate
doubt
forsake
sew
contravene
choose
resurrect
jeopardize
reflect
insist
succeed
spy
reinvest
rally
continue
critique
phone
impact
transcend
mature
outlaw
collaborate
deduct
nurture
assist
schedule
bake
protest
time
trick
expose
sift
trash
surrender
polish
mistake
address
last
associate
depend
toss
irritate
blunt
illuminate
ravage
simplify
fill
motivate
compound
act
confer
fuel
recall
omit
float
strip
lack
reconcile
consider
cure
forget
own
withstand
seal
remind
supervise
cease
respect
kiss
pierce
declare
bore
tire
avert
loan
revoke
blink
crack
top
keep
alter
point
hone
knit
weaken
host
captivate
damn
prosecute
line
dump
spark
adapt
liberate
inject
relay
entail
clog
struggle
mail
renounce
provide
prohibit
recreate
smile
scatter
rush
e-mail
purchase
start
commission
trust
contribute
taunt
impede
swap
ensure
notice
shower
solve
rent
confront
label
re-read
boil
fabricate
conduct
drag
overwhelm
miss
indicate
trail
amaze
preach
renew
concede
reinstall
harvest
behold
overthrow
enable
comply
grasp
mount
consume
talk
climb
sport
mock
track
expect
punch
revolutionize
pollute
sprinkle
outperform
digest
forge
slap
help
project
relate
ship
disrupt
wonder
pinch
scrap
average
complete
nail
navigate
voice
blow
prove
unlock
ditch
rotate
raid
dedicate
stain
envy
decode
concentrate
present
reference
brighten
save
chew
rate
conquer
detect
determine
shut
shrink
please
scare
chair
smack
improve
fling
deserve
hand
tie
drain
wander
blog
authorize
demand
tear
diss
flick
undertake
free
cross
outlive
refill
snatch
harbor
design
upset
regenerate
chop
roam
grieve
bother
sound
contaminate
awaken
click
rejuvenate
signify
drown
aggravate
pluck
specialize
screw
devote
stun
stroke
stalk
bag
bend
pursue
dictate
praise
need
erect
misinterpret
feed
email
institute
go
open
challenge
wreck
gamble
mirror
tour
lease
accentuate
merit
yank
resell
swallow
insert
solicit
trim
deny
chase
attribute
validate
gain
lift
invalidate
breed
rearrange
compress
serve
achieve
delight
indulge
round
infuse
offend
confine
magnify
devastate
give
procure
introduce
finance
lessen
violate
sustain
misunderstand
bombard
forbid
predate
console
demonstrate
about
construct
examine
tempt
impress
plan
pass
total
differentiate
dye
suffer
refinance
enclose
mimic
fake
ruin
piss
acquire
preserve
mow
lure
surf
stand
pop
embed
behave
question
blacklist
circumvent
suspect
shape
scrape
craft
block
isnt
aint
wasnt
size


In [ ]:
%%writefile /kaggle/working/lexicons/factives_hooper1975.txt
#########################################################################
# List of factive verbs
# extracted from:
# Joan B. Hooper. 1975. On assertive predicates. In J. Kimball, editor,
# Syntax and Semantics, volume 4, pages 91–124. Academic Press, New York.
#########################################################################

know
realize
regret
forget
find out
discover
learn
note
notice
observe
perceive
recall
remember
reveal
see
resent
amuse
suffice
bother
make sense
care
odd
strange
interesting
relevant
sorry
exciting

In [ ]:
%%writefile /kaggle/working/lexicons/hedges_hyland2005.txt
#########################################################################
# List of hedges
# extracted from:
# Ken Hyland. 2005. Metadiscourse: Exploring Interaction in Writing.
# Continuum, London and New York.
#########################################################################

about
almost
apparent
apparently
appear
appeared
appears
approximately
around
assume
assumed
certain amount
certain extent
certain level
claim
claimed
could
couldn't
doubt
doubtful
essentially
estimate
estimated
feel
felt
frequently
from our perspective
generally
guess
in general
in most cases
in most instances
in our view
indicate
indicated
largely
likely
mainly
may
maybe
might
mostly
often
on the whole
ought
perhaps
plausible
plausibly
possible
possibly
postulate
postulated
presumable
probable
probably
relatively
roughly
seems
should
sometimes
somewhat
suggest
suggested
suppose
suspect
tend to
tends to
typical
typically
uncertain
uncertainly
unclear
unclearly
unlikely
usually
would
broadly
tended to
presumably
suggests
from this perspective
from my perspective
in my view
in this view
in our opinion
in my opinion
to my knowledge
fairly
quite
rather
argue
argues
argued
claims
feels
indicates
supposed
supposes
suspects
postulates

In [ ]:
%%writefile /kaggle/working/lexicons/implicatives_karttunen1971.txt
#########################################################################
# List of implicative verbs
# extracted from:
# Lauri Karttunen. 1971. Implicative verbs. Language, 47(2):340–358.
#########################################################################

manage
remember
bother
get
dare
care
venture
condescend
happen
fit
careful
misfortune
sense
succeed
deign
forget
fail
neglect
decline
avoid
refrain
choose
able
can
cause
force
prevent
preclude
keep
allow
hesitate
attempt

In [ ]:
%%writefile /kaggle/working/lexicons/negative_liu2005.txt
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
; 
; Opinion Lexicon: Negative
;
; This file contains a list of NEGATIVE opinion words (or sentiment words).
;
; This file and the papers can all be downloaded from 
;    http://www.cs.uic.edu/~liub/FBS/sentiment-analysis.html
;
; If you use this list, please cite the following paper:
;
;   Minqing Hu and Bing Liu. "Mining and Summarizing Customer Reviews." 
;       Proceedings of the ACM SIGKDD International Conference on Knowledge 
;       Discovery and Data Mining (KDD-2004), Aug 22-25, 2004, Seattle, 
;       Washington, USA, 
;
; Notes: 
;    1. The appearance of an opinion word in a sentence does not necessarily  
;       mean that the sentence expresses a positive or negative opinion. 
;       See the paper below:
;
;       Bing Liu. "Sentiment Analysis and Subjectivity." An chapter in 
;          Handbook of Natural Language Processing, Second Edition, 
;          (editors: N. Indurkhya and F. J. Damerau), 2010.
;
;    2. You will notice many misspelled words in the list. They are not 
;       mistakes. They are included as these misspelled words appear 
;       frequently in social media content. 
;
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;

2-faced
2-faces
abnormal
abolish
abominable
abominably
abominate
abomination
abort
aborted
aborts
abrade
abrasive
abrupt
abruptly
abscond
absence
absent-minded
absentee
absurd
absurdity
absurdly
absurdness
abuse
abused
abuses
abusive
abysmal
abysmally
abyss
accidental
accost
accursed
accusation
accusations
accuse
accuses
accusing
accusingly
acerbate
acerbic
acerbically
ache
ached
aches
achey
aching
acrid
acridly
acridness
acrimonious
acrimoniously
acrimony
adamant
adamantly
addict
addicted
addicting
addicts
admonish
admonisher
admonishingly
admonishment
admonition
adulterate
adulterated
adulteration
adulterier
adversarial
adversary
adverse
adversity
afflict
affliction
afflictive
affront
afraid
aggravate
aggravating
aggravation
aggression
aggressive
aggressiveness
aggressor
aggrieve
aggrieved
aggrivation
aghast
agonies
agonize
agonizing
agonizingly
agony
aground
ail
ailing
ailment
aimless
alarm
alarmed
alarming
alarmingly
alienate
alienated
alienation
allegation
allegations
allege
allergic
allergies
allergy
aloof
altercation
ambiguity
ambiguous
ambivalence
ambivalent
ambush
amiss
amputate
anarchism
anarchist
anarchistic
anarchy
anemic
anger
angrily
angriness
angry
anguish
animosity
annihilate
annihilation
annoy
annoyance
annoyances
annoyed
annoying
annoyingly
annoys
anomalous
anomaly
antagonism
antagonist
antagonistic
antagonize
anti-
anti-american
anti-israeli
anti-occupation
anti-proliferation
anti-semites
anti-social
anti-us
anti-white
antipathy
antiquated
antithetical
anxieties
anxiety
anxious
anxiously
anxiousness
apathetic
apathetically
apathy
apocalypse
apocalyptic
apologist
apologists
appal
appall
appalled
appalling
appallingly
apprehension
apprehensions
apprehensive
apprehensively
arbitrary
arcane
archaic
arduous
arduously
argumentative
arrogance
arrogant
arrogantly
ashamed
asinine
asininely
asinininity
askance
asperse
aspersion
aspersions
assail
assassin
assassinate
assault
assult
astray
asunder
atrocious
atrocities
atrocity
atrophy
attack
attacks
audacious
audaciously
audaciousness
audacity
audiciously
austere
authoritarian
autocrat
autocratic
avalanche
avarice
avaricious
avariciously
avenge
averse
aversion
aweful
awful
awfully
awfulness
awkward
awkwardness
ax
babble
back-logged
back-wood
back-woods
backache
backaches
backaching
backbite
backbiting
backward
backwardness
backwood
backwoods
bad
badly
baffle
baffled
bafflement
baffling
bait
balk
banal
banalize
bane
banish
banishment
bankrupt
barbarian
barbaric
barbarically
barbarity
barbarous
barbarously
barren
baseless
bash
bashed
bashful
bashing
bastard
bastards
battered
battering
batty
bearish
beastly
bedlam
bedlamite
befoul
beg
beggar
beggarly
begging
beguile
belabor
belated
beleaguer
belie
belittle
belittled
belittling
bellicose
belligerence
belligerent
belligerently
bemoan
bemoaning
bemused
bent
berate
bereave
bereavement
bereft
berserk
beseech
beset
besiege
besmirch
bestial
betray
betrayal
betrayals
betrayer
betraying
betrays
bewail
beware
bewilder
bewildered
bewildering
bewilderingly
bewilderment
bewitch
bias
biased
biases
bicker
bickering
bid-rigging
bigotries
bigotry
bitch
bitchy
biting
bitingly
bitter
bitterly
bitterness
bizarre
blab
blabber
blackmail
blah
blame
blameworthy
bland
blandish
blaspheme
blasphemous
blasphemy
blasted
blatant
blatantly
blather
bleak
bleakly
bleakness
bleed
bleeding
bleeds
blemish
blind
blinding
blindingly
blindside
blister
blistering
bloated
blockage
blockhead
bloodshed
bloodthirsty
bloody
blotchy
blow
blunder
blundering
blunders
blunt
blur
bluring
blurred
blurring
blurry
blurs
blurt
boastful
boggle
bogus
boil
boiling
boisterous
bomb
bombard
bombardment
bombastic
bondage
bonkers
bore
bored
boredom
bores
boring
botch
bother
bothered
bothering
bothers
bothersome
bowdlerize
boycott
braggart
bragger
brainless
brainwash
brash
brashly
brashness
brat
bravado
brazen
brazenly
brazenness
breach
break
break-up
break-ups
breakdown
breaking
breaks
breakup
breakups
bribery
brimstone
bristle
brittle
broke
broken
broken-hearted
brood
browbeat
bruise
bruised
bruises
bruising
brusque
brutal
brutalising
brutalities
brutality
brutalize
brutalizing
brutally
brute
brutish
bs
buckle
bug
bugging
buggy
bugs
bulkier
bulkiness
bulky
bulkyness
bull****
bull----
bullies
bullshit
bullshyt
bully
bullying
bullyingly
bum
bump
bumped
bumping
bumpping
bumps
bumpy
bungle
bungler
bungling
bunk
burden
burdensome
burdensomely
burn
burned
burning
burns
bust
busts
busybody
butcher
butchery
buzzing
byzantine
cackle
calamities
calamitous
calamitously
calamity
callous
calumniate
calumniation
calumnies
calumnious
calumniously
calumny
cancer
cancerous
cannibal
cannibalize
capitulate
capricious
capriciously
capriciousness
capsize
careless
carelessness
caricature
carnage
carp
cartoonish
cash-strapped
castigate
castrated
casualty
cataclysm
cataclysmal
cataclysmic
cataclysmically
catastrophe
catastrophes
catastrophic
catastrophically
catastrophies
caustic
caustically
cautionary
cave
censure
chafe
chaff
chagrin
challenging
chaos
chaotic
chasten
chastise
chastisement
chatter
chatterbox
cheap
cheapen
cheaply
cheat
cheated
cheater
cheating
cheats
checkered
cheerless
cheesy
chide
childish
chill
chilly
chintzy
choke
choleric
choppy
chore
chronic
chunky
clamor
clamorous
clash
cliche
cliched
clique
clog
clogged
clogs
cloud
clouding
cloudy
clueless
clumsy
clunky
coarse
cocky
coerce
coercion
coercive
cold
coldly
collapse
collude
collusion
combative
combust
comical
commiserate
commonplace
commotion
commotions
complacent
complain
complained
complaining
complains
complaint
complaints
complex
complicated
complication
complicit
compulsion
compulsive
concede
conceded
conceit
conceited
concen
concens
concern
concerned
concerns
concession
concessions
condemn
condemnable
condemnation
condemned
condemns
condescend
condescending
condescendingly
condescension
confess
confession
confessions
confined
conflict
conflicted
conflicting
conflicts
confound
confounded
confounding
confront
confrontation
confrontational
confuse
confused
confuses
confusing
confusion
confusions
congested
congestion
cons
conscons
conservative
conspicuous
conspicuously
conspiracies
conspiracy
conspirator
conspiratorial
conspire
consternation
contagious
contaminate
contaminated
contaminates
contaminating
contamination
contempt
contemptible
contemptuous
contemptuously
contend
contention
contentious
contort
contortions
contradict
contradiction
contradictory
contrariness
contravene
contrive
contrived
controversial
controversy
convoluted
corrode
corrosion
corrosions
corrosive
corrupt
corrupted
corrupting
corruption
corrupts
corruptted
costlier
costly
counter-productive
counterproductive
coupists
covetous
coward
cowardly
crabby
crack
cracked
cracks
craftily
craftly
crafty
cramp
cramped
cramping
cranky
crap
crappy
craps
crash
crashed
crashes
crashing
crass
craven
cravenly
craze
crazily
craziness
crazy
creak
creaking
creaks
credulous
creep
creeping
creeps
creepy
crept
crime
criminal
cringe
cringed
cringes
cripple
crippled
cripples
crippling
crisis
critic
critical
criticism
criticisms
criticize
criticized
criticizing
critics
cronyism
crook
crooked
crooks
crowded
crowdedness
crude
cruel
crueler
cruelest
cruelly
cruelness
cruelties
cruelty
crumble
crumbling
crummy
crumple
crumpled
crumples
crush
crushed
crushing
cry
culpable
culprit
cumbersome
cunt
cunts
cuplrit
curse
cursed
curses
curt
cuss
cussed
cutthroat
cynical
cynicism
d*mn
damage
damaged
damages
damaging
damn
damnable
damnably
damnation
damned
damning
damper
danger
dangerous
dangerousness
dark
darken
darkened
darker
darkness
dastard
dastardly
daunt
daunting
dauntingly
dawdle
daze
dazed
dead
deadbeat
deadlock
deadly
deadweight
deaf
dearth
death
debacle
debase
debasement
debaser
debatable
debauch
debaucher
debauchery
debilitate
debilitating
debility
debt
debts
decadence
decadent
decay
decayed
deceit
deceitful
deceitfully
deceitfulness
deceive
deceiver
deceivers
deceiving
deception
deceptive
deceptively
declaim
decline
declines
declining
decrement
decrepit
decrepitude
decry
defamation
defamations
defamatory
defame
defect
defective
defects
defensive
defiance
defiant
defiantly
deficiencies
deficiency
deficient
defile
defiler
deform
deformed
defrauding
defunct
defy
degenerate
degenerately
degeneration
degradation
degrade
degrading
degradingly
dehumanization
dehumanize
deign
deject
dejected
dejectedly
dejection
delay
delayed
delaying
delays
delinquency
delinquent
delirious
delirium
delude
deluded
deluge
delusion
delusional
delusions
demean
demeaning
demise
demolish
demolisher
demon
demonic
demonize
demonized
demonizes
demonizing
demoralize
demoralizing
demoralizingly
denial
denied
denies
denigrate
denounce
dense
dent
dented
dents
denunciate
denunciation
denunciations
deny
denying
deplete
deplorable
deplorably
deplore
deploring
deploringly
deprave
depraved
depravedly
deprecate
depress
depressed
depressing
depressingly
depression
depressions
deprive
deprived
deride
derision
derisive
derisively
derisiveness
derogatory
desecrate
desert
desertion
desiccate
desiccated
desititute
desolate
desolately
desolation
despair
despairing
despairingly
desperate
desperately
desperation
despicable
despicably
despise
despised
despoil
despoiler
despondence
despondency
despondent
despondently
despot
despotic
despotism
destabilisation
destains
destitute
destitution
destroy
destroyer
destruction
destructive
desultory
deter
deteriorate
deteriorating
deterioration
deterrent
detest
detestable
detestably
detested
detesting
detests
detract
detracted
detracting
detraction
detracts
detriment
detrimental
devastate
devastated
devastates
devastating
devastatingly
devastation
deviate
deviation
devil
devilish
devilishly
devilment
devilry
devious
deviously
deviousness
devoid
diabolic
diabolical
diabolically
diametrically
diappointed
diatribe
diatribes
dick
dictator
dictatorial
die
die-hard
died
dies
difficult
difficulties
difficulty
diffidence
dilapidated
dilemma
dilly-dally
dim
dimmer
din
ding
dings
dinky
dire
direly
direness
dirt
dirtbag
dirtbags
dirts
dirty
disable
disabled
disaccord
disadvantage
disadvantaged
disadvantageous
disadvantages
disaffect
disaffected
disaffirm
disagree
disagreeable
disagreeably
disagreed
disagreeing
disagreement
disagrees
disallow
disapointed
disapointing
disapointment
disappoint
disappointed
disappointing
disappointingly
disappointment
disappointments
disappoints
disapprobation
disapproval
disapprove
disapproving
disarm
disarray
disaster
disasterous
disastrous
disastrously
disavow
disavowal
disbelief
disbelieve
disbeliever
disclaim
discombobulate
discomfit
discomfititure
discomfort
discompose
disconcert
disconcerted
disconcerting
disconcertingly
disconsolate
disconsolately
disconsolation
discontent
discontented
discontentedly
discontinued
discontinuity
discontinuous
discord
discordance
discordant
discountenance
discourage
discouragement
discouraging
discouragingly
discourteous
discourteously
discoutinous
discredit
discrepant
discriminate
discrimination
discriminatory
disdain
disdained
disdainful
disdainfully
disfavor
disgrace
disgraced
disgraceful
disgracefully
disgruntle
disgruntled
disgust
disgusted
disgustedly
disgustful
disgustfully
disgusting
disgustingly
dishearten
disheartening
dishearteningly
dishonest
dishonestly
dishonesty
dishonor
dishonorable
dishonorablely
disillusion
disillusioned
disillusionment
disillusions
disinclination
disinclined
disingenuous
disingenuously
disintegrate
disintegrated
disintegrates
disintegration
disinterest
disinterested
dislike
disliked
dislikes
disliking
dislocated
disloyal
disloyalty
dismal
dismally
dismalness
dismay
dismayed
dismaying
dismayingly
dismissive
dismissively
disobedience
disobedient
disobey
disoobedient
disorder
disordered
disorderly
disorganized
disorient
disoriented
disown
disparage
disparaging
disparagingly
dispensable
dispirit
dispirited
dispiritedly
dispiriting
displace
displaced
displease
displeased
displeasing
displeasure
disproportionate
disprove
disputable
dispute
disputed
disquiet
disquieting
disquietingly
disquietude
disregard
disregardful
disreputable
disrepute
disrespect
disrespectable
disrespectablity
disrespectful
disrespectfully
disrespectfulness
disrespecting
disrupt
disruption
disruptive
diss
dissapointed
dissappointed
dissappointing
dissatisfaction
dissatisfactory
dissatisfied
dissatisfies
dissatisfy
dissatisfying
dissed
dissemble
dissembler
dissension
dissent
dissenter
dissention
disservice
disses
dissidence
dissident
dissidents
dissing
dissocial
dissolute
dissolution
dissonance
dissonant
dissonantly
dissuade
dissuasive
distains
distaste
distasteful
distastefully
distort
distorted
distortion
distorts
distract
distracting
distraction
distraught
distraughtly
distraughtness
distress
distressed
distressing
distressingly
distrust
distrustful
distrusting
disturb
disturbance
disturbed
disturbing
disturbingly
disunity
disvalue
divergent
divisive
divisively
divisiveness
dizzing
dizzingly
dizzy
doddering
dodgey
dogged
doggedly
dogmatic
doldrums
domineer
domineering
donside
doom
doomed
doomsday
dope
doubt
doubtful
doubtfully
doubts
douchbag
douchebag
douchebags
downbeat
downcast
downer
downfall
downfallen
downgrade
downhearted
downheartedly
downhill
downside
downsides
downturn
downturns
drab
draconian
draconic
drag
dragged
dragging
dragoon
drags
drain
drained
draining
drains
drastic
drastically
drawback
drawbacks
dread
dreadful
dreadfully
dreadfulness
dreary
dripped
dripping
drippy
drips
drones
droop
droops
drop-out
drop-outs
dropout
dropouts
drought
drowning
drunk
drunkard
drunken
dubious
dubiously
dubitable
dud
dull
dullard
dumb
dumbfound
dump
dumped
dumping
dumps
dunce
dungeon
dungeons
dupe
dust
dusty
dwindling
dying
earsplitting
eccentric
eccentricity
effigy
effrontery
egocentric
egomania
egotism
egotistical
egotistically
egregious
egregiously
election-rigger
elimination
emaciated
emasculate
embarrass
embarrassing
embarrassingly
embarrassment
embattled
embroil
embroiled
embroilment
emergency
emphatic
emphatically
emptiness
encroach
encroachment
endanger
enemies
enemy
enervate
enfeeble
enflame
engulf
enjoin
enmity
enrage
enraged
enraging
enslave
entangle
entanglement
entrap
entrapment
envious
enviously
enviousness
epidemic
equivocal
erase
erode
erodes
erosion
err
errant
erratic
erratically
erroneous
erroneously
error
errors
eruptions
escapade
eschew
estranged
evade
evasion
evasive
evil
evildoer
evils
eviscerate
exacerbate
exagerate
exagerated
exagerates
exaggerate
exaggeration
exasperate
exasperated
exasperating
exasperatingly
exasperation
excessive
excessively
exclusion
excoriate
excruciating
excruciatingly
excuse
excuses
execrate
exhaust
exhausted
exhaustion
exhausts
exhorbitant
exhort
exile
exorbitant
exorbitantance
exorbitantly
expel
expensive
expire
expired
explode
exploit
exploitation
explosive
expropriate
expropriation
expulse
expunge
exterminate
extermination
extinguish
extort
extortion
extraneous
extravagance
extravagant
extravagantly
extremism
extremist
extremists
eyesore
f**k
fabricate
fabrication
facetious
facetiously
fail
failed
failing
fails
failure
failures
faint
fainthearted
faithless
fake
fall
fallacies
fallacious
fallaciously
fallaciousness
fallacy
fallen
falling
fallout
falls
false
falsehood
falsely
falsify
falter
faltered
famine
famished
fanatic
fanatical
fanatically
fanaticism
fanatics
fanciful
far-fetched
farce
farcical
farcical-yet-provocative
farcically
farfetched
fascism
fascist
fastidious
fastidiously
fastuous
fat
fat-cat
fat-cats
fatal
fatalistic
fatalistically
fatally
fatcat
fatcats
fateful
fatefully
fathomless
fatigue
fatigued
fatique
fatty
fatuity
fatuous
fatuously
fault
faults
faulty
fawningly
faze
fear
fearful
fearfully
fears
fearsome
feckless
feeble
feeblely
feebleminded
feign
feint
fell
felon
felonious
ferociously
ferocity
fetid
fever
feverish
fevers
fiasco
fib
fibber
fickle
fiction
fictional
fictitious
fidget
fidgety
fiend
fiendish
fierce
figurehead
filth
filthy
finagle
finicky
fissures
fist
flabbergast
flabbergasted
flagging
flagrant
flagrantly
flair
flairs
flak
flake
flakey
flakieness
flaking
flaky
flare
flares
flareup
flareups
flat-out
flaunt
flaw
flawed
flaws
flee
fleed
fleeing
fleer
flees
fleeting
flicering
flicker
flickering
flickers
flighty
flimflam
flimsy
flirt
flirty
floored
flounder
floundering
flout
fluster
foe
fool
fooled
foolhardy
foolish
foolishly
foolishness
forbid
forbidden
forbidding
forceful
foreboding
forebodingly
forfeit
forged
forgetful
forgetfully
forgetfulness
forlorn
forlornly
forsake
forsaken
forswear
foul
foully
foulness
fractious
fractiously
fracture
fragile
fragmented
frail
frantic
frantically
franticly
fraud
fraudulent
fraught
frazzle
frazzled
freak
freaking
freakish
freakishly
freaks
freeze
freezes
freezing
frenetic
frenetically
frenzied
frenzy
fret
fretful
frets
friction
frictions
fried
friggin
frigging
fright
frighten
frightening
frighteningly
frightful
frightfully
frigid
frost
frown
froze
frozen
fruitless
fruitlessly
frustrate
frustrated
frustrates
frustrating
frustratingly
frustration
frustrations
fuck
fucking
fudge
fugitive
full-blown
fulminate
fumble
fume
fumes
fundamentalism
funky
funnily
funny
furious
furiously
furor
fury
fuss
fussy
fustigate
fusty
futile
futilely
futility
fuzzy
gabble
gaff
gaffe
gainsay
gainsayer
gall
galling
gallingly
galls
gangster
gape
garbage
garish
gasp
gauche
gaudy
gawk
gawky
geezer
genocide
get-rich
ghastly
ghetto
ghosting
gibber
gibberish
gibe
giddy
gimmick
gimmicked
gimmicking
gimmicks
gimmicky
glare
glaringly
glib
glibly
glitch
glitches
gloatingly
gloom
gloomy
glower
glum
glut
gnawing
goad
goading
god-awful
goof
goofy
goon
gossip
graceless
gracelessly
graft
grainy
grapple
grate
grating
gravely
greasy
greed
greedy
grief
grievance
grievances
grieve
grieving
grievous
grievously
grim
grimace
grind
gripe
gripes
grisly
gritty
gross
grossly
grotesque
grouch
grouchy
groundless
grouse
growl
grudge
grudges
grudging
grudgingly
gruesome
gruesomely
gruff
grumble
grumpier
grumpiest
grumpily
grumpish
grumpy
guile
guilt
guiltily
guilty
gullible
gutless
gutter
hack
hacks
haggard
haggle
hairloss
halfhearted
halfheartedly
hallucinate
hallucination
hamper
hampered
handicapped
hang
hangs
haphazard
hapless
harangue
harass
harassed
harasses
harassment
harboring
harbors
hard
hard-hit
hard-line
hard-liner
hardball
harden
hardened
hardheaded
hardhearted
hardliner
hardliners
hardship
hardships
harm
harmed
harmful
harms
harpy
harridan
harried
harrow
harsh
harshly
hasseling
hassle
hassled
hassles
haste
hastily
hasty
hate
hated
hateful
hatefully
hatefulness
hater
haters
hates
hating
hatred
haughtily
haughty
haunt
haunting
havoc
hawkish
haywire
hazard
hazardous
haze
hazy
head-aches
headache
headaches
heartbreaker
heartbreaking
heartbreakingly
heartless
heathen
heavy-handed
heavyhearted
heck
heckle
heckled
heckles
hectic
hedge
hedonistic
heedless
hefty
hegemonism
hegemonistic
hegemony
heinous
hell
hell-bent
hellion
hells
helpless
helplessly
helplessness
heresy
heretic
heretical
hesitant
hestitant
hideous
hideously
hideousness
high-priced
hiliarious
hinder
hindrance
hiss
hissed
hissing
ho-hum
hoard
hoax
hobble
hogs
hollow
hoodium
hoodwink
hooligan
hopeless
hopelessly
hopelessness
horde
horrendous
horrendously
horrible
horrid
horrific
horrified
horrifies
horrify
horrifying
horrifys
hostage
hostile
hostilities
hostility
hotbeds
hothead
hotheaded
hothouse
hubris
huckster
hum
humid
humiliate
humiliating
humiliation
humming
hung
hurt
hurted
hurtful
hurting
hurts
hustler
hype
hypocricy
hypocrisy
hypocrite
hypocrites
hypocritical
hypocritically
hysteria
hysteric
hysterical
hysterically
hysterics
idiocies
idiocy
idiot
idiotic
idiotically
idiots
idle
ignoble
ignominious
ignominiously
ignominy
ignorance
ignorant
ignore
ill-advised
ill-conceived
ill-defined
ill-designed
ill-fated
ill-favored
ill-formed
ill-mannered
ill-natured
ill-sorted
ill-tempered
ill-treated
ill-treatment
ill-usage
ill-used
illegal
illegally
illegitimate
illicit
illiterate
illness
illogic
illogical
illogically
illusion
illusions
illusory
imaginary
imbalance
imbecile
imbroglio
immaterial
immature
imminence
imminently
immobilized
immoderate
immoderately
immodest
immoral
immorality
immorally
immovable
impair
impaired
impasse
impatience
impatient
impatiently
impeach
impedance
impede
impediment
impending
impenitent
imperfect
imperfection
imperfections
imperfectly
imperialist
imperil
imperious
imperiously
impermissible
impersonal
impertinent
impetuous
impetuously
impiety
impinge
impious
implacable
implausible
implausibly
implicate
implication
implode
impolite
impolitely
impolitic
importunate
importune
impose
imposers
imposing
imposition
impossible
impossiblity
impossibly
impotent
impoverish
impoverished
impractical
imprecate
imprecise
imprecisely
imprecision
imprison
imprisonment
improbability
improbable
improbably
improper
improperly
impropriety
imprudence
imprudent
impudence
impudent
impudently
impugn
impulsive
impulsively
impunity
impure
impurity
inability
inaccuracies
inaccuracy
inaccurate
inaccurately
inaction
inactive
inadequacy
inadequate
inadequately
inadverent
inadverently
inadvisable
inadvisably
inane
inanely
inappropriate
inappropriately
inapt
inaptitude
inarticulate
inattentive
inaudible
incapable
incapably
incautious
incendiary
incense
incessant
incessantly
incite
incitement
incivility
inclement
incognizant
incoherence
incoherent
incoherently
incommensurate
incomparable
incomparably
incompatability
incompatibility
incompatible
incompetence
incompetent
incompetently
incomplete
incompliant
incomprehensible
incomprehension
inconceivable
inconceivably
incongruous
incongruously
inconsequent
inconsequential
inconsequentially
inconsequently
inconsiderate
inconsiderately
inconsistence
inconsistencies
inconsistency
inconsistent
inconsolable
inconsolably
inconstant
inconvenience
inconveniently
incorrect
incorrectly
incorrigible
incorrigibly
incredulous
incredulously
inculcate
indecency
indecent
indecently
indecision
indecisive
indecisively
indecorum
indefensible
indelicate
indeterminable
indeterminably
indeterminate
indifference
indifferent
indigent
indignant
indignantly
indignation
indignity
indiscernible
indiscreet
indiscreetly
indiscretion
indiscriminate
indiscriminately
indiscriminating
indistinguishable
indoctrinate
indoctrination
indolent
indulge
ineffective
ineffectively
ineffectiveness
ineffectual
ineffectually
ineffectualness
inefficacious
inefficacy
inefficiency
inefficient
inefficiently
inelegance
inelegant
ineligible
ineloquent
ineloquently
inept
ineptitude
ineptly
inequalities
inequality
inequitable
inequitably
inequities
inescapable
inescapably
inessential
inevitable
inevitably
inexcusable
inexcusably
inexorable
inexorably
inexperience
inexperienced
inexpert
inexpertly
inexpiable
inexplainable
inextricable
inextricably
infamous
infamously
infamy
infected
infection
infections
inferior
inferiority
infernal
infest
infested
infidel
infidels
infiltrator
infiltrators
infirm
inflame
inflammation
inflammatory
inflammed
inflated
inflationary
inflexible
inflict
infraction
infringe
infringement
infringements
infuriate
infuriated
infuriating
infuriatingly
inglorious
ingrate
ingratitude
inhibit
inhibition
inhospitable
inhospitality
inhuman
inhumane
inhumanity
inimical
inimically
iniquitous
iniquity
injudicious
injure
injurious
injury
injustice
injustices
innuendo
inoperable
inopportune
inordinate
inordinately
insane
insanely
insanity
insatiable
insecure
insecurity
insensible
insensitive
insensitively
insensitivity
insidious
insidiously
insignificance
insignificant
insignificantly
insincere
insincerely
insincerity
insinuate
insinuating
insinuation
insociable
insolence
insolent
insolently
insolvent
insouciance
instability
instable
instigate
instigator
instigators
insubordinate
insubstantial
insubstantially
insufferable
insufferably
insufficiency
insufficient
insufficiently
insular
insult
insulted
insulting
insultingly
insults
insupportable
insupportably
insurmountable
insurmountably
insurrection
intefere
inteferes
intense
interfere
interference
interferes
intermittent
interrupt
interruption
interruptions
intimidate
intimidating
intimidatingly
intimidation
intolerable
intolerablely
intolerance
intoxicate
intractable
intransigence
intransigent
intrude
intrusion
intrusive
inundate
inundated
invader
invalid
invalidate
invalidity
invasive
invective
inveigle
invidious
invidiously
invidiousness
invisible
involuntarily
involuntary
irascible
irate
irately
ire
irk
irked
irking
irks
irksome
irksomely
irksomeness
irksomenesses
ironic
ironical
ironically
ironies
irony
irragularity
irrational
irrationalities
irrationality
irrationally
irrationals
irreconcilable
irrecoverable
irrecoverableness
irrecoverablenesses
irrecoverably
irredeemable
irredeemably
irreformable
irregular
irregularity
irrelevance
irrelevant
irreparable
irreplacible
irrepressible
irresolute
irresolvable
irresponsible
irresponsibly
irretating
irretrievable
irreversible
irritable
irritably
irritant
irritate
irritated
irritating
irritation
irritations
isolate
isolated
isolation
issue
issues
itch
itching
itchy
jabber
jaded
jagged
jam
jarring
jaundiced
jealous
jealously
jealousness
jealousy
jeer
jeering
jeeringly
jeers
jeopardize
jeopardy
jerk
jerky
jitter
jitters
jittery
job-killing
jobless
joke
joker
jolt
judder
juddering
judders
jumpy
junk
junky
junkyard
jutter
jutters
kaput
kill
killed
killer
killing
killjoy
kills
knave
knife
knock
knotted
kook
kooky
lack
lackadaisical
lacked
lackey
lackeys
lacking
lackluster
lacks
laconic
lag
lagged
lagging
laggy
lags
laid-off
lambast
lambaste
lame
lame-duck
lament
lamentable
lamentably
languid
languish
languor
languorous
languorously
lanky
lapse
lapsed
lapses
lascivious
last-ditch
latency
laughable
laughably
laughingstock
lawbreaker
lawbreaking
lawless
lawlessness
layoff
layoff-happy
lazy
leak
leakage
leakages
leaking
leaks
leaky
lech
lecher
lecherous
lechery
leech
leer
leery
left-leaning
lemon
lengthy
less-developed
lesser-known
letch
lethal
lethargic
lethargy
lewd
lewdly
lewdness
liability
liable
liar
liars
licentious
licentiously
licentiousness
lie
lied
lier
lies
life-threatening
lifeless
limit
limitation
limitations
limited
limits
limp
listless
litigious
little-known
livid
lividly
loath
loathe
loathing
loathly
loathsome
loathsomely
lone
loneliness
lonely
loner
lonesome
long-time
long-winded
longing
longingly
loophole
loopholes
loose
loot
lorn
lose
loser
losers
loses
losing
loss
losses
lost
loud
louder
lousy
loveless
lovelorn
low-rated
lowly
ludicrous
ludicrously
lugubrious
lukewarm
lull
lumpy
lunatic
lunaticism
lurch
lure
lurid
lurk
lurking
lying
macabre
mad
madden
maddening
maddeningly
madder
madly
madman
madness
maladjusted
maladjustment
malady
malaise
malcontent
malcontented
maledict
malevolence
malevolent
malevolently
malice
malicious
maliciously
maliciousness
malign
malignant
malodorous
maltreatment
mangle
mangled
mangles
mangling
mania
maniac
maniacal
manic
manipulate
manipulation
manipulative
manipulators
mar
marginal
marginally
martyrdom
martyrdom-seeking
mashed
massacre
massacres
matte
mawkish
mawkishly
mawkishness
meager
meaningless
meanness
measly
meddle
meddlesome
mediocre
mediocrity
melancholy
melodramatic
melodramatically
meltdown
menace
menacing
menacingly
mendacious
mendacity
menial
merciless
mercilessly
mess
messed
messes
messing
messy
midget
miff
militancy
mindless
mindlessly
mirage
mire
misalign
misaligned
misaligns
misapprehend
misbecome
misbecoming
misbegotten
misbehave
misbehavior
miscalculate
miscalculation
miscellaneous
mischief
mischievous
mischievously
misconception
misconceptions
miscreant
miscreants
misdirection
miser
miserable
miserableness
miserably
miseries
miserly
misery
misfit
misfortune
misgiving
misgivings
misguidance
misguide
misguided
mishandle
mishap
misinform
misinformed
misinterpret
misjudge
misjudgment
mislead
misleading
misleadingly
mislike
mismanage
mispronounce
mispronounced
mispronounces
misread
misreading
misrepresent
misrepresentation
miss
missed
misses
misstatement
mist
mistake
mistaken
mistakenly
mistakes
mistified
mistress
mistrust
mistrustful
mistrustfully
mists
misunderstand
misunderstanding
misunderstandings
misunderstood
misuse
moan
mobster
mock
mocked
mockeries
mockery
mocking
mockingly
mocks
molest
molestation
monotonous
monotony
monster
monstrosities
monstrosity
monstrous
monstrously
moody
moot
mope
morbid
morbidly
mordant
mordantly
moribund
moron
moronic
morons
mortification
mortified
mortify
mortifying
motionless
motley
mourn
mourner
mournful
mournfully
muddle
muddy
mudslinger
mudslinging
mulish
multi-polarization
mundane
murder
murderer
murderous
murderously
murky
muscle-flexing
mushy
musty
mysterious
mysteriously
mystery
mystify
myth
nag
nagging
naive
naively
narrower
nastily
nastiness
nasty
naughty
nauseate
nauseates
nauseating
nauseatingly
nave
nebulous
nebulously
needless
needlessly
needy
nefarious
nefariously
negate
negation
negative
negatives
negativity
neglect
neglected
negligence
negligent
nemesis
nepotism
nervous
nervously
nervousness
nettle
nettlesome
neurotic
neurotically
niggle
niggles
nightmare
nightmarish
nightmarishly
nitpick
nitpicking
noise
noises
noisier
noisy
non-confidence
nonexistent
nonresponsive
nonsense
nosey
notoriety
notorious
notoriously
noxious
nuisance
numb
obese
object
objection
objectionable
objections
oblique
obliterate
obliterated
oblivious
obnoxious
obnoxiously
obscene
obscenely
obscenity
obscure
obscured
obscures
obscurity
obsess
obsessive
obsessively
obsessiveness
obsolete
obstacle
obstinate
obstinately
obstruct
obstructed
obstructing
obstruction
obstructs
obtrusive
obtuse
occlude
occluded
occludes
occluding
odd
odder
oddest
oddities
oddity
oddly
odor
offence
offend
offender
offending
offenses
offensive
offensively
offensiveness
officious
ominous
ominously
omission
omit
one-sided
onerous
onerously
onslaught
opinionated
opponent
opportunistic
oppose
opposition
oppositions
oppress
oppression
oppressive
oppressively
oppressiveness
oppressors
ordeal
orphan
ostracize
outbreak
outburst
outbursts
outcast
outcry
outlaw
outmoded
outrage
outraged
outrageous
outrageously
outrageousness
outrages
outsider
over-acted
over-awe
over-balanced
over-hyped
over-priced
over-valuation
overact
overacted
overawe
overbalance
overbalanced
overbearing
overbearingly
overblown
overdo
overdone
overdue
overemphasize
overheat
overkill
overloaded
overlook
overpaid
overpayed
overplay
overpower
overpriced
overrated
overreach
overrun
overshadow
oversight
oversights
oversimplification
oversimplified
oversimplify
oversize
overstate
overstated
overstatement
overstatements
overstates
overtaxed
overthrow
overthrows
overturn
overweight
overwhelm
overwhelmed
overwhelming
overwhelmingly
overwhelms
overzealous
overzealously
overzelous
pain
painful
painfull
painfully
pains
pale
pales
paltry
pan
pandemonium
pander
pandering
panders
panic
panick
panicked
panicking
panicky
paradoxical
paradoxically
paralize
paralyzed
paranoia
paranoid
parasite
pariah
parody
partiality
partisan
partisans
passe
passive
passiveness
pathetic
pathetically
patronize
paucity
pauper
paupers
payback
peculiar
peculiarly
pedantic
peeled
peeve
peeved
peevish
peevishly
penalize
penalty
perfidious
perfidity
perfunctory
peril
perilous
perilously
perish
pernicious
perplex
perplexed
perplexing
perplexity
persecute
persecution
pertinacious
pertinaciously
pertinacity
perturb
perturbed
pervasive
perverse
perversely
perversion
perversity
pervert
perverted
perverts
pessimism
pessimistic
pessimistically
pest
pestilent
petrified
petrify
pettifog
petty
phobia
phobic
phony
picket
picketed
picketing
pickets
picky
pig
pigs
pillage
pillory
pimple
pinch
pique
pitiable
pitiful
pitifully
pitiless
pitilessly
pittance
pity
plagiarize
plague
plasticky
plaything
plea
pleas
plebeian
plight
plot
plotters
ploy
plunder
plunderer
pointless
pointlessly
poison
poisonous
poisonously
pokey
poky
polarisation
polemize
pollute
polluter
polluters
polution
pompous
poor
poorer
poorest
poorly
posturing
pout
poverty
powerless
prate
pratfall
prattle
precarious
precariously
precipitate
precipitous
predatory
predicament
prejudge
prejudice
prejudices
prejudicial
premeditated
preoccupy
preposterous
preposterously
presumptuous
presumptuously
pretence
pretend
pretense
pretentious
pretentiously
prevaricate
pricey
pricier
prick
prickle
prickles
prideful
prik
primitive
prison
prisoner
problem
problematic
problems
procrastinate
procrastinates
procrastination
profane
profanity
prohibit
prohibitive
prohibitively
propaganda
propagandize
proprietary
prosecute
protest
protested
protesting
protests
protracted
provocation
provocative
provoke
pry
pugnacious
pugnaciously
pugnacity
punch
punish
punishable
punitive
punk
puny
puppet
puppets
puzzled
puzzlement
puzzling
quack
qualm
qualms
quandary
quarrel
quarrellous
quarrellously
quarrels
quarrelsome
quash
queer
questionable
quibble
quibbles
quitter
rabid
racism
racist
racists
racy
radical
radicalization
radically
radicals
rage
ragged
raging
rail
raked
rampage
rampant
ramshackle
rancor
randomly
rankle
rant
ranted
ranting
rantingly
rants
rape
raped
raping
rascal
rascals
rash
rattle
rattled
rattles
ravage
raving
reactionary
rebellious
rebuff
rebuke
recalcitrant
recant
recession
recessionary
reckless
recklessly
recklessness
recoil
recourses
redundancy
redundant
refusal
refuse
refused
refuses
refusing
refutation
refute
refuted
refutes
refuting
regress
regression
regressive
regret
regreted
regretful
regretfully
regrets
regrettable
regrettably
regretted
reject
rejected
rejecting
rejection
rejects
relapse
relentless
relentlessly
relentlessness
reluctance
reluctant
reluctantly
remorse
remorseful
remorsefully
remorseless
remorselessly
remorselessness
renounce
renunciation
repel
repetitive
reprehensible
reprehensibly
reprehension
reprehensive
repress
repression
repressive
reprimand
reproach
reproachful
reprove
reprovingly
repudiate
repudiation
repugn
repugnance
repugnant
repugnantly
repulse
repulsed
repulsing
repulsive
repulsively
repulsiveness
resent
resentful
resentment
resignation
resigned
resistance
restless
restlessness
restrict
restricted
restriction
restrictive
resurgent
retaliate
retaliatory
retard
retarded
retardedness
retards
reticent
retract
retreat
retreated
revenge
revengeful
revengefully
revert
revile
reviled
revoke
revolt
revolting
revoltingly
revulsion
revulsive
rhapsodize
rhetoric
rhetorical
ricer
ridicule
ridicules
ridiculous
ridiculously
rife
rift
rifts
rigid
rigidity
rigidness
rile
riled
rip
rip-off
ripoff
ripped
risk
risks
risky
rival
rivalry
roadblocks
rocky
rogue
rollercoaster
rot
rotten
rough
rremediable
rubbish
rude
rue
ruffian
ruffle
ruin
ruined
ruining
ruinous
ruins
rumbling
rumor
rumors
rumours
rumple
run-down
runaway
rupture
rust
rusts
rusty
rut
ruthless
ruthlessly
ruthlessness
ruts
sabotage
sack
sacrificed
sad
sadden
sadly
sadness
sag
sagged
sagging
saggy
sags
salacious
sanctimonious
sap
sarcasm
sarcastic
sarcastically
sardonic
sardonically
sass
satirical
satirize
savage
savaged
savagery
savages
scaly
scam
scams
scandal
scandalize
scandalized
scandalous
scandalously
scandals
scandel
scandels
scant
scapegoat
scar
scarce
scarcely
scarcity
scare
scared
scarier
scariest
scarily
scarred
scars
scary
scathing
scathingly
sceptical
scoff
scoffingly
scold
scolded
scolding
scoldingly
scorching
scorchingly
scorn
scornful
scornfully
scoundrel
scourge
scowl
scramble
scrambled
scrambles
scrambling
scrap
scratch
scratched
scratches
scratchy
scream
screech
screw-up
screwed
screwed-up
screwy
scuff
scuffs
scum
scummy
second-class
second-tier
secretive
sedentary
seedy
seethe
seething
self-coup
self-criticism
self-defeating
self-destructive
self-humiliation
self-interest
self-interested
self-serving
selfinterested
selfish
selfishly
selfishness
semi-retarded
senile
sensationalize
senseless
senselessly
seriousness
sermonize
servitude
set-up
setback
setbacks
sever
severe
severity
sh*t
shabby
shadowy
shady
shake
shaky
shallow
sham
shambles
shame
shameful
shamefully
shamefulness
shameless
shamelessly
shamelessness
shark
sharply
shatter
shemale
shimmer
shimmy
shipwreck
shirk
shirker
shit
shiver
shock
shocked
shocking
shockingly
shoddy
short-lived
shortage
shortchange
shortcoming
shortcomings
shortness
shortsighted
shortsightedness
showdown
shrew
shriek
shrill
shrilly
shrivel
shroud
shrouded
shrug
shun
shunned
sick
sicken
sickening
sickeningly
sickly
sickness
sidetrack
sidetracked
siege
sillily
silly
simplistic
simplistically
sin
sinful
sinfully
sinister
sinisterly
sink
sinking
skeletons
skeptic
skeptical
skeptically
skepticism
sketchy
skimpy
skinny
skittish
skittishly
skulk
slack
slander
slanderer
slanderous
slanderously
slanders
slap
slashing
slaughter
slaughtered
slave
slaves
sleazy
slime
slog
slogged
slogging
slogs
sloooooooooooooow
sloooow
slooow
sloow
sloppily
sloppy
sloth
slothful
slow
slow-moving
slowed
slower
slowest
slowly
sloww
slowww
slowwww
slug
sluggish
slump
slumping
slumpping
slur
slut
sluts
sly
smack
smallish
smash
smear
smell
smelled
smelling
smells
smelly
smelt
smoke
smokescreen
smolder
smoldering
smother
smoulder
smouldering
smudge
smudged
smudges
smudging
smug
smugly
smut
smuttier
smuttiest
smutty
snag
snagged
snagging
snags
snappish
snappishly
snare
snarky
snarl
sneak
sneakily
sneaky
sneer
sneering
sneeringly
snob
snobbish
snobby
snobish
snobs
snub
so-cal
soapy
sob
sober
sobering
solemn
solicitude
somber
sore
sorely
soreness
sorrow
sorrowful
sorrowfully
sorry
sour
sourly
spade
spank
spendy
spew
spewed
spewing
spews
spilling
spinster
spiritless
spite
spiteful
spitefully
spitefulness
splatter
split
splitting
spoil
spoilage
spoilages
spoiled
spoilled
spoils
spook
spookier
spookiest
spookily
spooky
spoon-fed
spoon-feed
spoonfed
sporadic
spotty
spurious
spurn
sputter
squabble
squabbling
squander
squash
squeak
squeaks
squeaky
squeal
squealing
squeals
squirm
stab
stagnant
stagnate
stagnation
staid
stain
stains
stale
stalemate
stall
stalls
stammer
stampede
standstill
stark
starkly
startle
startling
startlingly
starvation
starve
static
steal
stealing
steals
steep
steeply
stench
stereotype
stereotypical
stereotypically
stern
stew
sticky
stiff
stiffness
stifle
stifling
stiflingly
stigma
stigmatize
sting
stinging
stingingly
stingy
stink
stinks
stodgy
stole
stolen
stooge
stooges
stormy
straggle
straggler
strain
strained
straining
strange
strangely
stranger
strangest
strangle
streaky
strenuous
stress
stresses
stressful
stressfully
stricken
strict
strictly
strident
stridently
strife
strike
stringent
stringently
struck
struggle
struggled
struggles
struggling
strut
stubborn
stubbornly
stubbornness
stuck
stuffy
stumble
stumbled
stumbles
stump
stumped
stumps
stun
stunt
stunted
stupid
stupidest
stupidity
stupidly
stupified
stupify
stupor
stutter
stuttered
stuttering
stutters
sty
stymied
sub-par
subdued
subjected
subjection
subjugate
subjugation
submissive
subordinate
subpoena
subpoenas
subservience
subservient
substandard
subtract
subversion
subversive
subversively
subvert
succumb
suck
sucked
sucker
sucks
sucky
sue
sued
sueing
sues
suffer
suffered
sufferer
sufferers
suffering
suffers
suffocate
sugar-coat
sugar-coated
sugarcoated
suicidal
suicide
sulk
sullen
sully
sunder
sunk
sunken
superficial
superficiality
superficially
superfluous
superstition
superstitious
suppress
suppression
surrender
susceptible
suspect
suspicion
suspicions
suspicious
suspiciously
swagger
swamped
sweaty
swelled
swelling
swindle
swipe
swollen
symptom
symptoms
syndrome
taboo
tacky
taint
tainted
tamper
tangle
tangled
tangles
tank
tanked
tanks
tantrum
tardy
tarnish
tarnished
tarnishes
tarnishing
tattered
taunt
taunting
tauntingly
taunts
taut
tawdry
taxing
tease
teasingly
tedious
tediously
temerity
temper
tempest
temptation
tenderness
tense
tension
tentative
tentatively
tenuous
tenuously
tepid
terrible
terribleness
terribly
terror
terror-genic
terrorism
terrorize
testily
testy
tetchily
tetchy
thankless
thicker
thirst
thorny
thoughtless
thoughtlessly
thoughtlessness
thrash
threat
threaten
threatening
threats
threesome
throb
throbbed
throbbing
throbs
throttle
thug
thumb-down
thumbs-down
thwart
time-consuming
timid
timidity
timidly
timidness
tin-y
tingled
tingling
tired
tiresome
tiring
tiringly
toil
toll
top-heavy
topple
torment
tormented
torrent
tortuous
torture
tortured
tortures
torturing
torturous
torturously
totalitarian
touchy
toughness
tout
touted
touts
toxic
traduce
tragedy
tragic
tragically
traitor
traitorous
traitorously
tramp
trample
transgress
transgression
trap
traped
trapped
trash
trashed
trashy
trauma
traumatic
traumatically
traumatize
traumatized
travesties
travesty
treacherous
treacherously
treachery
treason
treasonous
trick
tricked
trickery
tricky
trivial
trivialize
trouble
troubled
troublemaker
troubles
troublesome
troublesomely
troubling
troublingly
truant
tumble
tumbled
tumbles
tumultuous
turbulent
turmoil
twist
twisted
twists
two-faced
two-faces
tyrannical
tyrannically
tyranny
tyrant
ugh
uglier
ugliest
ugliness
ugly
ulterior
ultimatum
ultimatums
ultra-hardline
un-viewable
unable
unacceptable
unacceptablely
unacceptably
unaccessible
unaccustomed
unachievable
unaffordable
unappealing
unattractive
unauthentic
unavailable
unavoidably
unbearable
unbearablely
unbelievable
unbelievably
uncaring
uncertain
uncivil
uncivilized
unclean
unclear
uncollectible
uncomfortable
uncomfortably
uncomfy
uncompetitive
uncompromising
uncompromisingly
unconfirmed
unconstitutional
uncontrolled
unconvincing
unconvincingly
uncooperative
uncouth
uncreative
undecided
undefined
undependability
undependable
undercut
undercuts
undercutting
underdog
underestimate
underlings
undermine
undermined
undermines
undermining
underpaid
underpowered
undersized
undesirable
undetermined
undid
undignified
undissolved
undocumented
undone
undue
unease
uneasily
uneasiness
uneasy
uneconomical
unemployed
unequal
unethical
uneven
uneventful
unexpected
unexpectedly
unexplained
unfairly
unfaithful
unfaithfully
unfamiliar
unfavorable
unfeeling
unfinished
unfit
unforeseen
unforgiving
unfortunate
unfortunately
unfounded
unfriendly
unfulfilled
unfunded
ungovernable
ungrateful
unhappily
unhappiness
unhappy
unhealthy
unhelpful
unilateralism
unimaginable
unimaginably
unimportant
uninformed
uninsured
unintelligible
unintelligile
unipolar
unjust
unjustifiable
unjustifiably
unjustified
unjustly
unkind
unkindly
unknown
unlamentable
unlamentably
unlawful
unlawfully
unlawfulness
unleash
unlicensed
unlikely
unlucky
unmoved
unnatural
unnaturally
unnecessary
unneeded
unnerve
unnerved
unnerving
unnervingly
unnoticed
unobserved
unorthodox
unorthodoxy
unpleasant
unpleasantries
unpopular
unpredictable
unprepared
unproductive
unprofitable
unprove
unproved
unproven
unproves
unproving
unqualified
unravel
unraveled
unreachable
unreadable
unrealistic
unreasonable
unreasonably
unrelenting
unrelentingly
unreliability
unreliable
unresolved
unresponsive
unrest
unruly
unsafe
unsatisfactory
unsavory
unscrupulous
unscrupulously
unsecure
unseemly
unsettle
unsettled
unsettling
unsettlingly
unskilled
unsophisticated
unsound
unspeakable
unspeakablely
unspecified
unstable
unsteadily
unsteadiness
unsteady
unsuccessful
unsuccessfully
unsupported
unsupportive
unsure
unsuspecting
unsustainable
untenable
untested
unthinkable
unthinkably
untimely
untouched
untrue
untrustworthy
untruthful
unusable
unusably
unuseable
unuseably
unusual
unusually
unviewable
unwanted
unwarranted
unwatchable
unwelcome
unwell
unwieldy
unwilling
unwillingly
unwillingness
unwise
unwisely
unworkable
unworthy
unyielding
upbraid
upheaval
uprising
uproar
uproarious
uproariously
uproarous
uproarously
uproot
upset
upseting
upsets
upsetting
upsettingly
urgent
useless
usurp
usurper
utterly
vagrant
vague
vagueness
vain
vainly
vanity
vehement
vehemently
vengeance
vengeful
vengefully
vengefulness
venom
venomous
venomously
vent
vestiges
vex
vexation
vexing
vexingly
vibrate
vibrated
vibrates
vibrating
vibration
vice
vicious
viciously
viciousness
victimize
vile
vileness
vilify
villainous
villainously
villains
villian
villianous
villianously
villify
vindictive
vindictively
vindictiveness
violate
violation
violator
violators
violent
violently
viper
virulence
virulent
virulently
virus
vociferous
vociferously
volatile
volatility
vomit
vomited
vomiting
vomits
vulgar
vulnerable
wack
wail
wallow
wane
waning
wanton
war-like
warily
wariness
warlike
warned
warning
warp
warped
wary
washed-out
waste
wasted
wasteful
wastefulness
wasting
water-down
watered-down
wayward
weak
weaken
weakening
weaker
weakness
weaknesses
weariness
wearisome
weary
wedge
weed
weep
weird
weirdly
wheedle
whimper
whine
whining
whiny
whips
whore
whores
wicked
wickedly
wickedness
wild
wildly
wiles
wilt
wily
wimpy
wince
wobble
wobbled
wobbles
woe
woebegone
woeful
woefully
womanizer
womanizing
worn
worried
worriedly
worrier
worries
worrisome
worry
worrying
worryingly
worse
worsen
worsening
worst
worthless
worthlessly
worthlessness
wound
wounds
wrangle
wrath
wreak
wreaked
wreaks
wreck
wrest
wrestle
wretch
wretched
wretchedly
wretchedness
wrinkle
wrinkled
wrinkles
wrip
wripped
wripping
writhe
wrong
wrongful
wrongly
wrought
yawn
zap
zapped
zaps
zealot
zealous
zealously
zombie


In [ ]:
%%writefile /kaggle/working/lexicons/npov_lexicon.txt
abortion
abuse
abusive
accept
account
accusation
accuse
acknowledge
act
action
addition
adhere
admit
advance
advocate
affair
affirm
aggression
agree
aim
allegation
allege
allegedly
allow
alternative
amazing
america
american
annex
anti-semitism
anti-terrorist
apologetic
appear
apply
arab
area
argue
army
article
assert
attack
attempt
attribute
author
authority
award
away
baby
bad
bank
barbarian
base
bear
begin
behavior
belief
believe
best
best-selling
bestiality
betrayal
biblical
bishop
blame
bombing
book
branch
break
bring
british
building
bus
business
buy
call
capture
case
catholic
century
certain
challenge
change
characterize
charge
chauvenistic
child
china
christian
church
cite
citizen
city
civilian
claim
cleanse
clear
clearly
close
club
collapse
colony
come
commence
comment
commentator
common
communal
communist
community
complex
concept
concern
conclude
condemn
condition
confirm
conservative
consider
conspiracy
content
contradict
control
controversial
controversy
convert
cornerstone
correct
correspondent
corruption
cost
country
create
creation
crime
criminal
critic
criticism
criticize
croatium
crowd
cult
culture
current
currently
day
deal
death
debunk
declare
defend
defender
defense
define
demand
democratic
demonstrate
denomination
deny
design
despite
destructive
detail
development
dictator
different
disagree
disasterous
dispute
distortion
document
drop
dynasty
early
earmark
east
economic
effect
effort
end
enemy
ensure
escape
essential
establish
estimate
ethnic
evacuation
event
example
excellent
execute
exile
exist
exodus
experience
explain
expose
expulsion
extermination
extremist
fact
failure
faith
fake
famous
fan
fanaticism
far
far-right
fascist
feature
fight
fighter
film
find
finding
focus
follow
force
form
formation
former
forum
france
free
freedom
friend
function
future
fyrom
gay
general
genocide
global
goal
god
good
government
great
green
group
hand
happen
hear
heart
help
hero
hijack
hindu
historian
historic
historical
historically
history
hold
homeland
homosexual
homosexuality
honor
hope
hostile
house
human
idea
identify
identity
ideology
illegal
image
immigrant
imply
important
impose
imposition
impressive
improve
inappropriate
include
increase
independence
independent
indian
indicate
indium
individual
influence
insist
instance
instead
institution
international
internationally
intervene
intervention
invade
invasion
investigation
involve
involvement
ireland
irish
islam
islamic
israel
issue
japan
japanese
jew
journal
justice
keep
kill
king
kingdom
kke
know
korean
label
lack
land
large
larouche
later
launch
law
lawyer
lead
leader
left
left-wing
leftist
legalization
legend
legitimate
liberal
liberate
liberation
lie
life
light
literally
little
live
long
lose
macedonian
main
maintain
majority
make
man
march
marriage
martyr
massacre
match
mean
means
media
meet
member
mercenary
mexican
migration
militant
military
minority
misuse
mock
modern
monarchy
monk
moscow
mosque
mother
movement
mr
muhammad
murder
muslim
myth
name
nation
national
nationalist
natural
need
neighborhood
new
news
nineteenth
north
note
notoriety
notorious
number
obama
obviously
occupation
occupy
occur
old
open
operation
opponent
oppose
opposition
order
organization
original
palestine
palestinian
part
participate
participation
party
pass
pat
pbuh
peace
people
perceive
persecution
persistent
person
personal
piracy
pirate
place
plan
point
policy
polish
political
politician
poor
popular
population
positive
possibility
post
powerful
prejudice
present
pressure
print
prishtina
prisoner
pro-life
problem
process
profile
promote
prompt
propaganda
prophet
protect
protestant
prove
provide
province
purported
quality
question
race
racist
radical
range
rate
reach
reaction
reason
rebel
recently
recognize
record
refer
reform
refuse
refute
regard
regardless
regime
region
reject
relation
relationship
release
religion
religious
remark
remove
report
republic
reputation
research
researcher
resort
respect
response
restriction
result
retaliation
return
reveal
revival
revolution
right
right-wing
rights
riot
root
rule
run
russium
s
saddam
same-sex
saw
say
scandal
scholar
school
scrutiny
seat
second
sect
see
seem
sell
separate
separatist
serbian
serve
service
settlement
sex
sexual
shortcoming
show
sign
similarly
sit
small
so-called
social
socialist
society
soldier
source
south
soviet
specific
speculate
speed
st
standard
start
state
status
steam
strident
strongly
struggle
study
style
subject
successful
suggest
summer
support
supporter
suppose
supremacist
survivor
suspect
system
tactic
take
target
tax
technique
tell
term
territory
terrorism
terrorist
text
theorist
theory
thing
thinker
thought
thousand
threat
threaten
time
tiny
title
today
tormentor
total
traditional
traditionally
true
try
twentieth
unborn
undocumented
unfortunately
united
university
usa
use
usually
value
variety
various
vast
very
victim
victory
vietnam
vietnamese
view
violate
violence
violent
vojvodina
vote
wage
war
warming
warrior
way
website
west
western
white
widely
win
wing
winner
woman
word
work
world
write
wwius
yahweh
yazid
year
young
zionist


In [ ]:
%%writefile /kaggle/working/lexicons/positive_liu2005.txt
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
; 
; Opinion Lexicon: Positive
;
; This file contains a list of POSITIVE opinion words (or sentiment words).
;
; This file and the papers can all be downloaded from 
;    http://www.cs.uic.edu/~liub/FBS/sentiment-analysis.html
;
; If you use this list, please cite the following paper:
;
;   Minqing Hu and Bing Liu. "Mining and Summarizing Customer Reviews." 
;       Proceedings of the ACM SIGKDD International Conference on Knowledge 
;       Discovery and Data Mining (KDD-2004), Aug 22-25, 2004, Seattle, 
;       Washington, USA, 
; Notes: 
;    1. The appearance of an opinion word in a sentence does not necessarily  
;       mean that the sentence expresses a positive or negative opinion. 
;       See the paper below:
;
;       Bing Liu. "Sentiment Analysis and Subjectivity." An chapter in 
;          Handbook of Natural Language Processing, Second Edition, 
;          (editors: N. Indurkhya and F. J. Damerau), 2010.
;
;    2. You will notice many misspelled words in the list. They are not 
;       mistakes. They are included as these misspelled words appear 
;       frequently in social media content. 
;
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;

a+
abound
abounds
abundance
abundant
accessable
accessible
acclaim
acclaimed
acclamation
accolade
accolades
accommodative
accomodative
accomplish
accomplished
accomplishment
accomplishments
accurate
accurately
achievable
achievement
achievements
achievible
acumen
adaptable
adaptive
adequate
adjustable
admirable
admirably
admiration
admire
admirer
admiring
admiringly
adorable
adore
adored
adorer
adoring
adoringly
adroit
adroitly
adulate
adulation
adulatory
advanced
advantage
advantageous
advantageously
advantages
adventuresome
adventurous
advocate
advocated
advocates
affability
affable
affably
affectation
affection
affectionate
affinity
affirm
affirmation
affirmative
affluence
affluent
afford
affordable
affordably
afordable
agile
agilely
agility
agreeable
agreeableness
agreeably
all-around
alluring
alluringly
altruistic
altruistically
amaze
amazed
amazement
amazes
amazing
amazingly
ambitious
ambitiously
ameliorate
amenable
amenity
amiability
amiabily
amiable
amicability
amicable
amicably
amity
ample
amply
amuse
amusing
amusingly
angel
angelic
apotheosis
appeal
appealing
applaud
appreciable
appreciate
appreciated
appreciates
appreciative
appreciatively
appropriate
approval
approve
ardent
ardently
ardor
articulate
aspiration
aspirations
aspire
assurance
assurances
assure
assuredly
assuring
astonish
astonished
astonishing
astonishingly
astonishment
astound
astounded
astounding
astoundingly
astutely
attentive
attraction
attractive
attractively
attune
audible
audibly
auspicious
authentic
authoritative
autonomous
available
aver
avid
avidly
award
awarded
awards
awe
awed
awesome
awesomely
awesomeness
awestruck
awsome
backbone
balanced
bargain
beauteous
beautiful
beautifullly
beautifully
beautify
beauty
beckon
beckoned
beckoning
beckons
believable
believeable
beloved
benefactor
beneficent
beneficial
beneficially
beneficiary
benefit
benefits
benevolence
benevolent
benifits
best
best-known
best-performing
best-selling
better
better-known
better-than-expected
beutifully
blameless
bless
blessing
bliss
blissful
blissfully
blithe
blockbuster
bloom
blossom
bolster
bonny
bonus
bonuses
boom
booming
boost
boundless
bountiful
brainiest
brainy
brand-new
brave
bravery
bravo
breakthrough
breakthroughs
breathlessness
breathtaking
breathtakingly
breeze
bright
brighten
brighter
brightest
brilliance
brilliances
brilliant
brilliantly
brisk
brotherly
bullish
buoyant
cajole
calm
calming
calmness
capability
capable
capably
captivate
captivating
carefree
cashback
cashbacks
catchy
celebrate
celebrated
celebration
celebratory
champ
champion
charisma
charismatic
charitable
charm
charming
charmingly
chaste
cheaper
cheapest
cheer
cheerful
cheery
cherish
cherished
cherub
chic
chivalrous
chivalry
civility
civilize
clarity
classic
classy
clean
cleaner
cleanest
cleanliness
cleanly
clear
clear-cut
cleared
clearer
clearly
clears
clever
cleverly
cohere
coherence
coherent
cohesive
colorful
comely
comfort
comfortable
comfortably
comforting
comfy
commend
commendable
commendably
commitment
commodious
compact
compactly
compassion
compassionate
compatible
competitive
complement
complementary
complemented
complements
compliant
compliment
complimentary
comprehensive
conciliate
conciliatory
concise
confidence
confident
congenial
congratulate
congratulation
congratulations
congratulatory
conscientious
considerate
consistent
consistently
constructive
consummate
contentment
continuity
contrasty
contribution
convenience
convenient
conveniently
convience
convienient
convient
convincing
convincingly
cool
coolest
cooperative
cooperatively
cornerstone
correct
correctly
cost-effective
cost-saving
counter-attack
counter-attacks
courage
courageous
courageously
courageousness
courteous
courtly
covenant
cozy
creative
credence
credible
crisp
crisper
cure
cure-all
cushy
cute
cuteness
danke
danken
daring
daringly
darling
dashing
dauntless
dawn
dazzle
dazzled
dazzling
dead-cheap
dead-on
decency
decent
decisive
decisiveness
dedicated
defeat
defeated
defeating
defeats
defender
deference
deft
deginified
delectable
delicacy
delicate
delicious
delight
delighted
delightful
delightfully
delightfulness
dependable
dependably
deservedly
deserving
desirable
desiring
desirous
destiny
detachable
devout
dexterous
dexterously
dextrous
dignified
dignify
dignity
diligence
diligent
diligently
diplomatic
dirt-cheap
distinction
distinctive
distinguished
diversified
divine
divinely
dominate
dominated
dominates
dote
dotingly
doubtless
dreamland
dumbfounded
dumbfounding
dummy-proof
durable
dynamic
eager
eagerly
eagerness
earnest
earnestly
earnestness
ease
eased
eases
easier
easiest
easiness
easing
easy
easy-to-use
easygoing
ebullience
ebullient
ebulliently
ecenomical
economical
ecstasies
ecstasy
ecstatic
ecstatically
edify
educated
effective
effectively
effectiveness
effectual
efficacious
efficient
efficiently
effortless
effortlessly
effusion
effusive
effusively
effusiveness
elan
elate
elated
elatedly
elation
electrify
elegance
elegant
elegantly
elevate
elite
eloquence
eloquent
eloquently
embolden
eminence
eminent
empathize
empathy
empower
empowerment
enchant
enchanted
enchanting
enchantingly
encourage
encouragement
encouraging
encouragingly
endear
endearing
endorse
endorsed
endorsement
endorses
endorsing
energetic
energize
energy-efficient
energy-saving
engaging
engrossing
enhance
enhanced
enhancement
enhances
enjoy
enjoyable
enjoyably
enjoyed
enjoying
enjoyment
enjoys
enlighten
enlightenment
enliven
ennoble
enough
enrapt
enrapture
enraptured
enrich
enrichment
enterprising
entertain
entertaining
entertains
enthral
enthrall
enthralled
enthuse
enthusiasm
enthusiast
enthusiastic
enthusiastically
entice
enticed
enticing
enticingly
entranced
entrancing
entrust
enviable
enviably
envious
enviously
enviousness
envy
equitable
ergonomical
err-free
erudite
ethical
eulogize
euphoria
euphoric
euphorically
evaluative
evenly
eventful
everlasting
evocative
exalt
exaltation
exalted
exaltedly
exalting
exaltingly
examplar
examplary
excallent
exceed
exceeded
exceeding
exceedingly
exceeds
excel
exceled
excelent
excellant
excelled
excellence
excellency
excellent
excellently
excels
exceptional
exceptionally
excite
excited
excitedly
excitedness
excitement
excites
exciting
excitingly
exellent
exemplar
exemplary
exhilarate
exhilarating
exhilaratingly
exhilaration
exonerate
expansive
expeditiously
expertly
exquisite
exquisitely
extol
extoll
extraordinarily
extraordinary
exuberance
exuberant
exuberantly
exult
exultant
exultation
exultingly
eye-catch
eye-catching
eyecatch
eyecatching
fabulous
fabulously
facilitate
fair
fairly
fairness
faith
faithful
faithfully
faithfulness
fame
famed
famous
famously
fancier
fancinating
fancy
fanfare
fans
fantastic
fantastically
fascinate
fascinating
fascinatingly
fascination
fashionable
fashionably
fast
fast-growing
fast-paced
faster
fastest
fastest-growing
faultless
fav
fave
favor
favorable
favored
favorite
favorited
favour
fearless
fearlessly
feasible
feasibly
feat
feature-rich
fecilitous
feisty
felicitate
felicitous
felicity
fertile
fervent
fervently
fervid
fervidly
fervor
festive
fidelity
fiery
fine
fine-looking
finely
finer
finest
firmer
first-class
first-in-class
first-rate
flashy
flatter
flattering
flatteringly
flawless
flawlessly
flexibility
flexible
flourish
flourishing
fluent
flutter
fond
fondly
fondness
foolproof
foremost
foresight
formidable
fortitude
fortuitous
fortuitously
fortunate
fortunately
fortune
fragrant
free
freed
freedom
freedoms
fresh
fresher
freshest
friendliness
friendly
frolic
frugal
fruitful
ftw
fulfillment
fun
futurestic
futuristic
gaiety
gaily
gain
gained
gainful
gainfully
gaining
gains
gallant
gallantly
galore
geekier
geeky
gem
gems
generosity
generous
generously
genial
genius
gentle
gentlest
genuine
gifted
glad
gladden
gladly
gladness
glamorous
glee
gleeful
gleefully
glimmer
glimmering
glisten
glistening
glitter
glitz
glorify
glorious
gloriously
glory
glow
glowing
glowingly
god-given
god-send
godlike
godsend
gold
golden
good
goodly
goodness
goodwill
goood
gooood
gorgeous
gorgeously
grace
graceful
gracefully
gracious
graciously
graciousness
grand
grandeur
grateful
gratefully
gratification
gratified
gratifies
gratify
gratifying
gratifyingly
gratitude
great
greatest
greatness
grin
groundbreaking
guarantee
guidance
guiltless
gumption
gush
gusto
gutsy
hail
halcyon
hale
hallmark
hallmarks
hallowed
handier
handily
hands-down
handsome
handsomely
handy
happier
happily
happiness
happy
hard-working
hardier
hardy
harmless
harmonious
harmoniously
harmonize
harmony
headway
heal
healthful
healthy
hearten
heartening
heartfelt
heartily
heartwarming
heaven
heavenly
helped
helpful
helping
hero
heroic
heroically
heroine
heroize
heros
high-quality
high-spirited
hilarious
holy
homage
honest
honesty
honor
honorable
honored
honoring
hooray
hopeful
hospitable
hot
hotcake
hotcakes
hottest
hug
humane
humble
humility
humor
humorous
humorously
humour
humourous
ideal
idealize
ideally
idol
idolize
idolized
idyllic
illuminate
illuminati
illuminating
illumine
illustrious
ilu
imaculate
imaginative
immaculate
immaculately
immense
impartial
impartiality
impartially
impassioned
impeccable
impeccably
important
impress
impressed
impresses
impressive
impressively
impressiveness
improve
improved
improvement
improvements
improves
improving
incredible
incredibly
indebted
individualized
indulgence
indulgent
industrious
inestimable
inestimably
inexpensive
infallibility
infallible
infallibly
influential
ingenious
ingeniously
ingenuity
ingenuous
ingenuously
innocuous
innovation
innovative
inpressed
insightful
insightfully
inspiration
inspirational
inspire
inspiring
instantly
instructive
instrumental
integral
integrated
intelligence
intelligent
intelligible
interesting
interests
intimacy
intimate
intricate
intrigue
intriguing
intriguingly
intuitive
invaluable
invaluablely
inventive
invigorate
invigorating
invincibility
invincible
inviolable
inviolate
invulnerable
irreplaceable
irreproachable
irresistible
irresistibly
issue-free
jaw-droping
jaw-dropping
jollify
jolly
jovial
joy
joyful
joyfully
joyous
joyously
jubilant
jubilantly
jubilate
jubilation
jubiliant
judicious
justly
keen
keenly
keenness
kid-friendly
kindliness
kindly
kindness
knowledgeable
kudos
large-capacity
laud
laudable
laudably
lavish
lavishly
law-abiding
lawful
lawfully
lead
leading
leads
lean
led
legendary
leverage
levity
liberate
liberation
liberty
lifesaver
light-hearted
lighter
likable
like
liked
likes
liking
lionhearted
lively
logical
long-lasting
lovable
lovably
love
loved
loveliness
lovely
lover
loves
loving
low-cost
low-price
low-priced
low-risk
lower-priced
loyal
loyalty
lucid
lucidly
luck
luckier
luckiest
luckiness
lucky
lucrative
luminous
lush
luster
lustrous
luxuriant
luxuriate
luxurious
luxuriously
luxury
lyrical
magic
magical
magnanimous
magnanimously
magnificence
magnificent
magnificently
majestic
majesty
manageable
maneuverable
marvel
marveled
marvelled
marvellous
marvelous
marvelously
marvelousness
marvels
master
masterful
masterfully
masterpiece
masterpieces
masters
mastery
matchless
mature
maturely
maturity
meaningful
memorable
merciful
mercifully
mercy
merit
meritorious
merrily
merriment
merriness
merry
mesmerize
mesmerized
mesmerizes
mesmerizing
mesmerizingly
meticulous
meticulously
mightily
mighty
mind-blowing
miracle
miracles
miraculous
miraculously
miraculousness
modern
modest
modesty
momentous
monumental
monumentally
morality
motivated
multi-purpose
navigable
neat
neatest
neatly
nice
nicely
nicer
nicest
nifty
nimble
noble
nobly
noiseless
non-violence
non-violent
notably
noteworthy
nourish
nourishing
nourishment
novelty
nurturing
oasis
obsession
obsessions
obtainable
openly
openness
optimal
optimism
optimistic
opulent
orderly
originality
outdo
outdone
outperform
outperformed
outperforming
outperforms
outshine
outshone
outsmart
outstanding
outstandingly
outstrip
outwit
ovation
overjoyed
overtake
overtaken
overtakes
overtaking
overtook
overture
pain-free
painless
painlessly
palatial
pamper
pampered
pamperedly
pamperedness
pampers
panoramic
paradise
paramount
pardon
passion
passionate
passionately
patience
patient
patiently
patriot
patriotic
peace
peaceable
peaceful
peacefully
peacekeepers
peach
peerless
pep
pepped
pepping
peppy
peps
perfect
perfection
perfectly
permissible
perseverance
persevere
personages
personalized
phenomenal
phenomenally
picturesque
piety
pinnacle
playful
playfully
pleasant
pleasantly
pleased
pleases
pleasing
pleasingly
pleasurable
pleasurably
pleasure
plentiful
pluses
plush
plusses
poetic
poeticize
poignant
poise
poised
polished
polite
politeness
popular
portable
posh
positive
positively
positives
powerful
powerfully
praise
praiseworthy
praising
pre-eminent
precious
precise
precisely
preeminent
prefer
preferable
preferably
prefered
preferes
preferring
prefers
premier
prestige
prestigious
prettily
pretty
priceless
pride
principled
privilege
privileged
prize
proactive
problem-free
problem-solver
prodigious
prodigiously
prodigy
productive
productively
proficient
proficiently
profound
profoundly
profuse
profusion
progress
progressive
prolific
prominence
prominent
promise
promised
promises
promising
promoter
prompt
promptly
proper
properly
propitious
propitiously
pros
prosper
prosperity
prosperous
prospros
protect
protection
protective
proud
proven
proves
providence
proving
prowess
prudence
prudent
prudently
punctual
pure
purify
purposeful
quaint
qualified
qualify
quicker
quiet
quieter
radiance
radiant
rapid
rapport
rapt
rapture
raptureous
raptureously
rapturous
rapturously
rational
razor-sharp
reachable
readable
readily
ready
reaffirm
reaffirmation
realistic
realizable
reasonable
reasonably
reasoned
reassurance
reassure
receptive
reclaim
recomend
recommend
recommendation
recommendations
recommended
reconcile
reconciliation
record-setting
recover
recovery
rectification
rectify
rectifying
redeem
redeeming
redemption
refine
refined
refinement
reform
reformed
reforming
reforms
refresh
refreshed
refreshing
refund
refunded
regal
regally
regard
rejoice
rejoicing
rejoicingly
rejuvenate
rejuvenated
rejuvenating
relaxed
relent
reliable
reliably
relief
relish
remarkable
remarkably
remedy
remission
remunerate
renaissance
renewed
renown
renowned
replaceable
reputable
reputation
resilient
resolute
resound
resounding
resourceful
resourcefulness
respect
respectable
respectful
respectfully
respite
resplendent
responsibly
responsive
restful
restored
restructure
restructured
restructuring
retractable
revel
revelation
revere
reverence
reverent
reverently
revitalize
revival
revive
revives
revolutionary
revolutionize
revolutionized
revolutionizes
reward
rewarding
rewardingly
rich
richer
richly
richness
right
righten
righteous
righteously
righteousness
rightful
rightfully
rightly
rightness
risk-free
robust
rock-star
rock-stars
rockstar
rockstars
romantic
romantically
romanticize
roomier
roomy
rosy
safe
safely
sagacity
sagely
saint
saintliness
saintly
salutary
salute
sane
satisfactorily
satisfactory
satisfied
satisfies
satisfy
satisfying
satisified
saver
savings
savior
savvy
scenic
seamless
seasoned
secure
securely
selective
self-determination
self-respect
self-satisfaction
self-sufficiency
self-sufficient
sensation
sensational
sensationally
sensations
sensible
sensibly
sensitive
serene
serenity
sexy
sharp
sharper
sharpest
shimmering
shimmeringly
shine
shiny
significant
silent
simpler
simplest
simplified
simplifies
simplify
simplifying
sincere
sincerely
sincerity
skill
skilled
skillful
skillfully
slammin
sleek
slick
smart
smarter
smartest
smartly
smile
smiles
smiling
smilingly
smitten
smooth
smoother
smoothes
smoothest
smoothly
snappy
snazzy
sociable
soft
softer
solace
solicitous
solicitously
solid
solidarity
soothe
soothingly
sophisticated
soulful
soundly
soundness
spacious
sparkle
sparkling
spectacular
spectacularly
speedily
speedy
spellbind
spellbinding
spellbindingly
spellbound
spirited
spiritual
splendid
splendidly
splendor
spontaneous
sporty
spotless
sprightly
stability
stabilize
stable
stainless
standout
state-of-the-art
stately
statuesque
staunch
staunchly
staunchness
steadfast
steadfastly
steadfastness
steadiest
steadiness
steady
stellar
stellarly
stimulate
stimulates
stimulating
stimulative
stirringly
straighten
straightforward
streamlined
striking
strikingly
striving
strong
stronger
strongest
stunned
stunning
stunningly
stupendous
stupendously
sturdier
sturdy
stylish
stylishly
stylized
suave
suavely
sublime
subsidize
subsidized
subsidizes
subsidizing
substantive
succeed
succeeded
succeeding
succeeds
succes
success
successes
successful
successfully
suffice
sufficed
suffices
sufficient
sufficiently
suitable
sumptuous
sumptuously
sumptuousness
super
superb
superbly
superior
superiority
supple
support
supported
supporter
supporting
supportive
supports
supremacy
supreme
supremely
supurb
supurbly
surmount
surpass
surreal
survival
survivor
sustainability
sustainable
swank
swankier
swankiest
swanky
sweeping
sweet
sweeten
sweetheart
sweetly
sweetness
swift
swiftness
talent
talented
talents
tantalize
tantalizing
tantalizingly
tempt
tempting
temptingly
tenacious
tenaciously
tenacity
tender
tenderly
terrific
terrifically
thank
thankful
thinner
thoughtful
thoughtfully
thoughtfulness
thrift
thrifty
thrill
thrilled
thrilling
thrillingly
thrills
thrive
thriving
thumb-up
thumbs-up
tickle
tidy
time-honored
timely
tingle
titillate
titillating
titillatingly
togetherness
tolerable
toll-free
top
top-notch
top-quality
topnotch
tops
tough
tougher
toughest
traction
tranquil
tranquility
transparent
treasure
tremendously
trendy
triumph
triumphal
triumphant
triumphantly
trivially
trophy
trouble-free
trump
trumpet
trust
trusted
trusting
trustingly
trustworthiness
trustworthy
trusty
truthful
truthfully
truthfulness
twinkly
ultra-crisp
unabashed
unabashedly
unaffected
unassailable
unbeatable
unbiased
unbound
uncomplicated
unconditional
undamaged
undaunted
understandable
undisputable
undisputably
undisputed
unencumbered
unequivocal
unequivocally
unfazed
unfettered
unforgettable
unity
unlimited
unmatched
unparalleled
unquestionable
unquestionably
unreal
unrestricted
unrivaled
unselfish
unwavering
upbeat
upgradable
upgradeable
upgraded
upheld
uphold
uplift
uplifting
upliftingly
upliftment
upscale
usable
useable
useful
user-friendly
user-replaceable
valiant
valiantly
valor
valuable
variety
venerate
verifiable
veritable
versatile
versatility
vibrant
vibrantly
victorious
victory
viewable
vigilance
vigilant
virtue
virtuous
virtuously
visionary
vivacious
vivid
vouch
vouchsafe
warm
warmer
warmhearted
warmly
warmth
wealthy
welcome
well
well-backlit
well-balanced
well-behaved
well-being
well-bred
well-connected
well-educated
well-established
well-informed
well-intentioned
well-known
well-made
well-managed
well-mannered
well-positioned
well-received
well-regarded
well-rounded
well-run
well-wishers
wellbeing
whoa
wholeheartedly
wholesome
whooa
whoooa
wieldy
willing
willingly
willingness
win
windfall
winnable
winner
winners
winning
wins
wisdom
wise
wisely
witty
won
wonder
wonderful
wonderfully
wonderous
wonderously
wonders
wondrous
woo
work
workable
worked
works
world-famous
worth
worth-while
worthiness
worthwhile
worthy
wow
wowed
wowing
wows
yay
youthful
zeal
zenith
zest
zippy


In [ ]:
%%writefile /kaggle/working/lexicons/report_verbs.txt
#########################################################################
# List of report verbs used in 
#
# Marta Recasens, Cristian Danescu-Niculescu-Mizil, and Dan
# Jurafsky. 2013. Linguistic Models for Analyzing and Detecting Biased
# Language. Proceedings of ACL 2013.
#
#########################################################################

accuse
acknowledge
add
admit
advise
agree
alert
allege
announce
answer
apologize
argue
ask
assert
assure
beg
blame
boast
caution
charge
cite
claim
clarify
command
comment
compare
complain
concede
conclude
confirm
confront
congratulate
contend
contradict
convey
counter
criticize
debate
decide
declare
defend
demand
demonstrate
deny
describe
determine
disagree
disclose
discount
discover
discuss
dismiss
dispute
disregard
doubt
emphasize
encourage
endorse
equate
estimate
expect
explain
express
extoll
fear
feel
find
forbid
forecast
foretell
forget
gather
guarantee
guess
hear
hint
hope
illustrate
imagine
imply
indicate
inform
insert
insist
instruct
interpret
interview
invite
issue
justify
learn
maintain
mean
mention
negotiate
note
observe
offer
oppose
order
persuade
pledge
point
point out
praise
pray
predict
prefer
present
promise
prompt
propose
protest
prove
provoke
question
quote
raise
rally
read
reaffirm
realise
realize
rebut
recall
reckon
recommend
refer
reflect
refuse
refute
reiterate
reject
relate
remark
remember
remind
repeat
reply
report
request
respond
restate
reveal
rule
say
see
show
signal
sing
slam
speculate
spoke
spread
state
stipulate
stress
suggest
support
suppose
surmise
suspect
swear
teach
tell
testify
think
threaten
told
uncover
underline
underscore
urge
voice
vow
warn
welcome
wish
wonder
worry
write

In [ ]:
%%writefile /kaggle/working/lexicons/strong_subjectives_riloff2003.txt
irreconcilable
accede
overjoyed
madden
torturously
exuberant
soften
inflict
rewarding
anarchism
indicative
obviate
quandary
culpable
allegation
feel
polluter
satisfactory
geezer
worthlessness
lack
embroiled
complain
disagreeable
grace
madly
serious
distressing
sagacity
contentious
luck
adorer
declaim
famed
quite
tidy
view
disorient
dungeons
utterances
inequality
yep
appalled
exterminate
attentive
indescribably
mindless
imploringly
magnificent
strife
astound
dishonesty
impatiently
nagging
draconic
adulatory
joyous
prate
spookily
underdog
excessive
baffled
oddities
untrue
damn
gabble
sensibly
gravely
puppets
scorching
eulogize
drones
mellow
blister
cataclysmically
harsh
assiduous
fantastic
menacingly
repugnantly
gifted
stunned
pious
gaily
broken-hearted
indispensable
revel
jeering
boastful
disavowal
hapless
inarticulate
raging
naughty
rubbish
stigmatize
objections
penitent
ineptly
ingeniously
spurious
overzealously
neurotically
unsettling
guiltily
unbosom
smartest
reluctance
election-rigger
simmer
commend
blameworthy
sermonize
bemoan
ideally
pine
multi-polarization
nauseatingly
disappointing
enrapt
flak
disregard
virtuously
warmhearted
condescendingly
decrepit
clever
affable
wishing
outcry
magnificence
inestimable
repugn
doomsday
upliftment
aggravate
paltry
daring
enchantingly
troubling
persuasively
intrigue
seedy
mantra
indecisively
ugly
ungrateful
impractical
lament
forbid
freakishly
petrify
chit
sympathetically
temerity
fret
discordant
asunder
wishes
mesmerize
diametrically
tortuous
dauntingly
blame
stinging
trustworthy
egotistical
boggle
leery
exuberance
sociable
tantamount
antagonize
amazed
stooge
adolescents
stingingly
nervousness
disquietude
revoltingly
altogether
fondness
admire
mourner
congratulations
excessively
preferences
reproach
mudslinging
shadow
derogatory
shortcomings
crook
inexorable
fictitious
fabrication
awestruck
bless
truthful
irreplacible
negligence
furthermore
harried
harm
peevish
slander
elaborate
ambivalent
hideous
lamentably
insightful
clearer
swore
barbarous
need
wary
admiring
raptureously
subservient
exacerbate
vicious
honestly
nurturing
creative
perceptions
sarcastic
accentuate
pledge
degenerate
dissatisfy
appreciation
hothead
outstandingly
blistering
desperate
tiringly
improper
repentance
unnerving
mastery
moralize
regretful
dismally
acerbic
whimsical
nobly
scarcely
immodest
enjoyably
attraction
nettlesome
contrariness
lyrical
noteworthy
supportive
treachery
overstatements
eagerly
contradictory
havoc
pleadingly
recommendations
vagrant
brave
luxuriant
obstacle
unconvincing
stylish
strident
delirium
forgiven
narrower
engrossing
hoard
halfheartedly
misread
strut
unequal
smart
grudges
peerless
blab
relentlessly
sacred
apprehension
curious
tranquility
adoringly
felicitate
lucid
renowned
damning
disgraceful
rile
skittish
aimless
trivially
backwardness
contention
undue
hallowed
illuminati
lurk
stifle
plunderer
assure
horrors
endearing
ignorance
grit
chasten
pompous
hideously
valiant
defy
dazed
irresistibly
meaningless
assertions
prudently
ire
prosecute
unbearablely
jeopardy
valiantly
frustratingly
scar
fastuous
terrifically
uneasily
baffling
hater
menace
cynical
loveless
dissocial
air
hanker
puny
enviable
usurper
ultimate
miracles
smokescreen
fitting
terribleness
dissembler
upbeat
wily
dangle
regrettable
object
dismalness
mighty
daydreamer
inhumanity
immorality
wise
inexplainable
ill-treated
harpy
humiliating
grudging
condolence
gladden
lonely
steadfastness
sugarcoated
facetious
forsaken
infamous
villainously
ungovernable
belligerent
gaff
disruptive
denigrate
justifiable
disobedience
pathetically
angry
battleground
disgustfully
fiendish
forlorn
impeach
imperturbable
misbecoming
rampage
luckiest
titillatingly
trusting
crippling
vengeance
serene
illusory
so-called
triumphantly
blah
fortuitously
terrified
supremely
aggression
hotheaded
unlamentable
humor
constancy
dissension
erratic
apprehensions
recklessly
rightness
abomination
passionate
worsening
gleeful
inconceivable
incontrovertible
dissention
admonishment
admonisher
gainsayer
downer
statuesque
evidently
respectfully
condone
stew
careless
hysterics
impinge
disapproval
exhilarating
magnificently
frenetic
celebrate
scorchingly
charisma
confrontation
views
desecrate
certainly
evils
glory
undignified
overlook
prettily
madder
worse
awareness
obliterate
amusing
disloyalty
appall
delectable
indulgence
wholeheartedly
support
nightmare
loathing
anguish
fetid
manipulative
improvise
eager
hallucination
jaunty
stupor
despondency
devotee
lugubrious
paranoid
super
violate
blabber
unyielding
justifiably
lawlessness
conciliate
underlings
fervid
wonderful
audacious
cartoonish
inexcusably
thoughtfulness
marvels
catastrophes
insecurity
unreasonably
empathize
sidetracked
gallingly
disaffected
shirker
disgusting
nebulously
intuitive
besmirch
lure
dishonestly
daringly
get-rich
rebuff
needfully
horrify
misguide
dread
debaser
slur
radical
insist
rollercoaster
sap
fallacy
hypocrisy
timidly
misbehavior
righteousness
want
worthy
exhort
anti-white
unfettered
disturbed-let
flagrantly
amour
obscurity
bickering
bondage
yes
loath
glad
incredibly
nimble
gloat
allay
viewpoints
vivacious
abhorrence
frazzle
deserving
perfidious
bristle
smutty
quarrelsome
gaudy
remorsefully
right
incompetent
flexible
uproar
enjoyment
rapturously
radicals
revulsive
unpleasantries
well-managed
virulently
decadent
fool
pro-peace
hysterically
calumnious
jeopardize
wrongly
improbably
glow
potent
allure
fathomless
intolerablely
tantalizingly
epitome
obstinately
gaiety
mawkishness
amiabily
nauseating
indiscriminately
menacing
demise
demoralizing
glare
direly
downfallen
erroneously
iniquitous
fanatical
pleasing
little-known
aloof
disobedient
quarrel
pleasurable
troublesome
deepening
vengefulness
wow
egotism
apprehensive
lastly
commiserate
militancy
maniacal
villian
favorable
smilingly
unthinkable
gaffe
disturbing
acridly
seething
kindliness
euphorically
celebrated
roadblocks
treasonous
chatterbox
magnanimously
ethical
law-abiding
pariah
ameliorate
backbiting
misjudgment
annoy
obtrusive
joke
defiance
farcically
indelicate
ingratiating
timid
wretchedly
overkill
naively
reward
sensible
sensitively
complicit
busybody
impatience
lurch
outrage
fearlessly
obey
spurn
belittling
funny
mess
savagely
sorrow
stress
theoretize
selfinterested
unwieldy
fantasy
suffering
dissent
letch
deceiver
glib
undisputable
blinding
exact
intimidatingly
slothful
detestably
rewardingly
indefatigable
dexterous
divisive
affected
destinies
optimism
fatuous
salvation
ape
captivating
lucky
filthy
yearningly
appreciativeness
coveting
happy
indolent
languor
unfortunate
dishonor
abhorrent
insouciance
procrastinate
entrapment
onerously
laud
caustically
obsessive
laughably
apocalyptic
sparkle
zest
neglect
passionately
confuse
illogic
ranting
damned
befriend
enslave
handy
despondent
ambiguity
doubtfully
despair
disaccord
fortuitous
bowdlerize
impolitely
mentality
inaccurately
rant
trendy
dire
partiality
anarchy
intimacy
coldly
calamitous
amazement
overzealous
fatalistic
divisiveness
imperil
glibly
grate
hardships
languish
livid
simplicity
overcome
staggering
stalemate
cuplrit
blasted
disoriented
intimidating
romanticize
dejectedly
indifference
persuasive
consternation
coward
monumentally
forgivingly
untruthful
ruthlessness
shrew
rantingly
proclaim
compliment
gracefully
kind
disrepute
liar
vouchsafe
suave
rationalize
safeguard
compel
objectionable
revulsion
insubstantial
insensitively
sugar-coated
bemused
gleefully
opulent
rejection
intransigence
amazing
glean
philosophize
agitator
intolerant
depressingly
attractively
malcontented
daze
chastise
intrude
advocacy
slanderer
spur
hilariousness
profound
detriment
featly
lecher
destabilisation
talented
surely
divisively
fearless
foolish
glum
embrace
pitifully
self-respect
appalling
shirk
unfeeling
knew
unconcerned
delightful
fortunately
joyfully
languorously
pitilessly
fury
stressfully
meddlesome
radiant
problematic
indiscriminating
grouchy
importunate
persecute
sorrowfully
gloss
mere
ought
commendable
staunch
happiness
extermination
damnably
groundbreaking
heros
humbling
woe
entirely
adventuresome
villianously
intrusive
impressively
aspiration
resounding
appreciative
ominous
sarcastically
monstrously
plaything
forbidden
self-interested
sham
smugly
impassioned
offbeat
entreatingly
well-informed
cruelties
obsessions
poverty
hedonistic
heroize
hinder
honesty
realistically
regal
begging
gainful
flirty
grin
haven
notoriously
preferable
heedless
proficient
indulgent
plagiarize
incongruous
precipitous
misinform
shamefully
uproariously
dismay
genius
flaunt
plainly
flatteringly
charm
improbable
instigate
captivation
intimate
maltreatment
remarkably
resent
alarmingly
bastard
decrepitude
dumb
impetuously
intolerance
unkindly
indecency
inexplicable
distraught
shortchange
spite
absolute
cocky
badly
devil
hawkish
insistently
dawdle
pains
remarkable
dearth
scarier
blaspheme
bountiful
congratulatory
startling
hedge
conceit
outraged
wink
meek
monstrosities
predicament
acridness
just
fundamentally
quarrellously
unthinkably
wry
willful
ill-tempered
cogent
appal
disadvantage
plenty
redemption
burden
tragic
anger
immaculate
supurbly
outcast
torment
elatedly
invalid
frustrated
petrified
scrupulous
reprove
veritable
bloodshed
destitution
unhappiness
critics
yeah
adoring
elegant
penalize
fickle
inconveniently
marvel
pragmatic
disinclination
solicitude
wilt
fond
frictions
sad
acerbically
heartless
easygoing
fascinating
patiently
remunerate
aggressiveness
chastisement
drab
well-regarded
diatribe
fallacies
distaste
ingenuous
fortitude
sorry
sputter
inclinations
desertion
dunce
invidious
limp
perfunctory
phenomenal
pandemonium
respite
upheaval
god-awful
irony
arrogantly
liberate
miscreants
mistrust
rash
magnanimous
unfaithfully
inherent
extraordinary
immovable
merciful
celebratory
deplorably
fatuity
arresting
grouse
imperialist
rid
flagrant
saintliness
convenient
depraved
abidance
dizzingly
insinuation
displeasure
brimstone
insidiously
pardon
peeve
apathetic
disdainful
enthralled
hopelessness
boil
exemplary
acclaimed
idiocies
awkwardness
admonition
injustice
jauntily
disillusion
reprehensibly
troublingly
twist
grievously
deny
grossly
sanction
barbaric
dodge
complimentary
hearten
ferociously
studiously
absolutely
despairingly
offend
frustrating
disappointingly
indiscriminate
pleas
meddle
resourcefulness
eschew
worsen
innuendo
heroine
evocative
puzzlement
convincing
graceless
deduce
imprudence
desirable
respect
worrisome
laugh
licentiously
cheater
guilt
proscription
bizarre
joyful
dreadful
erratically
nice
posh
ulterior
fascinate
unfavorable
incorrectly
indomitable
rattle
poeticize
wellbeing
unravel
carp
haughty
cataclysmic
falter
insightfully
visionary
blithe
growl
irregardless
frankly
atrophy
banalize
premeditated
scathing
agreeably
discourteous
delude
impregnable
babble
wince
travesty
castigate
bogus
propagandize
quack
banish
annoyance
dexterously
oppression
nightmarishly
brutally
seriously
swindle
unfairly
fervor
morbid
gladly
integrity
exciting
compliant
masterfully
reassurance
rectification
belligerence
rude
beg
underestimate
paranoia
disconcertingly
alas
amenable
confusing
bestial
corrupt
cowardly
devastating
disinterested
inoffensive
mum
joyously
stifling
boiling
understand
ambivalence
immoderate
improperly
jaundiced
adventurism
disgustingly
mordantly
dim
weary
endear
contend
discredit
ridicule
triumphant
relish
furor
dangerousness
coercive
dubitable
mysterious
hypocritical
obedience
ingratiate
complement
obsession
misunderstood
squander
disinclined
poison
tint
beauty
curse
despoil
shyness
thoughtlessly
peculiar
sincere
stern
antithetical
extoll
bafflement
amaze
inimical
satisfactorily
arrogance
demonize
gruesome
affirmation
excitedly
forlornly
hell-bent
like
complaints
appreciatively
determination
eviscerate
unwise
huckster
rumor
ardently
glimmering
cheerful
daunt
plausibility
prodigiously
gracelessly
awed
mendacity
anarchistic
repulsively
ally
indefensible
irritable
foolishness
beneficiary
motivated
vex
villify
disrespectfulness
abhorrently
reckless
tattered
desolate
false
illusions
conciliatory
bungle
exasperatingly
foremost
spoon-feed
opportune
brashly
flattering
wrath
shriek
astounded
meanness
well-wishers
conspire
placate
zeal
posturing
obese
overdone
frantic
ordeal
illumine
cheap
therefore
demand
oddity
imperatively
derisive
inhospitable
doom
unpretentious
paramount
stupidly
disrupt
indestructible
priceless
luster
providence
disagreeably
brains
lawbreaking
desiccate
inspiring
glistening
fault
blunt
crusader
insufferably
bloodthirsty
polluters
re-conquest
temper
foulness
malicious
vindictiveness
nosey
emphatically
extemporize
unseemly
cruelty
inkling
protector
phobia
racists
bland
fussy
resentment
surpass
unequivocally
asinininity
considerate
caustic
degrading
err
enormities
exploitation
absurdity
astute
infuriating
neglected
senile
urgency
shortsightedness
smoldering
unnervingly
forgave
turmoil
dispiritedly
loathly
arrogant
pauper
senselessly
hardship
persuade
squarely
infuriated
ultimatum
nuisance
quibble
despicable
incompetently
dissonantly
indignation
unquestionable
venom
fame
irk
low-rated
imagine
gratitude
precaution
ponder
catastrophe
preach
maladjustment
malcontent
mire
uncommon
closeness
venomously
incomprehension
discriminatory
salutary
shimmeringly
utterly
vulgar
craziness
flimflam
presumptuously
enormous
astonishingly
delusional
insubstantially
vehemently
outrageousness
unfazed
facetiously
timidity
unimportant
stunning
believable
debase
throttle
angriness
rejoicingly
disrespectfully
greet
electrification
mock
aha
prosper
exorbitantance
revolting
unprecedent
vengeful
spectacular
glorify
horrendous
irrationality
luxury
hardball
accolade
exceeding
pacifist
self-determination
ultimatums
insult
witty
illustrious
trivialize
glee
heretic
profusely
bitchy
inconsiderately
pervert
shy
laconic
fatalistically
belie
admiringly
plague
smolder
tingle
catchy
gawk
unrest
incommensurate
beneficial
shamefulness
insincerity
pleasurably
shimmer
unwanted
handily
outlaw
demystify
ecstasy
partisan
vie
evildoer
frustration
mindful
gibe
pride
laudable
abusive
detrimental
shameful
lovely
worried
biting
inextricably
deplorable
drastic
lame-duck
imperative
misjudge
rejoicing
bravado
irksome
fearsome
martyrdom
impudent
hardhearted
exultingly
invaluablely
kook
profoundly
graciousness
jubilate
disloyal
lie
dauntless
alliances
dragon
excellence
overwhelming
sugar-coat
oppressiveness
scars
destiny
extravagantly
perversion
contortions
strictly
unlucky
worries
knowingly
sloppy
denounce
extraordinarily
inarguable
lofty
viciousness
hostility
surmount
correctly
treasure
infallibility
idiotically
malice
inundated
preaching
incendiary
bewail
deprave
haggle
breathlessness
grating
hardly
tragically
leech
worrying
thusly
unfaithful
confidence
delightfully
infringements
insanity
traitorous
famine
misconception
vengefully
delighted
willfully
glaringly
bold
hypocrites
trick
despotic
stiflingly
diabolical
nebulous
unacceptable
noble
dramatic
playful
sanity
tauntingly
comrades
hatred
sardonically
aggravating
indoctrinate
longing
skulk
abyss
nefariously
indomitably
venerably
kemp
emptiness
esteem
splendor
affront
conspirator
acrimonious
unabashedly
cloud
flounder
patience
gorgeously
surge
passive
perfect
grievous
stylishly
friction
astronomically
avarice
insensitivity
greatest
sunder
expunge
worry
impervious
endurable
ingratiatingly
attitude
cheer
destructive
indifferent
cheery
irreverent
savages
troublesomely
enrage
antagonism
fusty
entrap
killjoy
offensiveness
disquietingly
horrid
coercion
hail
impatient
mulish
stranger
grateful
cognizant
expertly
far-fetched
troublemaker
debaucher
brat
painstakingly
atrocities
discerning
ill-usage
ugh
incautious
euphoria
dissatisfied
importune
profuse
eminent
sublime
astronomic
best
infidels
rumors
aggravation
amazingly
dehumanization
salute
fuss
astonishment
depressing
pillory
disbelieve
freakish
kaput
sneering
confusion
unhappily
unpleasant
disrespecting
bitch
disbeliever
lacking
eminence
heck
think
yawn
selfishly
insolently
teasingly
reviled
inconsequent
incoherence
agreeable
frighten
invidiously
disobey
might
diabolic
unpopular
impudently
marvelously
domineer
bloated
manipulators
brutal
effortless
agonizingly
inconvenience
better-than-expected
mania
panic
desperation
monster
sanctuary
stubborn
considerably
vouch
drunk
spoon-fed
diabolically
illogically
abasement
exonerate
irrationally
miscreant
quarrellous
discourage
grumble
insupportable
upsettingly
pugnaciously
moral
excellent
haughtily
nastily
admittedly
obstinate
rotten
shrewdly
dreary
astutely
pernicious
deplore
impending
insidious
exclaim
unfriendly
estranged
savage
grievances
hamper
reverent
rancor
incomprehensible
taboo
perverse
farce
monotonous
hm
inordinate
distrusting
skeptically
acrimoniously
justly
daunting
grind
misfit
fortress
forgiveness
fidelity
enthusiast
horrific
impeccably
contrived
anyway
denunciate
studious
autocrat
discomfititure
exasperation
extravagant
felicitous
imaginary
ill-mannered
ineloquently
desolately
enthusiasm
plead
disavow
cautionary
dissuasive
sentimentality
unscrupulous
egotistically
immaculately
perversely
anyways
dear
disappointment
intolerable
implore
insupportably
extensively
manic
uplifting
disputed
mockery
one-sided
contrive
distortion
grimace
humiliation
eloquence
conspicuously
wicked
amicable
perceptive
defective
uneasiness
unrealistic
illusion
rumbling
spitefully
yearn
condescension
abscond
astronomical
flourish
courtly
insatiable
misunderstandings
qualms
regrettably
inevitably
reverently
maximize
compassionate
pain
remorseless
unnaturally
tolerance
laughable
incorrigible
blasphemous
impasse
ascendant
inviolate
obedient
distraughtly
gracious
condescend
joy
calumniation
endorse
cannibal
tremendously
desolation
oblivious
peaceable
rapport
chivalrous
demonic
chide
somber
gullible
clearly
courageously
defiler
mawkish
smiling
fawn
admiration
discontented
irredeemable
flawless
unwilling
stereotype
disinterest
ridiculous
insulting
brutalize
joyless
tantalizing
tramp
immoral
surprising
unwisely
allusion
flabbergast
scandals
debacle
unlamentably
impiety
mortifying
guile
immensity
awkward
tyrannical
happily
utter
ingenious
activist
enthusiastic
fearfully
awesomely
awful
inopportune
melodramatic
edify
oppress
stellarly
contradiction
promising
unimaginably
frantically
scrutinize
sensitivity
preposterously
prevaricate
midget
belittle
frenzied
imprudent
insinuating
feelings
best-performing
lackeys
mundane
outright
run-down
excruciating
wretchedness
must
aspersion
defile
homage
harmless
prognosticate
moreover
beast
recoil
whips
depressed
liking
non-confidence
floundering
freak
pathetic
terribly
distastefully
mishap
sensation
undoubtedly
deadbeat
contort
dreadfulness
greatly
mesmerizing
hunger
smack
weird
cliche
agitation
overstatement
rue
travesties
sillily
undependable
chafe
impressiveness
accursed
disquieting
stagnate
vilify
recalcitrant
stunningly
worthlessly
charismatic
provoke
pugnacious
regret
boldly
ecstatically
belligerently
relentless
sure
nevertheless
touchy
goad
incredulously
humiliate
slanderously
disturb
hopeless
ill-natured
admonish
perplex
vociferous
criticize
maledict
sworn
terrifyingly
worrier
biased
braggart
mislike
disingenuous
complaint
celebration
confused
gratuitously
forgetfully
haunting
rectifying
shambles
enjoyable
last-ditch
assess
greed
condemn
loveliness
macabre
scream
serenity
ax
catastrophically
wonderous
unorthodoxy
enticing
jeer
debatable
riled
ineloquent
impaired
deluded
foully
paradoxically
raving
seethe
vanquish
irate
licentious
blandish
farfetched
seamless
trustingly
groundless
thirst
deadlock
anti-occupation
puzzle
mollify
pinnacle
antiquated
asininely
lying
scum
pledges
timidness
ashamed
unqualified
jovial
prowess
raptureous
ramshackle
oh
intelligent
fascinatingly
subjugate
meaningful
goodwill
ebulliently
adamantly
undermine
divine
pity
polarisation
falsify
foolishly
terrible
backward
lier
ghastly
monotony
levity
reproachful
confide
baseless
renunciation
rhetorical
inefficacy
scoff
shortsighted
steadfastly
obsessiveness
graciously
swear
horrendously
reflecting
rejoice
herald
unwarranted
indoctrination
insanely
rally
arousing
startlingly
adulterate
chic
incisiveness
favour
hatefully
alarming
fashionable
jaded
gloom
shamelessly
swagger
contemplate
devious
hilariously
inevitable
knowing
mean
provocation
failing
extreme
harmonize
forsake
mockeries
mournful
relief
friendliness
ignominy
ruthless
scourge
pithy
ache
cackle
pretentious
satirize
insight
fanatic
redeem
boast
scornful
acrimony
debasement
adulation
blissfully
calamities
courage
exalting
felt
oppositions
wariness
scheming
snub
virulent
cataclysmal
prideful
promoter
truthfulness
jealously
racist
spectacularly
wickedness
uneasy
irreversible
succumb
upbraid
savagery
unanimous
craven
exultation
negligent
oppressive
inefficacious
idolized
gloriously
ironies
entreat
hopes
miserably
thrillingly
iniquity
misery
boredom
harshly
inflame
upside
infringe
reprehensible
splayed-finger
mourn
outshine
uproarious
humane
boundless
forgiving
illuminate
insensitive
unrelentingly
condescending
faithfulness
misleading
flawlessly
undependability
sympathetic
inspirational
chaff
unleash
bedlam
futilely
hardier
invincibility
quarrels
venomous
inestimably
grudge
smug
dare
jubilation
dissatisfaction
awesome
distrustful
hindrance
irreverence
overact
profane
calming
sensations
avalanche
perseverance
deceit
distort
romantically
belief
best-known
repulsiveness
lamentable
striving
richness
gallant
austere
outsmart
thoughtful
applaud
indecisive
righteous
blatant
gusto
mad
matchless
honor
audaciousness
peevishly
calumniously
stigma
shoddy
revenge
denial
incredible
cross
envious
sinful
quash
orthodoxy
tawdry
virtuous
fallout
unsound
insolent
standstill
offenses
conceited
outdo
glowingly
fiasco
refute
dubious
sadly
dilapidated
fallaciousness
superfluous
fruitlessly
excel
prickles
dismayed
ejaculate
conveniently
dissatisfactory
inglorious
appallingly
kooky
devastatingly
assassin
inhospitality
supreme
fervently
admonishingly
friggin
savvy
self-coup
slaves
mirth
runaway
awfulness
brute
mm
disconsolate
keenly
monstrosity
sass
disturbingly
gripe
defender
hale
infidel
inflexible
shrivel
dishonest
downhearted
torturous
despise
hug
lovably
repugnance
unjustifiable
hasty
perfectly
reactionary
scandalous
rosy
wiles
personality
inconsolably
superstition
dogged
mislead
inconsequentially
quitter
scowl
desiccated
brag
clear-cut
fears
allegorize
despite
dragons
glaring
enraptured
abash
licentiousness
stirringly
pleasure
moderate
generous
bother
seriousness
contemptuously
inferior
calamitously
remorseful
hardheaded
relent
quasi-ally
subjugation
famished
deadweight
distasteful
understated
effrontery
fervent
enthrall
incontestable
indisposed
desert
heathen
disapproving
injudicious
lavish
merry
conceivable
batty
denunciations
fleer
trauma
regardlessly
recklessness
kingmaker
cursed
devotion
nastiness
unrelenting
unreliable
prattle
incognizant
glossy
shamelessness
dispirited
trumpet
palatable
insulted
jubilant
fascism
stars
lavishly
shimmering
unfounded
advocate
virus
astonishing
mordant
disgusted
incisive
abject
disdainfully
incoherently
sneer
ineptitude
inessential
merriment
resourceful
debauch
precautions
aggrieved
disgustful
savaged
friendly
stirring
deviously
indecision
deploring
officious
fact
criticism
lewdness
unabashed
disparage
fractious
light-hearted
inferiority
fatuously
finally
embarrass
exasperate
contentment
exhilarate
lucidly
terror
dissonance
hope
darn
impure
beauteous
thankfully
comprehend
court
uncivilized
uncommonly
mischievous
spitefulness
mortify
inequitable
weariness
disapprove
joker
beloved
bungler
anarchist
discern
evil
frightful
moribund
vexation
hoax
dodgey
infringement
inundate
beneficent
agitated
enchanted
disconsolately
cajole
downbeat
dupe
forebodingly
forgetfulness
gloatingly
heroic
intriguingly
jealousy
moving
respectable
understood
fantastically
maliciously
villainous
shady
myth
weirdly
luckily
supposing
willingness
insincere
balk
sheer
regardless
terrify
selfishness
fundamental
ardent
misgiving
trump
gorgeous
hate
ecstasies
dedicated
brusque
defiantly
payback
violator
onerous
ignominiously
screamingly
allusions
beggarly
sweeping
pillar
skillfully
confess
slime
spiritual
stuffy
whine
mercilessly
prestige
despairing
incapable
whimper
berserk
shortcoming
pressing
zealously
world-famous
prize
madman
chaos
genocide
enticingly
scolding
loopholes
unsophisticated
phenomenally
promise
procrastination
disreputable
actuality
complacent
hatefulness
earnest
indiscretion
hesitant
pro-Cuba
rage
passe
pro-Beijing
overdue
resplendent
pleasingly
seductive
indelible
rectify
bias
beleaguer
mismanage
perversity
thoughtlessness
taunting
tolerantly
worryingly
hysteric
notably
favorite
unkind
delightfulness
brutalising
drunken
graceful
considerable
incoherent
lifeblood
outstrip
anxiousness
proactive
contempt
hopefulness
malevolent
brazenness
tediously
agree
puppet
anxiously
hateful
fainthearted
direness
deceive
ignominious
vileness
perfidity
anxieties
explicit
animosity
unattractive
unselfish
forget
disastrous
facts
entrancing
heinous
heartily
distrust
enduring
spade
superlative
suspiciously
war-like
outrages
scold
obsess
deception
spotless
malevolence
reprovingly
wisdom
rack
unworthy
arduously
maddeningly
stodgy
stooges
disgrace
heartfelt
truculently
enthusiastically
emotional
weaknesses
respectful
palliate
endanger
befit
empathy
vehement
consent
guiltless
stridently
alliance
boisterous
frighteningly
hideousness
refutation
agreeableness
maliciousness
doubtful
annoyed
lively
thus
admit
grief
greedy
invigorating
instigators
misdirection
winners
boring
morality
offensively
phony
sulk
traduce
indecorum
distraction
agonies
illuminating
nervously
wonder
smitten
picky
indiscreetly
incisively
polemize
masters
brook
nemesis
frightening
earnestness
mournfully
ploy
compulsion
revile
breathtakingly
gratefully
uncompromisingly
traumatize
aghast
avenge
inconsequential
mirage
capriciously
blundering
scrutiny
stride
gruesomely
disconcerted
dishonorable
disrespectful
hellion
thrilling
victimize
exasperating
demean
rightly
spellbindingly
expansive
though
immaterial
poignant
perplexing
impurity
worriedly
doldrums
exuberantly
tyrant
inclined
lunaticism
perspicuously
wretch
unfit
furious
charmingly
harmony
adored
baffle
really
content
mindlessly
covet
laudably
botch
chagrin
relentlessness
inexcusable
almighty
justification
fiend
bemoaning
dragoon
lone
embroil
appreciate
fallaciously
insufferable
sage
peril
pretend
delirious
privilege
fun
self-interest
calumniate
derisively
overworked
shatter
infallible
recommendation
sinisterly
exalted
perish
stinking
goading
righteously
smuttiest
thinkable
smartly
fudge
falsely
plot
fastidiously
degradation
hegemonistic
scoundrel
lies
astonish
reprehension
thank
taunt
affectionate
credible
solicitous
avariciously
cure-all
fearful
outrageous
revealing
likable
catalyst
quicken
unjustly
traumatized
devilry
merrily
pray
contemptuous
imply
insultingly
soliloquize
self-humiliation
beastly
strange
barbarity
dismissive
bad
indescribable
embodiment
glitter
ridiculously
skepticism
heavyhearted
striking
faith
adorable
disillusioned
spoil
civility
ovation
vitality
falsehood
invalidity
imminence
smile
rapture
scapegoat
byzantine
faithless
inaccurate
idiot
stricken
please
pout
pleased
angrily
calamity
gauche
agonize
gaga
masterpieces
prejudge
sentiment
aver
wisely
ill-favored
downright
digress
gratifying
bereavement
motley
brutish
luxurious
disservice
arcane
pleading
hopelessly
fib
squash
deceitfully
displeasing
impotent
lovelorn
disrespect
gumption
incorrigibly
sensationally
poise
untimely
upsetting
plush
dastardly
kick
consequently
ingenuity
reprehensive
disarray
sympathy
excited
sprightly
imploring
unsettlingly
cornerstone
harassment
traumatically
atrocious
strikingly
floored
eclectic
sagely
effortlessly
dreadfully
implausible
distracting
winnable
lurid
passion
vent
sexy
sanguine
miraculous
feckless
covetous
melancholy
wondrous
cliched
inconsolable
impugn
destitute
welcome
debate
hmm
accuse
egregious
compensate
presume
entertaining
disgustedly
drastically
anti-US
frenzy
vibrant
exploit
overemphasize
imaginative
cute
pillage
blockhead
enjoin
knowledgeable
apotheosis
deserve
tantrum
taint
hassle
bereft
imprecate
misbecome
adulterated
fastidious
positively
scrupulously
cranky
dismaying
cry
mistake
pessimistic
stately
cherish
giddy
comely
execrate
disconcerting
scummy
impetuous
bliss
confident
inexorably
bitterly
impel
foolhardy
stupified
ultimately
obscene
distressingly
discordance
toughness
sincerely
pertinaciously
tenaciously
luxuriously
carelessness
twisted
satirical
gladness
spiritless
accuses
inapt
foretell
dismayingly
explicitly
selfish
well-received
amuse
inept
torrid
inanely
censure
ill-fated
draconian
persevere
inane
repay
exorbitant
indignantly
uncouth
godlike
zenith
sufferers
pretence
willing
bane
misreading
bellicose
self-destructive
toil
uproarously
oasis
tenacious
torridly
capriciousness
pundits
hubris
beneficially
humorous
propitious
fidgety
imperiously
smoulder
avidly
fawningly
workable
misunderstand
solidarity
irrational
gratify
harangue
inequities
pittance
ideal
impulsive
rankle
true
partisans
mercy
gainsay
pedantic
pugnacity
shunned
vibrantly
reprimand
scare
sin
faithful
hypocrite
calumny
disheartening
infirm
liars
rumple
terrific
merit
hopefully
unethical
biases
inflammatory
cherub
prejudicial
hopeful
lechery
irritate
finely
burdensomely
dissuade
debauchery
frazzled
saint
inklings
unjustifiably
stubbornness
wonderfully
woebegone
devoted
aware
merely
spellbound
spiteful
highlight
manifest
novel
bitter
discouragement
fallacious
toast
flat-out
loathsome
vain
indelibly
alluringly
monumental
hilarious
maniac
fanaticism
rail
disaster
lovable
barbarian
prodigy
prefer
dramatically
bewildered
avaricious
presumptuous
fibber
irredeemably
superficiality
dissidence
jeeringly
tremendous
damnable
flatter
berate
criticisms
unbelievable
blindingly
worthiness
enthuse
fraternize
agonizing
mediocrity
farcical-yet-provocative
shrewdness
astoundingly
misconceptions
compulsive
demoralize
arbitrary
caricature
dastard
diatribes
heartbreaker
incapably
mockingly
lackluster
shameless
wonderously
touches
awe
devilishly
heavy-handed
sloppily
brazen
degenerately
rusty
coerce
barbarously
gall
magic
spank
bewildering
listless
motivate
understandable
break-point
over-acted
pretense
apprehensively
grouch
fanciful
ingenuously
dissonant
notorious
stupify
infallibly
mocking
lust
radicalization
apostle
indiscreet
apathetically
woo
beautifully
discouraging
dishonorablely
sucker
drunkard
crazily
excoriate
love
absent-minded
bullyingly
warily
shun
likewise
frustrate
trivial
tiresome
obviously
inspire
lionhearted
renown
astonished
disrespectable
effigy
lush
fear
grandeur
hilarity
hysteria
richly
glowing
renounce
condemnation
abominably
rumours
cherished
oppressively
vestiges
adulate
distinguished
incompetence
shockingly
insincerely
scathingly
bicker
vindictive
accolades
dispirit
extravagance
blurt
offending
profanity
dissolute
jolly
preponderance
preferably
foreboding
impedance
sobering
banal
bonny
disgracefully
feeling
scornfully
farcical
gawky
desultory
fulminate
opinion
credence
disregardful
smother
derisiveness
mishandle
tempt
disappoint
edgy
long
triumph
formidable
indignity
dizzy
clumsy
dispiriting
dullard
adroitly
eternity
bitingly
ignorant
maddening
unclean
fidget
fanfare
blasphemy
wish
goodness
revere
bleak
deservedly
mightily
patriotic
horrifyingly
engaging
dextrous
scary
abrasive
ardor
aggressor
filth
neurotic
plight
sickeningly
infernal
limitless
crave
afflictive
conspicuous
exaltingly
weaken
intransigent
fragrant
heartbreak
prophesy
hamstring
strict
disbelief
forsee
deference
shocking
illogical
languid
receptive
jest
charming
abominable
mediocre
dishearten
concur
lonesome
feisty
hospitable
cheat
deceivers
unquestionably
delusions
alarmed
angelic
uphold
doggedly
unfair
abhors
critic
terrifying
difficulty
scared
unnerve
brash
peculiarly
lorn
domination
elation
plausible
concerned
horrible
sully
delicious
discontinuity
mendacious
inconceivably
advantageous
ill-advised
understatedly
skillful
treason
unjust
deprived
halfhearted
famously
annoying
despotism
memorable
pervasive
bait
renaissance
unconvincingly
self-serving
absurd
tempting
idol
excellency
loathsomely
squabble
sickly
insane
brainwash
tranquil
fervidly
truly
inarguably
vital
obediently
audacity
shame
ennoble
liability
scoffingly
bewitch
meritorious
watchdog
okay
anxiety
compassion
distressed
unsuspecting
valuable
demoralizingly
harmonious
jerk
praise
infer
captivate
worth
unorthodox
memorialize
mortification
healthful
keenness
gratuitous
deceptive
hoodwink
shyly
stereotypical
smuttier
desirous
gratification
second-class
recommended
scarred
tedious
extremely
sympathies
grotesque
smarter
recommend
revengefully
gape
deceptively
stir
fabulously
swamped
yearning
credulous
defrauding
disgruntled
so-cal
despicably
franticly
loathe
vice
superb
endorsement
aspirations
strangest
bore
goddamn
truth
flirt
erudite
aggressive
ludicrous
euphoric
effusiveness
inculcate
zealot
miraculousness
envy
benevolent
disconsolation
hell
viciously
beautiful
honorable
ruthlessly
puzzling
sympathize
fell
curiously
infuriate
cripple
ill-conceived
inescapably
tenacity
overbearingly
inelegant
spook
crooked
naive
questionable
wickedly
scariest
incessantly
trickery
paradoxical
deprive
effusion
mesmerizingly
thoughtfully
disgruntle
decadence
irreparable
admirably
malevolently
satisfaction
egocentric
stammer
embarrassing
beatify
unprecedented
rightfully
brazenly
lover
exactly
inequitably
prodigious
outstanding
carefree
saintly
ruffian
imbroglio
lazy
disfavor
thoughtless
discontentedly
hard-liner
impudence
immensurable
apologist
rascal
picturesque
tyranny
shroud
uttermost
indecently
emphatic
fretful
impressive
betray
impropriety
rift
divinely
believe
childish
mistrustful
avid
laughingstock
rife
unbelievably
harmoniously
dazzling
dispute
plunder
master
hero
acerbate
crazy
accusation
admirer
discomfit
imagination
luckier
rational
exceptionally
rapturous
simplistically
tainted
woeful
adulteration
irately
fustigate
misinformed
earsplitting
realistic
invaluable
judicious
mistaken
nightmarish
feels
gloomy
lark
abjectly
fashionably
losing
devilment
distraughtness
especially
coherence
opinionated
plebeian
excellently
solicitously
shrouded
beseech
disagreement
instigator
amiable
enormously
damnation
diffidence
dissidents
consummate
maverick
invective
scruples
better
annihilate
galling
spooky
unwillingly
meager
enviously
star
wheedle
superbly
awesomeness
horribly
maybe
averse
molest
hollow
nuances
ill-treatment
provocative
well-intentioned
mangle
impulsively
commotion
dashing
fascination
leer
melodramatically
decry
imbalance
implode
obtuse
abhor
arouse
asinine
hypocritically
delight
agony
invidiousness
surprise
dote
candid
regretfully
merciless
repugnant
mercifully
awfully
sabotage
crafty
disposition
hegemony
tyrannically
lackadaisical
amicably
accusations
rapt
tact
enviousness
condemnable
fume
tired
inhumane
inveigle
ambiguous
concern
bitterness
generously
immediately
flabbergasted
inescapable
heroically
lax
needful
sinfully
stereotypically
ragged
darling
litigious
discountenance
cumbersome
spookiest
antipathy
overbearing
scandalize
frolic
disdain
heckle
unbearable
humour
dejection
devoid
shrilly
lewdly
poised
lurking
smut
disruption
humorously
brutality
spoonfed
impious
ill-used
halcyon
skeletons
insolence
nicely
absurdly
emotion
autocratic
deride
wrangle
critical
immensely
incredulous
worthwhile
agitate
exaltedly
palatial
skeptical
glower
ravage
earnestly
endorser
vainly
degeneration
disvalue
blather
playfully
proscriptions
ultra
opponent
cataclysm
handsome
severely
grandiose
altruist
betrayals
great
amusement
outburst
immorally
blessing
tricky
sardonic
prudent
conspiratorial
blameless
loot
radiance
torrent
staunchness
consensus
thwart
flout
dreamland
overacted
gibberish
disrespectablity
metaphorize
sumptuous
unequivocal
convincingly
abysmally
denunciation
preposterous
justify
cutthroat
miserly
stressful
disturbed
imbecile
oppressors
apocalypse
dazzled
idealize
banishment
curses
forgetful
grave
irreverently
lewd
rebuke
bewilderingly
aversion
refreshing
obsessively
self-satisfaction
flighty
grudgingly
gibber
scandalized
brainy
soothingly
goof
superficially
understate
surmise
devilish
dejected
beguile
bewilderment
idyllic
overwhelmingly
unfortunately
worst
sloth
bombastic
shark
shrewd
slanderous
blatantly
pro-American
enfeeble
supremacy
altruistically
heretical
mistrustfully
stink
callous
effusive
pratfall
triumphal
embroilment
reject
realist
kindness
baby
antagonistic
misrepresent
insinuate
impose
cheapen
inhuman
rank
cruel
ferocity
pitiless
affirm
felicity
brilliant
loneliness
oddest
vexingly
defamations
remorselessly
unnerved
invincible
acclamation
spellbinding
indulge
attest
moderation
seemingly
congratulate
grisly
bewilder
momentous
deceiving
rebellious
cogitate
sacrifice
inordinately
apologists
perverted
staggeringly
undoubted
jabber
interesting
prickle
rather
resentful
well-connected
dismal
lawbreaker
outrageously
unspeakable
candor
devastated
precious
entice
frenetically
gratifyingly
grieving
wholesome
wound
jealous
abhorred
blissful
disparaging
sadden
exorbitantly
pertinacious
contemptible
obnoxiously
scarily
delusion
knowledge
allegations
itch
brilliantly
magical
fabulous
unsettle
anti-Semites
exalt
lascivious
surprisingly
absolve
vow
togetherness
enmity
lividly
corruption
gossip
perplexed
splendidly
spookier
maxi-devaluation
cheerless
longingly
degrade
befitting
sly
defame
forswear
perhaps
vociferously
inconstant
proficiently
encouraging
complaining
garish
indubitably
perspicuous
haste
reassure
thankless
thrill
urge
disgraced
implacable
disconcert
impenitent
equivocal
concerns
grim
rightful
demon
fraught
majestic
discontent
befoul
contradict
deviousness
inattentive
despised
left-leaning
molestation
optimal
excitement
heartwarming
undaunted
detestable
inappropriate
absurdness
chatter
glisten
indignant
sentiments
fruitless
extortion
dehumanize
stubbornly
trustworthiness
perfection
amiability
pettifog
ego
obliged
sensationalize
rampant
intriguing
sensational
taunts
impediment
grievance
trouble
ugliness
know
hallucinate
puzzled
massacre
loyalty
sorely
vile
shabby
hysterical
arduous
immoderately
indispensability
overblown
fanatically
ebullient
sophisticated
subservience
crass
untrustworthy
servitude
egregiously
recant
set-up
heartbreakingly
stupid
defamation
distress
excusable
passiveness
vindictively
cravenly
fanatics
inquisitive
brutalizing
rifts
sufferer
insociable
beautify
aspire
downside
thankful
better-known
poorly
smouldering
yelp
thorny
ecstatic
purposeful
miserable
staunchly
agreeability
forgive
heartrending
uproarous
bereave
blunder
deft
devastate
ominously
stark
obnoxious
misbehave
wanton
lackey
nix
enormity
sentimentally
belabor
mistakes
faithfully
derision
irritation
pretty
sneeringly
disastrously
panicky
belittled
anti-social
commendably
elated
collude
knave
overdo
shiver
slap
villianous
indeed
lustrous
indisputable
harrow
domineering
imminently
stellar
adore
articulate
injustices
battle-lines
audaciously
horrifying
dizzing
merriness
steadfast
fondly
craving
exaggerate
ingratitude
counterproductive
greatness
catastrophic
phobic
irritating
plotters
repent
radically
upliftingly
elegantly
vindicate
heartbreaking
disquiet
unreasonable
doubtless
veto
nasty
apathy
acumen
bleakly
stimulating
defamatory
jubilantly
coax
gutsy
congenial
betrayal
encouragement
imposition
notable
miseries
misleadingly
deceitful
unspeakablely
weakening
irrelevant
marvellous
abase
heady
time-honored
deprecate
disprove
eloquently
remorse
doubts
confession
scandalously
rousing
titillating
dream
punish
silly
craze
despot
well-run
petty
irritated
sanctity
insistent
foul
luckiness
dilly-dally
indubitable
extol
destroyer
drain
irrelevance
attractive
insights
adamant
contravene
hard-line
inviolable
mischievously
enchanting
mysteriously
stupidity
brashness
argumentative
horror
refuse
amiss
exhilaration
weakness
brutalities
harmful
kindly
dissemble
lech
sadness
inextricable
miracle
asperse
clout
revengeful
enraged
favor
incessant
calumnies
pacifists
irresolute
bonkers
outbursts
unwillingness
admirable
repudiation
impair
crusade
fixer
adversity
frightfully
evaluate
nervous
starkly
positiveness
patronize
backbite
well-being
ferocious
anti-Israeli
downcast
repulsive
lunatic
breathtaking
gruff
suspicious
covetously
sickening
cynicism
sumptuously
refuge
bully
annihilation
shiny
utmost
misstatement
eloquent
unusually
opinions
unwelcome
misbegotten
goddam
exquisitely
praising
boldness
oddly
idiotic
mistakenly
astounding
bothersome
contrary
inconsequently
upheld
consider
severity
foresight
dismissively
inconsiderate
dummy
entrenchment
exhilaratingly
assiduously
disgust
fortune
imposers
reflective
undesirable
despondently
incivility
concede
mudslinger
wretched
excruciatingly
screech
intelligible
rave
irritably
idolize
burdensome
laughter
nonsense
disunity
exquisite
wail
hamstrung
proud
stale
martyrdom-seeking
disorganized
misguided
definitively
foe
chaste
romantic
dissident
condolences
enviably
abuse
discouragingly
brood
askance
misinterpret
invulnerable
superstitious
confessions
innocuous
villains
entranced
righten
senseless
accusingly
fancy
fortunate
miserableness
salacious
reverence
aberration
harridan
anxious
pessimism
tiring
daydream
presumably
stagnation
burn
humankind
covetingly
disparagingly
excitedness
fatefully
liberty
mar
slashing
unipolar
fascist
mortified
affection
disaffirm
imminent
reminiscent
sleazy
deject
defiant
enervate
manly
courteous
fright
improbability
overstate
truthfully
venerable
untenable
sarcasm
dislike
nuance
undisputably
unforgettable
emasculate
privileged
courageous
exceed
ostracize
effusively
encouragingly
dogmatic
festive
virulence
paradise
depravedly
mischief
terrorize
sanctimonious
fit
epic
deign
obliterated
obscenity
pitiable
struggle
desire
detest
surging
unilateralism
sob
misunderstanding
gaggle
majesty
disapprobation
lowly
propaganda
stupendously
will
bombardment
chivalry
benevolence
disputable
kid
pestilent
soothe
idiocy
masterpiece
vulnerable
unexpectedly
capricious
erroneous
anti-American
maladjusted
quaint
ironic
imperious
waning
languorous
incongruously
infatuated
nefarious
poetic
despoiler
impeccable
auspicious
objection
straightforward
sullen
supporter
abominate
scorn
pretentiously
mawkishly
terror-genic
bargain
misfortune
heavily
suffocate
invalidate
temptingly
accusing
redeeming
stench
impolitic
irrefutably
jealousness
alluring
suppose
ludicrously
cuss
slanders
truculent
collusion
disappointed
misrepresentation
sore
splendid
gallantly
sidetrack
dishearteningly
amicability
massacres
inaptitude
inexpiable
humble
preference
nifty
irreproachable
futile
decayed
fractiously
perplexity
inconsistencies
overture
eagerness
exaggeration
traitor
madness
feebleminded
honest
unacceptablely
superficial
dignity
malodorous
shrill
acclaim
dotingly
virtue
impertinent
obscure
praiseworthy
disclaim
unworkable
sane
motivation
disadvantageous
exult
parody
arousal
butchery
conjecture
displease
dungeon
elegance
frank
furiously
grieve
prejudice
scoldingly
first-rate
revelatory
anyhow
repudiate
blindside
glorious
dissenter
opportunity
persecution
worthless
misuse
spellbind
irresponsibly
pertinacity
thinking
drama
unreliability
disagree
keen
degradingly
tolerably
jeers
decay
masterful
exemplar
doubt
convoluted
misguidance
moody
regard
screaming
suspicions
ebullience
infamously
shock
dignified
talent
inappropriately
deploringly
discombobulate
emaciated
superiority
glimmer
tolerable
inconvenient
abysmal
marvelousness
overplay
simplistic
mystery
disaffect
compelling
desperately
everlasting
odder
lawless
ignoble
actual
twinkly
embarrassment
courtesy
unease
severe
bragger
sinister
irresistible
overachiever
trepidation
futility
pitiful
deeply
self-criticism
wayward
glamorous
exceptional
annoyingly
bleakness
courageousness
coupists
profess
oppose
downfall
unruly
acrid
rogue
thrills
confute
pest
wearisome
aggrieve
skittishly
unimaginable
enflame
galore
appealing
bedlamite
miraculously
pessimistically
exceedingly
inconsistence
squirm
tribute
atrocity
altruistic
outwit
one-side
punch
reticent
ultra-hardline
infamy
unscrupulously
anti-proliferation
betrayer
vexing
embellish
regally
spilling
tormented
egomania
propitiously
inspiration
doddering
penetrating
dissatisfying
humility
irresponsible
carnage
vivid
zealous
sting
chaotic
endure
opposition
supurb
flourishing
actually
besides
nag
infuriatingly
enthral
heresy
monstrous
dubiously
miser
inelegance
disown
frivolous
dazzle
bullies
irrepressible
irrefutable
startle
unassailable
enchant
assuredly
dumbfounded
perturb
downheartedly
wee
discord
lousy
hotbeds
swipe
heartening
repulsing
moan
stupendous
bolster
little
conscientious
woefully
peeved
nationalism
sumptuousness
heavenly
profusion
unparalleled
defunct
mannerly
barren
inclination
indecent
traitorously
uncompromising
fateful
lecherous
solace
marvelous
frown
haggard
embarrassingly
argue
remorselessness
comforting
humourous
ill-sorted
deserved
punctual
warmly
archaic
disingenuously
disallow
lambast
sanctify
discourteously
afraid
jittery
sorrowful
bashful
devote
loophole
misgivings
incorruptible
cannibalize
precipitate
malign
elan
god-given
temptation
unhappy
cozy
paucity
staid
irreformable
inimically
patriot
cringe
paupers
abide
assent
idiots
perturbed
strangely
muscle-flexing
obscenely
deceitfulness
usurp
morbidly
generosity
uncivil
anti-
fake
sty
sober
barbarically
gainfully
ingrate
traumatic
impede
ascertainable
bravery
dumbfound
lambaste
implausibly
nurture
adroit
excitingly
gasp
trample
aspersions
browbeat
worth-while
wrest
impolite
demeaning
expire
harms
despondence
lukewarm
unsavory
squabbling
protest
painstaking
valor
warlike
exaltation
amusingly
horrifically
blunders


In [ ]:
%%writefile /kaggle/working/lexicons/weak_subjectives_riloff2003.txt
stability
clamor
stands
transparency
ineligible
slowly
unencumbered
poor
spinster
insurrection
debilitating
repress
self-sufficiency
even
flaws
spare
ignite
likely
viable
destruction
difference
unsuccessfully
famed
expert
commensurate
adverse
shelter
recognizable
trial
captive
escapade
improved
resistant
sound
conquer
pre-eminent
intractable
flakey
disintegration
devout
temperance
distinctly
inefficiency
press
effectiveness
fading
revolution
crisis
intense
nascent
weep
modest
recession
reaffirm
enemies
terrorism
complicate
repressive
adrift
cramped
intrigue
breakdown
retire
faint
unaccustomed
fugitive
distinct
incompliant
delicate
benefit
disproportionate
colorful
lenient
balanced
busy
deficient
manipulation
light
suitable
differ
stumble
flimsy
justice
shrug
confrontational
complicated
need
vintage
matter
electrify
ruin
challenge
broad-based
enrichment
prove
durability
heavy-duty
prospect
sickness
warning
reputed
sustainable
allergic
clandestine
healthy
taste
challenging
displaced
full-blown
smooth
accountable
responsibly
shaky
demolisher
swift
autonomous
relapse
liable
indigent
persistence
vocal
revelation
evasion
stimulative
avoidance
apprehend
acceptance
brightness
restless
uncertain
open-ended
fever
fat
absence
decisive
repetitive
subvert
harass
stole
dictatorial
reliable
brotherly
aristocratic
peripheral
opportunistic
revert
try
loss
sleek
blindness
criminal
outgoing
responsible
broke
weed
factual
prospects
chilly
hostile
decrease
extremists
reunite
immune
insular
mend
oblique
miscalculate
co-operation
death
major
concessions
conflict
manipulate
chill
crime
support
obsolete
record-setting
adventure
inexpert
shortage
limitation
strain
proportionately
abandoned
firmly
cleanse
appease
unsettled
achieve
darkness
handicapped
should
wrestle
good
imprisonment
fully
deadly
odd
continuous
covert
messy
upward
wealthy
universal
security
expedient
overwhelm
important
violent
extort
uprising
entangle
heaven
unsupported
dilemma
openness
transport
entertain
resilient
right
imperfect
assassinate
further
detract
potential
snare
violently
isolated
industrious
lessen
quarrel
surrender
fulfillment
asylum
fraudulent
responsibility
judgement
faulty
unsure
gamble
completely
achievable
toxic
survivor
award
problems
refinement
unlicensed
vastly
obvious
memories
possibility
allege
accessible
obstruction
involuntarily
increasingly
benefactor
hobble
commodious
interested
adept
fiery
reasoned
energize
sorrow
preeminent
amplify
narrow
spirited
cohere
easiest
renewable
careful
oversight
fluent
eligible
comprehensive
divorce
non-violence
wide
cooperate
logical
punitive
incorrect
steep
impartiality
cold
affect
high
bent
resound
impressions
liberal
driven
bask
premier
bullish
infectious
exhaust
motionless
commonsensical
diplomacy
bunk
short-lived
cognizance
alienate
perspective
compulsory
afloat
imposing
uniform
impersonal
adversary
interrupt
offence
constitutions
dispensable
lame
miff
competency
viability
usable
magnify
diligent
stun
else
lanky
jumpy
integral
harden
conspiracies
unconstitutional
damaging
infected
nourish
withheld
permissible
influence
replete
blood
envisions
civil
disorder
comical
dead
warfare
restlessness
shake
help
malady
assuage
reliability
resistance
chum
overpower
delinquency
failure
fist
diversified
liberalism
subscribe
subversive
live
stealing
simple
subordinate
lifelong
impress
intimate
warp
nourishment
numb
deficiency
splitting
reiterates
fleeting
uneven
concrete
unnecessary
fast
correct
peace
illiquid
sketchy
depression
addiction
just
apt
positive
foolproof
resolved
esoteric
move
horde
afflict
frequent
incompatible
dance
large-scale
show
abdicate
civilization
powerful
disable
prohibitive
wane
equality
cash-strapped
feeble
inflated
marginal
minor
modern
unraveled
shield
discriminatingly
dignify
expressions
sneak
grail
thrifty
murderous
pivotal
dawn
stood
diseased
remark
maneuver
precarious
congested
boom
comeback
convenient
ample
feint
refresh
competent
guardian
semblance
adhesion
abundant
achievement
argument
crisp
stolen
toleration
vomit
preoccupy
treacherous
touch
risk-free
assurance
calmness
legitimacy
tardy
loser
war
butcher
displace
massive
corrosion
inadequacy
contamination
idealism
objective
all-time
hectic
respect
perilously
malignant
bar
scholarly
deviate
ineffective
revolt
subtle
floor
restricted
expel
chant
rationality
needlessly
upfront
gain
sweetly
full
practical
undone
debilitate
refine
crushing
imperfectly
evenly
clash
belated
first-class
disordered
controversy
fairly
disinterested
steeply
sweet
tire
predatory
relax
instructive
deplete
flawed
neatly
piety
entrance
dim
hazard
weary
fiction
reactions
agreement
bright
outdated
learned
portable
complement
luxuriate
qualified
expose
indiscernible
guilty
acceptable
slack
urgent
affluence
rich
politeness
submissive
subtract
normal
renewal
mind
overbalanced
distinguish
discriminating
altercation
historic
ineffectively
simply
muddy
stimulate
contribution
amenity
dirty
coddle
unusual
ail
negation
insignificantly
protect
wrought
instability
demand
hardy
tumultuous
peacekeepers
legendary
protection
decency
retaliatory
incomparably
blow
decrement
specifically
prime
clear
feat
discrimination
intention
effectual
reinforcement
elate
temper
fair
stupefy
flaw
however
precariously
authoritarian
commentator
promptly
deterioration
demolish
economical
abrade
sturdy
qualify
protective
stump
prosperity
compatible
egalitarian
consideration
secretive
sounding
syndrome
valid
adaptive
evasive
low-risk
slow
tangled
unrestricted
unnoticed
starvation
recognition
judgments
gigantic
swiftness
lost
repulsed
commonsensible
void
amputate
much
sour
abound
struck
claim
intermittent
giant
sleepy
purity
unauthentic
voluntary
unbound
plague
miss
unresolved
survive
rhetoric
sustainability
achievements
beneficial
beacon
efficient
innovation
simplify
leniently
diminishing
engross
dig
headway
waste
conclusive
plenary
outsider
pamper
golden
best-selling
confounded
strenuous
straggler
youthful
wrongful
crumple
protracted
energetic
lie
reiterated
suppression
destiny
lapse
confront
invasive
noticeable
jar
uninsured
asset
ceaseless
alarm
particular
appeal
complication
low
assessment
wildly
readily
inexact
unprepared
giants
hurt
nauseate
redundancy
tease
wide-open
battered
enlightenment
principle
lethal
failures
sparkling
easy
upgrade
discreet
passive
extremism
pry
utilitarian
ruffle
mild
uniformly
pale
expropriate
flair
imprecisely
cartoon
maturely
assail
nature
strength
unneeded
tantalize
crush
consistent
volatile
finagle
forthright
symptom
impermissible
placid
inhibit
bug
rhapsodize
efficacious
intents
honeymoon
insubordinate
dwindling
rot
sedentary
stunt
overtaxed
implication
signals
adeptly
vigorously
reasonable
scarcity
damage
settle
unhealthy
irregularity
rare
extraneous
agreeable
might
pointless
pressures
whatever
emotions
heal
understanding
very
original
hardliner
tentatively
blast
omission
garbage
unspecified
exacting
cursory
precedent
ignore
inequalities
principled
verifiable
rupture
fissures
prosperous
hardened
dictator
engulf
afford
gnawing
assessments
exhaustive
temperate
white
pure
bumpy
persistent
bulky
intensive
wreck
dynamic
divided
overawe
threaten
murderously
gross
harboring
pain
wild
forged
swoon
diplomatic
undo
empty
lawfully
dominant
groove
invisible
free
adversarial
susceptible
cleanliness
booming
champion
hegemonism
reliably
holy
nourishing
deter
statements
skyrocketing
naturally
charity
tall
angel
smear
bearish
subversion
dangerous
repel
prudence
go-ahead
immobilized
revive
informative
dud
too
buckle
primarily
implicate
thriving
ensure
specific
capitalize
insufficiency
pressure
deaf
trying
revoke
salivate
spacious
lifeless
imperial
hostage
appear
swing
recompense
grand
unlawful
satisfy
patient
looking
myriad
stance
readiness
discrepant
charitable
infiltrators
blossom
barely
limited
bid-rigging
ineffectiveness
reform
big
warm
influential
nettle
predictablely
violation
remedy
navigable
risk
bruise
trust
sharp
choke
error
fierce
uninformed
defense
impossibly
significant
unaudited
graft
showdown
realization
retaliate
abilities
capably
advanced
combative
aptitude
imprecise
famous
properly
slug
cordial
inexperience
competitive
insolvent
unique
poisonously
mope
decreasing
galvanize
brisk
colossal
mean
extreme
urgently
resignation
helplessness
thought
disadvantaged
quit
painfully
inadvisable
casualty
spirit
stake
educated
irritant
controversial
incompatibility
tension
securely
elevate
bankrupt
prepared
punishable
problem
success
imprison
volatility
disorderly
sever
celebrity
fruitful
affordable
high-quality
enliven
lag
ruinous
unfunded
deviation
freedom
hefty
isolation
leverage
apparently
fatigue
substantial
stab
overtures
deal
breach
stain
practically
admission
block
astray
deform
destine
sneaky
starve
tale
unskilled
polution
quiet
worship
advantages
illiterate
authentic
educable
meticulous
lull
innocently
deteriorating
incitement
inclement
honor
erosion
excuse
inefficiently
curt
discompose
battle
blackmail
knock
painful
overshadow
frozen
scant
implicit
luminous
pricey
stand
empowerment
efficiency
buoyant
devastation
safe
intend
commitment
inertia
against
least
proper
central
primitive
cow
substantive
grill
unjustified
hardliners
successful
hasty
attitudes
cancerous
consciousness
lower-priced
restriction
imprecision
negative
negate
advantage
innumerably
eventful
subjection
batons
explode
static
drive
shred
throw
torture
pertinent
comment
rescue
halt
felon
accomplishment
mystify
revitalize
undisputed
comfortably
possible
individuality
oversimplify
exhaustively
suicidal
worn
epidemic
dreams
unconditional
smash
threatening
transgression
infiltrator
ambush
outspoken
advice
juvenile
selective
evaluation
react
inconsistent
slim
adherence
blind
repair
comfort
friends
insurmountably
delicacy
division
coarse
bloody
tender
alienated
arguments
unsteadiness
tenuous
forbidding
distinctive
solemn
sluggish
suffice
thrift
authoritative
outbreak
benefits
helpful
inaction
regressive
insignificance
addict
wounds
subversively
less
congestion
onward
affirmative
legacies
renovate
cohesion
sensitive
compromises
murder
scarce
spot
rocky
sense
sag
upright
increasing
strong
eternal
certain
indebted
could
rehabilitate
adaptability
figurehead
accordance
intricate
insufficient
needless
likelihood
malaise
beggar
unfinished
vague
sustained
well-educated
relaxed
black
facilitate
agility
shine
alleviate
stabilize
value
established
tense
capsize
power
liberally
meticulously
low-cost
still
peaceful
enhanced
predictable
bleed
elimination
corrosive
feeblely
complex
discriminate
unscathed
accomplishments
wasteful
dull
break
popularity
precision
truant
chore
unproductive
excuses
riches
suspect
democratic
competence
legacy
intentions
killer
oversized
entrust
errant
ineffectually
unsteadily
rigorous
twists
beware
functional
tiny
hustler
enhance
incomplete
treacherously
entanglement
thrash
compromise
confer
reservations
pro
cost-saving
unclear
recourses
content
inexperienced
vanity
so
bonus
isolate
experienced
drowning
well-positioned
concerted
muddle
top
flash
erode
glitch
look
progress
untrained
gems
inactive
accept
appropriate
less-expensive
smelling
capability
rarely
allowable
posture
snarl
deterrent
special
cancer
dominance
complete
enjoy
viper
endless
frigid
obstruct
exclusion
prohibit
disease
stringently
whiff
deformed
guidance
jolt
vanish
cooperative
vastness
prisoner
fight
less-developed
elevated
treat
redundant
antagonist
stronger-than-expected
fraud
inference
feign
avoid
envision
resolve
quench
contaminate
adventurous
aptly
profit
detraction
wide-ranging
useful
welcome
lucrative
game
unable
steadiness
dependable
solid
impunity
kiss
scandal
collapse
suit
stable
fatally
rigid
substandard
compact
deluge
embolden
genuine
reasonably
abundance
satisfying
snatch
backbone
rapid
reaffirmation
enterprising
difficulties
prompt
most
thorough
sufficient
confounding
cooperation
affably
comedy
notion
direct
fumble
definitely
grapple
relevant
independent
penalty
skill
subjected
accordantly
friendship
injury
tortured
rehabilitation
skilled
inventive
indeterminably
painlessly
accord
sneakily
sovereignty
fatal
duty
slender
improve
orderly
straight
advisable
accidental
inconclusive
powerfully
junk
self-sufficient
enrich
sweeten
incite
tamper
above
decent
eccentricity
care
bum
sharply
natural
soundness
funded
paralize
beset
adherent
resurgent
indefinite
capitulate
exclusively
retard
tepid
refined
undefined
oversimplified
innocent
popular
unexplained
uncontrolled
tenuously
restraint
sicken
effectively
pacify
vigilant
weak
cost-effective
huge
needs
accommodative
tarnish
agilely
maturity
uncollectible
downgrade
incense
lethargy
cave
vast
involuntary
speculate
niggle
omit
ambitiously
conspiracy
subside
uncomfortable
jobless
feasible
enough
scenic
suicide
forfeit
unity
continuity
extensive
haunt
encroachment
brittle
inefficient
sourly
distract
champ
elite
impossiblity
eradicate
abandonment
secure
wrong
inexpertly
above-average
legalistic
retreat
minister
judgements
impoverished
decisiveness
slightly
contagious
reiterate
siege
pleasantly
declining
germane
flashy
divide
brilliance
resolute
hothouse
enable
powerless
outmoded
animated
pointlessly
covenant
strangle
trim
restrictive
decide
immediate
inadverently
essential
danger
expensive
frail
humanity
necessarily
progressive
enhancement
droop
peacefully
durable
insensible
dwindle
brighten
lighten
well-established
exhaustion
absentee
approval
informational
polished
usefulness
slaughtered
crackdown
discretion
prominent
interests
dope
attack
familiar
suffer
breakthrough
significance
thrive
inconsistency
invigorate
strike
plea
convince
speedy
uplift
menial
tendency
indirect
hypnotize
leaky
warped
non-violent
fine
okay
valuable
divergent
idea
unstable
unavailable
split
unfulfilled
shipwreck
motive
succeed
fictional
inalienable
suppress
tenderly
originality
stir
gritty
nap
pleasant
inadequately
dying
negligible
farsighted
affinity
precisely
full-scale
collide
gestures
largely
indeterminable
reconcile
divorced
unexpected
tradition
over-valuation
close
vigilance
leading
approve
sunny
slaughter
leak
easier
inability
impartially
sinking
darken
overbalance
encroach
infest
indistinct
heart
improvement
parasite
insufficiently
instable
threats
dark
robust
dominate
regress
stringent
enemy
point
purification
obligation
attune
innocence
blur
definite
stormy
flake
proportionate
bomb
delinquent
headache
growing
rough
icy
stiff
learning
better
deep
prominence
primary
fairness
commonsense
polite
clarity
deflationary
welfare
appreciable
survival
overrun
signify
particularly
revival
mistified
innumerous
reason
expression
tickle
hegemony
inaccessible
rapprochement
flaky
lecture
overturn
fatty
legitimate
anomaly
forceful
concern
favored
kill
bull
dreamy
haphazard
knife
fuzzy
soreness
importantly
impartial
helpless
traditional
effective
repression
lean
capable
open
diminish
self-examination
substantially
seasoned
repulse
wallow
adaptable
dislocated
eccentric
possibly
overt
fastest-growing
evident
queer
reestablish
fiat
uneconomical
relevance
junky
objectively
slump
miscalculation
consume
evade
large
disintegrate
jollify
empower
anomalous
reparation
exile
corrective
ghetto
insignificant
reconciliation
well
amity
full-fledged
lesser
supposed
impassive
practicable
corrode
embattled
slow-moving
confound
ineffectualness
prestigious
intrusion
calm
impression
slight
definitive
decline
cooperatively
oversimplification
hazardous
bloom
fragile
profitable
quick
untouched
defend
reputable
analytical
real
useless
idle
fast-paced
leakage
rights
basically
disabled
sufficiently
position
sporadic
key
dump
alert
player
hot
illness
murky
dusty
disarm
rat
immense
plentiful
depress
accost
dirt
besiege
clean
goodly
well-publicized
irresolvable
rather
basic
tolerant
prolific
accomplish
friend
assault
wound
nonviolent
force
impossible
second-tier
unprofitable
constrain
injurious
insecure
premium
appearance
inclusive
personages
gold
literate
improving
conscience
flame
onslaught
hard
inaccuracy
shallow
relations
sincerity
gush
unavoidable
firm
excite
constraint
taxing
steady
interruption
neutralize
inaccuracies
cautious
inhibition
prevalent
rival
ill
hurtful
sick
values
unnatural
fathom
extinguish
productive
unforeseen
paralyzed
timely
crowded
incomparable
neat
affliction
orphan
stunted
vagueness
shadowy
trouble
undocumented
diligence
coherent
pinch
boost
flagging
unlawfulness
preemptive
openly
optimistic
stuffed
superior
back
tenderness
unsafe
wedge
abjure
moon
amply
inadvisably
competitiveness
strides
ability
wastefulness
indefinitely
dash
interference
racism
defect
abolish
perilous
soft-spoken
reaction
active
felonious
would
impetus
broad
glut
blemish
erase
versatility
able
outlook
scheme
hostilities
fluster
choppy
regression
plain
conviction
unreasonable
impoverish
fresh
unsuccessful
abate
inadequate
entire
straggle
turbulent
unsteady
affluent
subdued
reinforce
guarantee
indistinctive
mentor
ease
dream
concerning
intent
suggest
tentative
foul
upset
dexterity
restrict
headaches
fail
feasibly
unsustainable
benign
sweetness
absorbed
attainable
favor
titillate
dependent
infested
feverish
breakthroughs
innovative
humanists
marginally
alienation
mature
harbors
intoxicate
risky
magnetic
offset
lethargic
upscale
shiny
versatile
destroy
unconfirmed
acknowledge
dinky
emphasise
eyebrows
venerate
undecided
tempest
widespread
writhe
misapprehend
salable
classic
expropriation
unavoidably
dissolution
successfully
savor
commonsensibly
insistence
lawful
sparingly
overthrow
abuse
stampede
vigorous
expulse
demands
unlikely
pique
battering
abnormal
defensive
enrapture
boycott
victory
fortunate
underpaid
coping
faze
fragmented
apparent
precise
jam
clique
easiness
easily
chronic
clog
gem
restore
unmoved
tolerate
concession
insurmountable
uneventful
painless
infraction
instrumental
unlimited
down
halfway
anemic
interest
liberation
undid
transparent
rivalry
irregular
acknowledgement
certified
extremist
high-powered
hungry
withhold
limit
discomfort
immature
pan
warmth
intelligence
assertive
comfortable
reclaim
illicit
illegally
allow
unfamiliar
far-reaching
intercede
retract
prohibitively
stagger
clamorous
intimidation
engage
ready
bombard
hazy
restful
cut
totalitarian
topple
adequate
illegal
assurances
abuses
fertile
costly
honest
posterity
reluctantly
relieve
accurate
obscure
idealist
screw
emancipate
innumerable
lesser-known
reluctant
toll
abandon
renovation
indication
amnesty
refusal
fall
loyal
thumb
fabricate
lose
unobserved
olympic
defeat
streamlined
conducive
unpredictable
such
rigor
purify
agile
protests
threat
equitable
voluntarily
offensive
fast-growing
recessionary
ambitious
illegitimate
responsive
severe
assumption
affability
difficult
fracture
expectation
learn
stagnant
attain
diligently
self-defeating
intact
hole
necessary
hard-working
judgment
stainless
commonplace
indeterminate
debility
resigned
enlighten
affectation
irretrievable
battlefield
abrupt
airs
acquit
strained
eliminate
expressive
din
inflationary
exclusive
sting
deteriorate
distinction
genial
intensively
steal
thumbs
gentle
drought
hard-hit
tragedy
destined
uproot
speculation
systematic
unconventional
constructive
expediencies
cohesive
intimidate
explosive
moot
spotty
skimpy
reveal
restoration
simplified
exposed
overreach
victorious
sparing
suspicion
artificial
importance
inexpensive
ineffectual
poisonous
inadverent
life-threatening
freedoms
fundamentalism
beliefs
ripe
modesty
crumble
crude
helplessly
nonexistent
console
unlawfully
injure
ailment
educational
prison
unbroken
exclude
recover
mostly
pollute
encourage
legitimately
validity
uncompetitive
although
sober
pedestrian
manageable
needy
unsatisfactory
undetermined
interfere
untested
storm
accurately
unbiased
vocally
suggestions
invader
transgress
civilize
stances
expound
sweetheart
noisy
seem
sticky
independence
predominant


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

import biasneut
print("biasneut package reconstructed, version", biasneut.__version__)


## 3. Sanity check (tiny models, ~1 minute)

Runs the detector and editor forward/generate paths against tiny
random-weight HF checkpoints (`hf-internal-testing/tiny-random-*`) — the same
smoke test used during local development. This exists to catch an import
typo, a shape mismatch, or an API-version drift (transformers/peft version
skew between this environment and Kaggle's) in about a minute, instead of
discovering it after an hour of real training.

In [ ]:
from transformers import AutoTokenizer

from biasneut.data.collate import DetectorCollator
from biasneut.data.schema import DetectionExample, EditExample
from biasneut.detector.infer import DetectorInference
from biasneut.detector.model import BiasDetector
from biasneut.editor.seq2seq_model import SeqEditor
from biasneut.pipeline import BiasNeutralizationPipeline

TINY_DETECTOR_BACKBONE = "hf-internal-testing/tiny-random-bert"
TINY_EDITOR_BACKBONE = "hf-internal-testing/tiny-random-t5"

# Detector: forward pass + loss
detector_model = BiasDetector(backbone=TINY_DETECTOR_BACKBONE)
detector_tokenizer = AutoTokenizer.from_pretrained(TINY_DETECTOR_BACKBONE)
batch = [
    DetectionExample(text="the regime collapsed", is_biased=True, bio_tags=["O", "B-BIAS", "O"], source="smoke"),
    DetectionExample(text="the meeting ended", is_biased=False, bio_tags=["O", "O", "O"], source="smoke"),
]
collated = DetectorCollator(tokenizer=detector_tokenizer, max_length=32)(batch)
out = detector_model(**collated)
assert out.loss is not None and out.loss.item() > 0
print("[OK] detector forward + loss")

# Detector inference wrapper
detector_inference = DetectorInference(detector_model, detector_tokenizer, max_length=32)
predictions = detector_inference.predict(["the regime collapsed", "a calm meeting"])
assert len(predictions) == 2
print("[OK] detector inference:", [(p.text, round(p.bias_prob, 3)) for p in predictions])

# Editor: span markers + generate
editor = SeqEditor.from_pretrained_backbone(TINY_EDITOR_BACKBONE)
outputs = editor.neutralize(
    ["the regime collapsed", "a total disaster happened"],
    [["O", "B-BIAS", "O"], ["O", "B-BIAS", "I-BIAS", "O"]],
    num_beams=1, max_new_tokens=8,
)
assert len(outputs) == 2
print("[OK] editor generate:", outputs)

# Full pipeline
pipeline = BiasNeutralizationPipeline(detector_inference, editor, skip_unflagged=False)
results = pipeline(["the regime collapsed under pressure"], num_beams=1, max_new_tokens=8)
assert len(results) == 1
print("[OK] end-to-end pipeline:", results[0])

print("\nAll sanity checks passed -- safe to proceed to real training.")


## 4. Data acquisition (§4, §6.5)

BABE from the HF Hub, BASIL via a shallow git clone of `launchnlp/BASIL`
(Kaggle images ship `git`), WNC downloaded from the original Stanford release
URL (the same one Pryzant's own `download_data_ckpt_and_run_inference.sh`
used). All three are then split story/outlet-disjoint (§6.5: no story
straddles a split boundary) and cached as JSON under `/kaggle/working/data_cache/`.

In [ ]:
import dataclasses
import json
import logging
from pathlib import Path

from biasneut.common.logging_utils import setup_logging
setup_logging()
logger = logging.getLogger("data_prep")

from biasneut.data.babe import load_babe
from biasneut.data.splits import assert_no_leakage, group_disjoint_split

CACHE_DIR = Path("/kaggle/working/data_cache")
SEED = 42


def dump(examples, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps([dataclasses.asdict(e) for e in examples]))
    logger.info("wrote %d examples to %s", len(examples), path)


logger.info("loading BABE from the HF Hub...")
babe = load_babe()
babe_train, babe_dev, babe_test = group_disjoint_split(babe, seed=SEED)
assert_no_leakage(babe_train, babe_dev, babe_test)
dump(babe_train, CACHE_DIR / "babe" / "train.json")
dump(babe_dev, CACHE_DIR / "babe" / "dev.json")
dump(babe_test, CACHE_DIR / "babe" / "test.json")


In [ ]:
from biasneut.data.basil import load_basil

logger.info("loading BASIL (shallow git clone on first run)...")
basil = load_basil(CACHE_DIR)  # clones into CACHE_DIR/BASIL
basil_train, basil_dev, basil_test = group_disjoint_split(basil, seed=SEED)
assert_no_leakage(basil_train, basil_dev, basil_test)
dump(basil_train, CACHE_DIR / "basil" / "train.json")
dump(basil_dev, CACHE_DIR / "basil" / "dev.json")
dump(basil_test, CACHE_DIR / "basil" / "test.json")


In [ ]:
import shutil
import time
import urllib.request
import zipfile

from biasneut.data.wnc import load_wnc

WNC_ZIP_URL = "https://nlp.stanford.edu/projects/bias/bias_data.zip"
WNC_DIR = Path("/kaggle/working/bias_data/WNC")  # the zip's own top-level folder is "bias_data/"


def _progress(block_num, block_size, total_size, _last=[0.0]):
    if total_size <= 0:
        return
    pct = min(100.0, block_num * block_size / total_size * 100)
    now = time.monotonic()
    if pct >= 100 or now - _last[0] > 5:  # throttle to ~1 line every 5s
        print(f"  downloading WNC... {pct:5.1f}% ({block_num * block_size / 1e6:.0f}MB / {total_size / 1e6:.0f}MB)")
        _last[0] = now


if not WNC_DIR.exists():
    # ~110MB over a research-server connection -- can take a few minutes;
    # the progress line below is so this doesn't look hung.
    logger.info("downloading WNC from %s ...", WNC_ZIP_URL)
    zip_path = Path("/kaggle/working/bias_data.zip")
    urllib.request.urlretrieve(WNC_ZIP_URL, zip_path, reporthook=_progress)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/")
    zip_path.unlink()
logger.info("WNC files: %s", list(WNC_DIR.glob("biased.word.*")))

dump(load_wnc(WNC_DIR / "biased.word.train"), CACHE_DIR / "wnc" / "train.json")
dump(load_wnc(WNC_DIR / "biased.word.dev"), CACHE_DIR / "wnc" / "dev.json")
dump(load_wnc(WNC_DIR / "biased.word.test"), CACHE_DIR / "wnc" / "test.json")


In [ ]:
from biasneut.data.cache_io import load_detection_examples, load_edit_examples

detector_train = load_detection_examples(CACHE_DIR / "babe" / "train.json") + \
    load_detection_examples(CACHE_DIR / "basil" / "train.json")
detector_dev = load_detection_examples(CACHE_DIR / "babe" / "dev.json") + \
    load_detection_examples(CACHE_DIR / "basil" / "dev.json")

wnc_train = load_edit_examples(CACHE_DIR / "wnc" / "train.json")
wnc_dev = load_edit_examples(CACHE_DIR / "wnc" / "dev.json")

print(f"detector: {len(detector_train)} train / {len(detector_dev)} dev")
print(f"WNC editor: {len(wnc_train)} train / {len(wnc_dev)} dev")


## 5. Stage 1 — Detector training (§5.1, §8 Phase 1)

Trained **twice**: `runs/detector` is the instance used everywhere in the
pipeline (span localization, Strategy-B/C filtering); `runs/detector_eval_independent`
is a second, differently-seeded instance that never sees any filtering
decision, reserved purely for the Aggregate Bias Score at evaluation time
(§4.3, §5.1, §6.1 — independence from the training/filtering signal is
essential to avoid Goodharting the metric).

Defaults below target a single P100/T4 (16GB): `bf16` mixed precision (switch
to `fp16` in the P100 case — no tensor cores, §7.2 point 3), gradient
checkpointing on, 8-bit AdamW via bitsandbytes if installed. Re-running this
cell resumes automatically from the latest checkpoint under `output_dir` if
one exists (§7.4) — safe to re-run after a session interruption.

In [ ]:
from transformers import AutoTokenizer

from biasneut.common.config import ComputeConfig, DetectorConfig
from biasneut.common.seeding import set_seed
from biasneut.detector.train import save_pretrained, train_detector

set_seed(42)

detector_cfg = DetectorConfig(
    backbone="launch/POLITICS",       # falls back to distilroberta-base below if this 404s for you
    max_seq_length=128,
    learning_rate=2e-5,
    train_batch_size=16,
    eval_batch_size=32,
    num_epochs=5,
    output_dir="/kaggle/working/runs/detector",
    checkpoint_every_steps=200,
    compute=ComputeConfig(mixed_precision="bf16", gradient_checkpointing=True, optimizer="adamw_8bit"),
)

detector_model = train_detector(detector_train, detector_dev, detector_cfg)
detector_tokenizer = AutoTokenizer.from_pretrained(detector_cfg.backbone)
save_pretrained(detector_model, detector_tokenizer, detector_cfg.output_dir)
print("saved pipeline detector to", detector_cfg.output_dir)


In [ ]:
# Second, independent instance for evaluation only (different seed, never
# touched by any filtering/localization decision elsewhere in this notebook).
set_seed(1337)

independent_detector_cfg = DetectorConfig(
    **{**detector_cfg.__dict__, "output_dir": "/kaggle/working/runs/detector_eval_independent"}
)
independent_detector_model = train_detector(detector_train, detector_dev, independent_detector_cfg)
save_pretrained(independent_detector_model, detector_tokenizer, independent_detector_cfg.output_dir)
print("saved independent eval detector to", independent_detector_cfg.output_dir)


## 6. Stage 2 — Editor training under three data strategies (§4.2, §5.2, §8)

Each strategy is independent — run whichever subset your compute budget
allows this session; they don't depend on each other (Strategy B's optional
warm-start from Strategy A is the one exception, noted below).

### 6A. Strategy A — WNC pretrain (+ optional in-domain adaptation)

Also **is** the "Pryzant off-the-shelf (WNC-trained)" baseline from §6.3 when
run without `adapt_train` (see `biasneut/baselines/trivial.py`'s docstring for
why: the original 2019 release is pinned to `torch==1.1.0` and isn't
loadable in a modern stack, so this checkpoint stands in for it directly).

In [ ]:
from biasneut.common.config import EditorConfig
from biasneut.editor.train_strategy_a import run_strategy_a

editor_cfg_a = EditorConfig(
    strategy="a",
    backbone="t5-small",
    learning_rate=3e-4,
    train_batch_size=16,
    eval_batch_size=32,
    num_epochs=4,
    output_dir="/kaggle/working/runs/editor_strategy_a",
    checkpoint_every_steps=200,
    compute=ComputeConfig(mixed_precision="bf16", gradient_checkpointing=True, optimizer="adamw_8bit"),
)

editor_a = run_strategy_a(wnc_train, wnc_dev, editor_cfg_a)  # pass adapt_train=... for phase 2 domain adaptation
editor_a.save(editor_cfg_a.output_dir)
print("saved Strategy-A editor to", editor_cfg_a.output_dir)


### 6B. Strategy B — LLM-synthesized pseudo-parallel corpus

**Currently disabled (`RUN_STRATEGY_B = False`).** With no `ANTHROPIC_API_KEY`
this arm falls back to `EchoClient`, whose "pairs" have `source == target` —
so it trains an editor to copy its input and reports a `strategy_b` row that
looks like a real comparison but is not one. It is skipped rather than
reported as a non-result.

To turn it into a genuine arm: set `ANTHROPIC_API_KEY` as a Kaggle secret
(Add-ons -> Secrets), then flip `RUN_STRATEGY_B = True` below. Generation
happens offline (no GPU cost) and goes through the mandatory §4.3
circularity mitigations (independent-classifier bias-drop filter, SBERT
similarity floor) before anything trains on it.


In [ ]:
import os

from biasneut.data.pseudo_parallel import (
    AnthropicClient, EchoClient, filter_pseudo_parallel, generate_pseudo_parallel, sample_for_human_review,
)
from biasneut.detector.infer import DetectorInference
from biasneut.eval.preservation import PreservationScorer

USE_REAL_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))  # set via Kaggle Secrets
# Echo-mode pairs are source == target, which makes "strategy_b" a
# copy-the-input model rather than a real arm -- so only run this when a
# genuine LLM client is available.
RUN_STRATEGY_B = USE_REAL_LLM

# Needed by the evaluation section below regardless of whether Strategy B runs:
# the independent detector is the §4.3/§6.1 evaluator, never the pipeline instance.
independent_detector_infer = DetectorInference.from_pretrained("/kaggle/working/runs/detector_eval_independent")
preservation_scorer = PreservationScorer()

filtered_pairs = []
if RUN_STRATEGY_B:
    clients = [AnthropicClient()]
    print("Strategy B generation clients:", [c.name for c in clients], "(real)")

    biased_train_examples = [e for e in detector_train if e.is_biased]
    pairs = generate_pseudo_parallel(biased_train_examples, clients)

    filtered_pairs = filter_pseudo_parallel(
        pairs,
        bias_score_fn=independent_detector_infer.bias_score,
        similarity_fn=lambda s, t: preservation_scorer._sbert.similarity(
            preservation_scorer._sbert.encode(s), preservation_scorer._sbert.encode(t)
        ).item(),
        similarity_floor=0.6,
        require_bias_drop=True,
    )
    print(f"pseudo-parallel pairs: {len(pairs)} generated -> {len(filtered_pairs)} kept after §4.3 filtering")

    review_path = sample_for_human_review(filtered_pairs, out_path="/kaggle/working/runs/pseudo_parallel_review.csv")
    print("human spot-check sample:", review_path)
else:
    print("Strategy B skipped -- no ANTHROPIC_API_KEY, so the only available client is EchoClient "
          "(source == target). Set the secret and flip RUN_STRATEGY_B to run this arm for real.")


In [ ]:
from biasneut.editor.train_strategy_b import run_strategy_b

if RUN_STRATEGY_B and filtered_pairs:
    editor_cfg_b = EditorConfig(
        strategy="b",
        backbone="t5-small",
        train_batch_size=16,
        eval_batch_size=32,
        num_epochs=4,
        output_dir="/kaggle/working/runs/editor_strategy_b",
        checkpoint_every_steps=200,
        compute=ComputeConfig(mixed_precision="bf16", gradient_checkpointing=True, optimizer="adamw_8bit"),
    )
    split = max(1, int(0.9 * len(filtered_pairs)))
    editor_b = run_strategy_b(filtered_pairs[:split], filtered_pairs[split:], editor_cfg_b)
    editor_b.save(editor_cfg_b.output_dir)
    print("saved Strategy-B editor to", editor_cfg_b.output_dir)
else:
    # editor_b intentionally left undefined -- the evaluation section below
    # checks for it and will omit strategy_b from the report entirely.
    print("Strategy B training skipped; it will be omitted from the evaluation report.")


### 6C. Strategy C — fully unsupervised editing (LEWIS-lite + pure mask-and-infill)

Two arms: **mask-and-infill** needs no seq2seq training at all (the trained
detector + an MLM infiller are sufficient at inference time); **LEWIS-lite**
uses that same detector-guided masking to synthesize pseudo-parallel pairs at
scale, then trains the shared seq2seq editor on them (see
`biasneut/editor/lewis.py`'s docstring for why this substitutes for training
a second bespoke tagger).

In [ ]:
from biasneut.detector.infer import DetectorInference
from biasneut.editor.mask_infill import MaskInfiller
from biasneut.pipeline import MaskInfillEditorAdapter

pipeline_detector = DetectorInference.from_pretrained("/kaggle/working/runs/detector")
infiller = MaskInfiller(backbone="roberta-base")

# Optional: nudge the infiller's predictions toward neutral register by
# continuing its MLM objective on human-labeled neutral (label=0) sentences.
neutral_sentences = [e.text for e in detector_train if not e.is_biased][:2000]
infiller.finetune_on_neutral_corpus(neutral_sentences, epochs=1)

mask_infill_adapter = MaskInfillEditorAdapter(infiller)
print("mask-and-infill arm ready (no seq2seq training needed) -- try it via biasneut.pipeline.BiasNeutralizationPipeline")


In [ ]:
from biasneut.editor.train_strategy_c import run_strategy_c_lewis

editor_cfg_c = EditorConfig(
    strategy="c",
    backbone="t5-small",
    train_batch_size=16,
    eval_batch_size=32,
    num_epochs=4,
    output_dir="/kaggle/working/runs/editor_strategy_c_lewis",
    checkpoint_every_steps=200,
    compute=ComputeConfig(mixed_precision="bf16", gradient_checkpointing=True, optimizer="adamw_8bit"),
)

basil_train = load_detection_examples(CACHE_DIR / "basil" / "train.json")
basil_dev = load_detection_examples(CACHE_DIR / "basil" / "dev.json")
biased_basil_train = [e for e in basil_train if e.is_biased]
biased_basil_dev = [e for e in basil_dev if e.is_biased]

editor_c = run_strategy_c_lewis(biased_basil_train, biased_basil_dev, pipeline_detector, infiller, editor_cfg_c)
editor_c.save(editor_cfg_c.output_dir)
print("saved Strategy-C (LEWIS-lite) editor to", editor_cfg_c.output_dir)


### 6C-stretch (optional) — QLoRA'd instruction-tuned decoder editor

Not required by any other part of the pipeline (§7.3) — only run this if you
installed the `bitsandbytes` extra above and want the higher-fluency stretch
arm. Needs a T4 (or better); the 4-bit NF4 quantization is chosen precisely
so this still fits in 16GB.

In [ ]:
from biasneut.common.config import QLoRAEditorConfig
from biasneut.editor.qlora_decoder import train_qlora_editor

RUN_QLORA_STRETCH = False  # flip to True once bitsandbytes is confirmed installed and you have a T4+

if RUN_QLORA_STRETCH:
    qlora_cfg = QLoRAEditorConfig(output_dir="/kaggle/working/runs/editor_qlora")
    qlora_model, qlora_tokenizer = train_qlora_editor(wnc_train[:5000], qlora_cfg)
    print("QLoRA stretch editor trained; adapter saved under", qlora_cfg.output_dir)
else:
    print("Skipped -- set RUN_QLORA_STRETCH = True to run this arm.")


## 7. Baselines + full evaluation (§6)

The mandatory honesty-check baselines (§6.3) plus whichever editor(s) you
trained above, scored on the triad — transfer strength × content
preservation × fluency — with Pareto plotting and significance testing
against `copy_input`.

In [ ]:
from biasneut.baselines.trivial import copy_input_baseline, delete_flagged_word_with_detector
from biasneut.eval.fluency import FluencyScorer
from biasneut.eval.harness import EvaluationHarness, run_full_evaluation
from biasneut.eval.lexicon import LexiconBiasScorer

babe_test = load_detection_examples(CACHE_DIR / "babe" / "test.json")
eval_sources = [e.text for e in babe_test]

eval_predictions = pipeline_detector.predict(eval_sources)
eval_word_tags = [p.word_tags for p in eval_predictions]

def neutralize_chunked(editor, sources, tags, chunk_size=16, **kwargs):
    """Same as editor.neutralize() but processed in small chunks instead of
    one call over all sources. A local reproduction of the unbiased ablation
    (chunks of 16, torch 2.13/transformers 5.15) produced zero degenerate
    (empty/single-token) outputs across all 312 sentences, while the real
    single-batch evaluation call here produced mean_perplexity=inf for every
    editor variant tried so far -- this isolates whether batch size/padding
    (not model quality or copy-bias strength) is the actual cause."""
    outputs = []
    for i in range(0, len(sources), chunk_size):
        batch_sources = sources[i:i + chunk_size]
        batch_tags = tags[i:i + chunk_size]
        outputs.extend(editor.neutralize(batch_sources, batch_tags, **kwargs))
    return outputs

systems = {
    "copy_input": copy_input_baseline(eval_sources),
    "delete_flagged_word": delete_flagged_word_with_detector(eval_sources, pipeline_detector),
}
# Diagnostic ablations (§ two prior runs): copy_bias_strength=2.5 boosts every
# non-flagged subword's logits uniformly at every decoding step, which showed
# only marginal effect on degenerate output; disabling it entirely (v7) still
# left mean_perplexity=inf for every editor variant. A local reproduction with
# the exact same v7 checkpoint, evaluated in small chunks instead of one big
# batch, produced zero degenerate outputs across all 312 sentences -- so this
# adds a same-run, same-checkpoint chunked-vs-single-batch comparison to
# isolate whether large-batch padding is the actual cause, at zero extra
# training cost (generation-only).
if "editor_a" in dir():
    systems["strategy_a"] = editor_a.neutralize(eval_sources, eval_word_tags)
    systems["strategy_a_unbiased"] = editor_a.neutralize(eval_sources, eval_word_tags, use_constrained_decoding=False)
    systems["strategy_a_unbiased_chunked16"] = neutralize_chunked(
        editor_a, eval_sources, eval_word_tags, chunk_size=16, use_constrained_decoding=False
    )
if "editor_b" in dir():
    systems["strategy_b"] = editor_b.neutralize(eval_sources, eval_word_tags)
if "editor_c" in dir():
    systems["strategy_c_lewis"] = editor_c.neutralize(eval_sources, eval_word_tags)
    systems["strategy_c_lewis_unbiased"] = editor_c.neutralize(eval_sources, eval_word_tags, use_constrained_decoding=False)
    systems["strategy_c_lewis_unbiased_chunked16"] = neutralize_chunked(
        editor_c, eval_sources, eval_word_tags, chunk_size=16, use_constrained_decoding=False
    )

lexicon_scorer = LexiconBiasScorer("/kaggle/working/lexicons")
preservation_scorer = PreservationScorer()
fluency_scorer = FluencyScorer(lm_name="gpt2", grammaticality_model="textattack/roberta-base-CoLA")

harness = EvaluationHarness(independent_detector_infer, preservation_scorer, fluency_scorer, lexicon_scorer)
results = harness.evaluate_all(systems, eval_sources)

run_full_evaluation(results, "/kaggle/working/runs/eval")
print("evaluation report written to /kaggle/working/runs/eval/ (report.md, pareto.png, significance.json, summaries.json)")


In [ ]:
with open("/kaggle/working/runs/eval/report.md") as f:
    print(f.read())


## 8. Inference demo

In [ ]:
from biasneut.pipeline import BiasNeutralizationPipeline

demo_editor = editor_a if "editor_a" in dir() else editor_c
demo_pipeline = BiasNeutralizationPipeline(pipeline_detector, demo_editor)

examples = [
    "The corrupt regime brutally cracked down on peaceful protesters.",
    "Officials slammed the plan as reckless and irresponsible.",
    "The committee met on Friday to review the quarterly budget.",
]
for r in demo_pipeline(examples):
    print(f"[{'BIASED' if r.was_flagged_biased else 'neutral'} p={r.bias_prob:.2f}] {r.source}")
    print(f"  -> {r.neutralized}\n")


## 9. Cross-session persistence & resume (§7.4)

`CheckpointManager` already resumes automatically *within* whatever storage
Kaggle gives this session — every training cell above is safe to re-run after
an interrupted session and will pick up from its latest checkpoint rather
than restarting. What it can't do is survive Kaggle recycling
`/kaggle/working` between *separate* edit sessions. To actually carry
progress across sessions (needed given the ~9-12h session cap vs. a
16-week work plan, §7):

1. When you stop for the day, **Save Version** (this notebook's output,
   including everything under `/kaggle/working/runs/`, becomes a Kaggle
   Dataset attached to this notebook's output).
2. Next session, add that previous output as an **input** dataset
   (`+ Add Input` → your notebook's own prior output), then run the cell
   below *before* re-running any training cell — it copies any checkpoints it
   finds back into `/kaggle/working/runs/` so `CheckpointManager.load()`
   picks them up on the very next training call.

In [ ]:
import shutil
from pathlib import Path

PREVIOUS_OUTPUT_DIR = None  # e.g. "/kaggle/input/<your-notebook-slug>/runs" once you've added it as input

if PREVIOUS_OUTPUT_DIR and Path(PREVIOUS_OUTPUT_DIR).exists():
    shutil.copytree(PREVIOUS_OUTPUT_DIR, "/kaggle/working/runs", dirs_exist_ok=True)
    print("restored checkpoints from", PREVIOUS_OUTPUT_DIR)
else:
    print("no previous-session output configured -- set PREVIOUS_OUTPUT_DIR above once you have one")
